In [1]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell1_validate_inputs_v1"
DOLLAR_YEAR = 2022
READ_CHUNK_ROWS = 250_000
ROW_EQUATION_TOLERANCE_USD = 1e-6
AGGREGATE_TOLERANCE_USD = 0.01
NUMERIC_TOLERANCE = 1e-10
ACCEPTED_CELL12_WARNING_IDS = {"extreme_return_period_tail_support_documented"}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_5_damage_loss"
            / "notebook_5_cell_12_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def output_record_path(
    summary: dict[str, Any], output_name: str, project_root: Path
) -> Path:
    record = summary["outputs"][output_name]
    if isinstance(record, dict):
        return resolve_recorded_path(project_root, str(record["path"]))
    return resolve_recorded_path(project_root, str(record))


def verify_record_hash(
    summary: dict[str, Any], output_name: str, path: Path
) -> tuple[bool, str, str | None]:
    record = summary["outputs"][output_name]
    actual = sha256_file(path)
    expected = str(record.get("sha256")) if isinstance(record, dict) else None
    return expected is None or actual == expected, actual, expected


PROJECT_ROOT = find_project_root()
NB5_METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_5_damage_loss"
NB6_METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
NB6_PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_insurance_parameters"
NB6_METADATA_DIR.mkdir(parents=True, exist_ok=True)
NB6_PARAMETER_DIR.mkdir(parents=True, exist_ok=True)

CELL11_SUMMARY_PATH = NB5_METADATA_DIR / "notebook_5_cell_11_summary.json"
CELL11_VALIDATION_PATH = NB5_METADATA_DIR / "notebook_5_cell_11_validation.csv"
CELL12_SUMMARY_PATH = NB5_METADATA_DIR / "notebook_5_cell_12_summary.json"
CELL12_VALIDATION_PATH = NB5_METADATA_DIR / "notebook_5_cell_12_validation.csv"
NOTEBOOK4_HANDOFF_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_4_final_handoff"
    / "notebook_5_input_handoff.json"
)

INPUT_INVENTORY_PATH = NB6_METADATA_DIR / "notebook_6_cell_1_input_inventory.csv"
VALIDATION_PATH = NB6_METADATA_DIR / "notebook_6_cell_1_validation.csv"
SUMMARY_PATH = NB6_METADATA_DIR / "notebook_6_cell_1_summary.json"
HANDOFF_PATH = NB6_METADATA_DIR / "notebook_6_input_handoff.json"
EXPOSURE_BASE_PATH = NB6_PARAMETER_DIR / "seaside_w2_insurance_exposure_base.csv"

validation_rows: list[dict[str, Any]] = []
required_metadata_paths = [
    CELL11_SUMMARY_PATH,
    CELL11_VALIDATION_PATH,
    CELL12_SUMMARY_PATH,
    CELL12_VALIDATION_PATH,
    NOTEBOOK4_HANDOFF_PATH,
]
for path in required_metadata_paths:
    append_check(
        validation_rows,
        f"required_metadata_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_metadata = [str(path) for path in required_metadata_paths if not path.exists()]
if missing_metadata:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required Notebook 5 inputs are missing: {missing_metadata}")

cell11_summary = load_json(CELL11_SUMMARY_PATH)
cell12_summary = load_json(CELL12_SUMMARY_PATH)
notebook4_handoff = load_json(NOTEBOOK4_HANDOFF_PATH)

cell11_validation = pd.read_csv(CELL11_VALIDATION_PATH)
cell12_validation = pd.read_csv(CELL12_VALIDATION_PATH)
for table, name in [
    (cell11_validation, "cell11"),
    (cell12_validation, "cell12"),
]:
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    append_check(
        validation_rows,
        f"{name}_validation_schema_complete",
        not missing,
        f"missing={missing}",
    )
    if missing:
        raise KeyError(f"{name} validation is missing columns: {missing}")
    table["passed"] = parse_bool_series(table["passed"])

cell11_failures = cell11_validation.loc[
    cell11_validation["severity"].astype(str).str.lower().eq("critical")
    & ~cell11_validation["passed"]
]
cell12_failures = cell12_validation.loc[
    cell12_validation["severity"].astype(str).str.lower().eq("critical")
    & ~cell12_validation["passed"]
]
append_check(
    validation_rows,
    "cell11_critical_validation_passed",
    cell11_failures.empty and bool(cell11_summary.get("all_critical_checks_passed")),
    f"critical_failures={len(cell11_failures)}",
)
append_check(
    validation_rows,
    "cell12_critical_validation_passed",
    cell12_failures.empty and bool(cell12_summary.get("all_critical_checks_passed")),
    f"critical_failures={len(cell12_failures)}",
)

cell12_open_warnings = cell12_validation.loc[
    cell12_validation["severity"].astype(str).str.lower().eq("warning")
    & ~cell12_validation["passed"]
]
warning_ids = set(cell12_open_warnings["check_id"].astype(str))
unexpected_warning_ids = sorted(warning_ids.difference(ACCEPTED_CELL12_WARNING_IDS))
append_check(
    validation_rows,
    "cell12_only_documented_tail_warning_remains",
    not unexpected_warning_ids,
    (
        f"open_warning_ids={sorted(warning_ids)}; "
        f"unexpected_warning_ids={unexpected_warning_ids}"
    ),
)
append_check(
    validation_rows,
    "cell12_tail_warning_retained_in_handoff",
    warning_ids.issubset(ACCEPTED_CELL12_WARNING_IDS),
    f"accepted_warning_ids={sorted(warning_ids)}",
    severity="warning",
)

annual_catalog = notebook4_handoff["annual_catalog"]
declared_years = int(annual_catalog["declared_duration_years"])
expected_occurrences = int(annual_catalog["occurrences"])
expected_buildings = int(cell11_summary["annual_catalog"]["sites"])
expected_building_rows = int(cell11_summary["annual_catalog"]["rows"])
expected_occupied_years = int(annual_catalog["occupied_years"])
expected_zero_event_years = int(annual_catalog["zero_event_years"])
expected_multiple_event_years = int(annual_catalog["multiple_event_years"])
portfolio_replacement_value = float(
    cell11_summary["portfolio_valuation"]["total_replacement_value_2022_usd"]
)
expected_sampled_total = float(
    cell11_summary["production"]["sampled_total_ground_up_loss_2022_usd"]
)
expected_analytical_total = float(
    cell11_summary["production"]["analytical_expected_total_ground_up_loss_2022_usd"]
)
expected_sampled_aal = float(
    cell12_summary["risk_metrics"]["sampled_total_ground_up_aal_2022_usd"]
)
expected_analytical_aal = float(
    cell12_summary["risk_metrics"]["analytical_expected_total_ground_up_aal_2022_usd"]
)

full_loss_path = output_record_path(cell11_summary, "full_total_ground_up_loss", PROJECT_ROOT)
event_loss_path = output_record_path(cell11_summary, "event_total_ground_up_loss", PROJECT_ROOT)
annual_series_path = output_record_path(cell12_summary, "annual_loss_series", PROJECT_ROOT)
pml_path = output_record_path(cell12_summary, "pml_table", PROJECT_ROOT)
risk_metrics_path = output_record_path(cell12_summary, "risk_metrics", PROJECT_ROOT)
source_aal_path = output_record_path(cell12_summary, "source_aal_summary", PROJECT_ROOT)
component_aal_path = output_record_path(cell12_summary, "component_aal_summary", PROJECT_ROOT)

input_records = [
    ("full_building_ground_up_loss", full_loss_path, cell11_summary, "full_total_ground_up_loss"),
    ("occurrence_ground_up_loss", event_loss_path, cell11_summary, "event_total_ground_up_loss"),
    ("annual_ground_up_loss_series", annual_series_path, cell12_summary, "annual_loss_series"),
    ("ground_up_pml_table", pml_path, cell12_summary, "pml_table"),
    ("ground_up_risk_metrics", risk_metrics_path, cell12_summary, "risk_metrics"),
    ("source_aal_summary", source_aal_path, cell12_summary, "source_aal_summary"),
    ("component_aal_summary", component_aal_path, cell12_summary, "component_aal_summary"),
]

inventory_rows: list[dict[str, Any]] = []
for role, path, summary, output_name in input_records:
    append_check(
        validation_rows,
        f"input_exists_{role}",
        path.exists(),
        str(path),
    )
    if not path.exists():
        raise FileNotFoundError(path)
    hash_matches, actual_hash, expected_hash = verify_record_hash(summary, output_name, path)
    append_check(
        validation_rows,
        f"input_hash_matches_{role}",
        hash_matches,
        f"expected={expected_hash}; actual={actual_hash}",
    )
    record = summary["outputs"][output_name]
    inventory_rows.append(
        {
            "role": role,
            "path": str(path),
            "sha256": actual_hash,
            "size_bytes": int(path.stat().st_size),
            "recorded_rows": record.get("rows") if isinstance(record, dict) else None,
        }
    )

required_building_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    "sa0p4_simulated_g",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    "floor_area_sqft",
    "replacement_cost_per_sqft_2022_usd",
    "building_replacement_value_2022_usd",
    "sampled_structural_ground_up_loss_2022_usd",
    "sampled_nsd_ground_up_loss_2022_usd",
    "sampled_nsa_ground_up_loss_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_structural_ground_up_loss_2022_usd",
    "analytical_expected_nsd_ground_up_loss_2022_usd",
    "analytical_expected_nsa_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
}
full_header = pd.read_csv(full_loss_path, compression="gzip", nrows=0)
missing_building_columns = sorted(required_building_columns.difference(full_header.columns))
append_check(
    validation_rows,
    "building_loss_schema_complete",
    not missing_building_columns,
    f"missing={missing_building_columns}",
)
if missing_building_columns:
    raise KeyError(f"Full ground-up loss file is missing columns: {missing_building_columns}")

stream_columns = sorted(required_building_columns)
seen_pairs = np.zeros(expected_building_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int32)
site_counts = np.zeros(expected_buildings, dtype=np.int32)
site_records: dict[int, dict[str, Any]] = {}
site_consistency_errors = 0
total_rows = 0
sampled_total_sum = 0.0
analytical_total_sum = 0.0
maximum_sampled_equation_error = 0.0
maximum_analytical_equation_error = 0.0
maximum_sampled_loss_ratio = 0.0
minimum_sampled_loss = np.inf
minimum_analytical_loss = np.inf
source_types: set[str] = set()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        full_loss_path,
        compression="gzip",
        usecols=stream_columns,
        chunksize=READ_CHUNK_ROWS,
    ),
    start=1,
):
    numeric_columns = [
        "catalog_year",
        "occurrence_ordinal",
        "site_ordinal",
        "magnitude",
        "sa0p4_simulated_g",
        "year_built",
        "floor_area_sqft",
        "replacement_cost_per_sqft_2022_usd",
        "building_replacement_value_2022_usd",
        "sampled_structural_ground_up_loss_2022_usd",
        "sampled_nsd_ground_up_loss_2022_usd",
        "sampled_nsa_ground_up_loss_2022_usd",
        "sampled_total_ground_up_loss_2022_usd",
        "analytical_expected_structural_ground_up_loss_2022_usd",
        "analytical_expected_nsd_ground_up_loss_2022_usd",
        "analytical_expected_nsa_ground_up_loss_2022_usd",
        "analytical_expected_total_ground_up_loss_2022_usd",
    ]
    for column in numeric_columns:
        chunk[column] = pd.to_numeric(chunk[column], errors="raise")

    occurrence_ordinals = chunk["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = chunk["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(occurrence_ordinals >= expected_occurrences):
        raise RuntimeError(f"Chunk {chunk_number} has occurrence ordinals outside range.")
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_buildings):
        raise RuntimeError(f"Chunk {chunk_number} has site ordinals outside range.")

    pair_indices = occurrence_ordinals * expected_buildings + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_number} contains duplicate occurrence-site pairs.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(f"Chunk {chunk_number} repeats an occurrence-site pair.")
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    replacement = chunk["building_replacement_value_2022_usd"].to_numpy(dtype=float)
    sampled_structural = chunk["sampled_structural_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    sampled_nsd = chunk["sampled_nsd_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    sampled_nsa = chunk["sampled_nsa_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    sampled_total = chunk["sampled_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    expected_structural = chunk[
        "analytical_expected_structural_ground_up_loss_2022_usd"
    ].to_numpy(dtype=float)
    expected_nsd = chunk[
        "analytical_expected_nsd_ground_up_loss_2022_usd"
    ].to_numpy(dtype=float)
    expected_nsa = chunk[
        "analytical_expected_nsa_ground_up_loss_2022_usd"
    ].to_numpy(dtype=float)
    expected_total = chunk[
        "analytical_expected_total_ground_up_loss_2022_usd"
    ].to_numpy(dtype=float)

    sampled_equation_error = np.abs(
        sampled_total - (sampled_structural + sampled_nsd + sampled_nsa)
    )
    expected_equation_error = np.abs(
        expected_total - (expected_structural + expected_nsd + expected_nsa)
    )
    maximum_sampled_equation_error = max(
        maximum_sampled_equation_error, float(sampled_equation_error.max(initial=0.0))
    )
    maximum_analytical_equation_error = max(
        maximum_analytical_equation_error, float(expected_equation_error.max(initial=0.0))
    )
    minimum_sampled_loss = min(minimum_sampled_loss, float(sampled_total.min(initial=np.inf)))
    minimum_analytical_loss = min(
        minimum_analytical_loss, float(expected_total.min(initial=np.inf))
    )
    positive_replacement = replacement > 0.0
    if positive_replacement.any():
        maximum_sampled_loss_ratio = max(
            maximum_sampled_loss_ratio,
            float(np.max(sampled_total[positive_replacement] / replacement[positive_replacement])),
        )

    sampled_total_sum += float(sampled_total.sum(dtype=np.float64))
    analytical_total_sum += float(expected_total.sum(dtype=np.float64))
    total_rows += len(chunk)
    source_types.update(chunk["source_type"].astype(str).str.strip().unique().tolist())

    exposure_columns = [
        "site_ordinal",
        "site_id",
        "structural_type",
        "design_level",
        "occ_type",
        "year_built",
        "floor_area_sqft",
        "replacement_cost_per_sqft_2022_usd",
        "building_replacement_value_2022_usd",
    ]
    for record in chunk[exposure_columns].drop_duplicates("site_ordinal").to_dict("records"):
        site_ordinal = int(record["site_ordinal"])
        normalized_record = {
            "site_ordinal": site_ordinal,
            "site_id": str(record["site_id"]),
            "structural_type": str(record["structural_type"]),
            "design_level": str(record["design_level"]),
            "occ_type": str(record["occ_type"]),
            "year_built": int(record["year_built"]),
            "floor_area_sqft": float(record["floor_area_sqft"]),
            "replacement_cost_per_sqft_2022_usd": float(
                record["replacement_cost_per_sqft_2022_usd"]
            ),
            "building_replacement_value_2022_usd": float(
                record["building_replacement_value_2022_usd"]
            ),
        }
        previous = site_records.get(site_ordinal)
        if previous is None:
            site_records[site_ordinal] = normalized_record
        else:
            text_columns = [
                "site_id",
                "structural_type",
                "design_level",
                "occ_type",
            ]
            numeric_exposure_columns = [
                "year_built",
                "floor_area_sqft",
                "replacement_cost_per_sqft_2022_usd",
                "building_replacement_value_2022_usd",
            ]
            if any(previous[column] != normalized_record[column] for column in text_columns):
                site_consistency_errors += 1
            if any(
                abs(float(previous[column]) - float(normalized_record[column]))
                > NUMERIC_TOLERANCE
                for column in numeric_exposure_columns
            ):
                site_consistency_errors += 1

append_check(
    validation_rows,
    "building_loss_row_count_matches_handoff",
    total_rows == expected_building_rows,
    f"actual={total_rows}; expected={expected_building_rows}",
)
append_check(
    validation_rows,
    "all_occurrence_site_pairs_present_once",
    bool(seen_pairs.all()),
    f"covered={int(seen_pairs.sum())}; expected={expected_building_rows}",
)
append_check(
    validation_rows,
    "every_occurrence_contains_complete_portfolio",
    bool(np.all(occurrence_counts == expected_buildings)),
    (
        f"minimum={int(occurrence_counts.min())}; "
        f"maximum={int(occurrence_counts.max())}; expected={expected_buildings}"
    ),
)
append_check(
    validation_rows,
    "every_building_appears_in_every_occurrence",
    bool(np.all(site_counts == expected_occurrences)),
    (
        f"minimum={int(site_counts.min())}; "
        f"maximum={int(site_counts.max())}; expected={expected_occurrences}"
    ),
)
append_check(
    validation_rows,
    "sampled_component_sum_matches_total_per_row",
    maximum_sampled_equation_error <= ROW_EQUATION_TOLERANCE_USD,
    f"maximum_error={maximum_sampled_equation_error:.3e}",
)
append_check(
    validation_rows,
    "analytical_component_sum_matches_total_per_row",
    maximum_analytical_equation_error <= ROW_EQUATION_TOLERANCE_USD,
    f"maximum_error={maximum_analytical_equation_error:.3e}",
)
append_check(
    validation_rows,
    "building_losses_are_nonnegative",
    minimum_sampled_loss >= -NUMERIC_TOLERANCE
    and minimum_analytical_loss >= -NUMERIC_TOLERANCE,
    (
        f"sampled_minimum={minimum_sampled_loss:.6f}; "
        f"analytical_minimum={minimum_analytical_loss:.6f}"
    ),
)
append_check(
    validation_rows,
    "sampled_building_loss_does_not_exceed_replacement_value",
    maximum_sampled_loss_ratio <= 1.0 + NUMERIC_TOLERANCE,
    f"maximum_ratio={maximum_sampled_loss_ratio:.12f}",
)
append_check(
    validation_rows,
    "sampled_total_reconciles_with_cell11",
    abs(sampled_total_sum - expected_sampled_total) <= AGGREGATE_TOLERANCE_USD,
    (
        f"streamed={sampled_total_sum:.6f}; recorded={expected_sampled_total:.6f}; "
        f"difference={sampled_total_sum - expected_sampled_total:.3e}"
    ),
)
append_check(
    validation_rows,
    "analytical_total_reconciles_with_cell11",
    abs(analytical_total_sum - expected_analytical_total) <= AGGREGATE_TOLERANCE_USD,
    (
        f"streamed={analytical_total_sum:.6f}; recorded={expected_analytical_total:.6f}; "
        f"difference={analytical_total_sum - expected_analytical_total:.3e}"
    ),
)
append_check(
    validation_rows,
    "building_exposure_is_constant_across_occurrences",
    site_consistency_errors == 0,
    f"consistency_errors={site_consistency_errors}",
)
append_check(
    validation_rows,
    "all_portfolio_buildings_resolved",
    len(site_records) == expected_buildings,
    f"resolved={len(site_records)}; expected={expected_buildings}",
)
append_check(
    validation_rows,
    "source_types_represented",
    len(source_types) >= 1,
    f"source_types={sorted(source_types)}",
)

exposure_base = pd.DataFrame(site_records.values()).sort_values("site_ordinal").reset_index(drop=True)
append_check(
    validation_rows,
    "exposure_site_ordinals_are_complete",
    exposure_base["site_ordinal"].tolist() == list(range(expected_buildings)),
    (
        f"minimum={int(exposure_base['site_ordinal'].min())}; "
        f"maximum={int(exposure_base['site_ordinal'].max())}; rows={len(exposure_base)}"
    ),
)
exposure_value = float(exposure_base["building_replacement_value_2022_usd"].sum())
append_check(
    validation_rows,
    "exposure_replacement_value_reconciles",
    abs(exposure_value - portfolio_replacement_value) <= AGGREGATE_TOLERANCE_USD,
    (
        f"exposure={exposure_value:.6f}; recorded={portfolio_replacement_value:.6f}; "
        f"difference={exposure_value - portfolio_replacement_value:.3e}"
    ),
)
exposure_base.to_csv(EXPOSURE_BASE_PATH, index=False)

required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "buildings",
    "portfolio_replacement_value_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
}
events = pd.read_csv(event_loss_path, compression="gzip")
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "occurrence_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    raise KeyError(f"Occurrence loss file is missing columns: {missing_event_columns}")
for column in [
    "catalog_year",
    "buildings",
    "portfolio_replacement_value_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "analytical_expected_total_ground_up_loss_2022_usd",
]:
    events[column] = pd.to_numeric(events[column], errors="raise")
append_check(
    validation_rows,
    "occurrence_loss_rows_match_catalog",
    len(events) == expected_occurrences,
    f"actual={len(events)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "occurrence_ids_are_unique",
    not events["occurrence_id"].astype(str).duplicated().any(),
    f"duplicates={int(events['occurrence_id'].astype(str).duplicated().sum())}",
)
append_check(
    validation_rows,
    "occurrence_portfolios_are_complete",
    events["buildings"].eq(expected_buildings).all(),
    (
        f"minimum={int(events['buildings'].min())}; "
        f"maximum={int(events['buildings'].max())}; expected={expected_buildings}"
    ),
)
append_check(
    validation_rows,
    "occurrence_sampled_total_reconciles",
    abs(float(events["sampled_total_ground_up_loss_2022_usd"].sum()) - sampled_total_sum)
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"event={events['sampled_total_ground_up_loss_2022_usd'].sum():.6f}; "
        f"building={sampled_total_sum:.6f}"
    ),
)
append_check(
    validation_rows,
    "occurrence_analytical_total_reconciles",
    abs(
        float(events["analytical_expected_total_ground_up_loss_2022_usd"].sum())
        - analytical_total_sum
    )
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"event={events['analytical_expected_total_ground_up_loss_2022_usd'].sum():.6f}; "
        f"building={analytical_total_sum:.6f}"
    ),
)

annual_required_columns = {
    "catalog_year",
    "event_count",
    "zero_event_year",
    "sampled_aep_total_ground_up_loss_2022_usd",
    "sampled_oep_total_ground_up_loss_2022_usd",
    "analytical_expected_aep_total_ground_up_loss_2022_usd",
    "analytical_expected_oep_total_ground_up_loss_2022_usd",
}
annual_header = pd.read_csv(annual_series_path, compression="gzip", nrows=0)
missing_annual_columns = sorted(annual_required_columns.difference(annual_header.columns))
append_check(
    validation_rows,
    "annual_loss_schema_complete",
    not missing_annual_columns,
    f"missing={missing_annual_columns}",
)
if missing_annual_columns:
    raise KeyError(f"Annual loss series is missing columns: {missing_annual_columns}")

annual_rows = 0
annual_sampled_sum = 0.0
annual_expected_sum = 0.0
annual_occupied_years = 0
annual_zero_event_years = 0
annual_multiple_event_years = 0
first_catalog_year: int | None = None
last_catalog_year: int | None = None
maximum_sampled_aep = 0.0
maximum_sampled_oep = 0.0

for chunk in pd.read_csv(
    annual_series_path,
    compression="gzip",
    usecols=sorted(annual_required_columns),
    chunksize=READ_CHUNK_ROWS,
):
    for column in [
        "catalog_year",
        "event_count",
        "sampled_aep_total_ground_up_loss_2022_usd",
        "sampled_oep_total_ground_up_loss_2022_usd",
        "analytical_expected_aep_total_ground_up_loss_2022_usd",
        "analytical_expected_oep_total_ground_up_loss_2022_usd",
    ]:
        chunk[column] = pd.to_numeric(chunk[column], errors="raise")
    if first_catalog_year is None:
        first_catalog_year = int(chunk["catalog_year"].iloc[0])
    last_catalog_year = int(chunk["catalog_year"].iloc[-1])
    event_count = chunk["event_count"].to_numpy(dtype=np.int64)
    sampled_aep = chunk["sampled_aep_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    sampled_oep = chunk["sampled_oep_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    expected_aep = chunk[
        "analytical_expected_aep_total_ground_up_loss_2022_usd"
    ].to_numpy(dtype=float)
    annual_rows += len(chunk)
    annual_sampled_sum += float(sampled_aep.sum(dtype=np.float64))
    annual_expected_sum += float(expected_aep.sum(dtype=np.float64))
    annual_occupied_years += int(np.count_nonzero(event_count > 0))
    annual_zero_event_years += int(np.count_nonzero(event_count == 0))
    annual_multiple_event_years += int(np.count_nonzero(event_count > 1))
    maximum_sampled_aep = max(maximum_sampled_aep, float(sampled_aep.max(initial=0.0)))
    maximum_sampled_oep = max(maximum_sampled_oep, float(sampled_oep.max(initial=0.0)))
    if np.any(sampled_aep + ROW_EQUATION_TOLERANCE_USD < sampled_oep):
        raise RuntimeError("Annual AEP is smaller than OEP for at least one year.")

annual_sampled_aal = annual_sampled_sum / declared_years
annual_expected_aal = annual_expected_sum / declared_years
append_check(
    validation_rows,
    "annual_series_contains_all_declared_years",
    annual_rows == declared_years and first_catalog_year == 1 and last_catalog_year == declared_years,
    (
        f"rows={annual_rows}; first={first_catalog_year}; last={last_catalog_year}; "
        f"expected_rows={declared_years}"
    ),
)
append_check(
    validation_rows,
    "annual_year_counts_reconcile_with_catalog",
    annual_occupied_years == expected_occupied_years
    and annual_zero_event_years == expected_zero_event_years
    and annual_multiple_event_years == expected_multiple_event_years,
    (
        f"occupied={annual_occupied_years}/{expected_occupied_years}; "
        f"zero={annual_zero_event_years}/{expected_zero_event_years}; "
        f"multiple={annual_multiple_event_years}/{expected_multiple_event_years}"
    ),
)
append_check(
    validation_rows,
    "annual_sampled_loss_total_reconciles",
    abs(annual_sampled_sum - sampled_total_sum) <= AGGREGATE_TOLERANCE_USD,
    f"annual={annual_sampled_sum:.6f}; occurrence={sampled_total_sum:.6f}",
)
append_check(
    validation_rows,
    "annual_analytical_loss_total_reconciles",
    abs(annual_expected_sum - analytical_total_sum) <= AGGREGATE_TOLERANCE_USD,
    f"annual={annual_expected_sum:.6f}; occurrence={analytical_total_sum:.6f}",
)
append_check(
    validation_rows,
    "sampled_aal_reconciles_with_cell12",
    abs(annual_sampled_aal - expected_sampled_aal) <= ROW_EQUATION_TOLERANCE_USD,
    (
        f"calculated={annual_sampled_aal:.9f}; "
        f"recorded={expected_sampled_aal:.9f}"
    ),
)
append_check(
    validation_rows,
    "analytical_aal_reconciles_with_cell12",
    abs(annual_expected_aal - expected_analytical_aal) <= ROW_EQUATION_TOLERANCE_USD,
    (
        f"calculated={annual_expected_aal:.9f}; "
        f"recorded={expected_analytical_aal:.9f}"
    ),
)

for path, label, required_columns in [
    (pml_path, "pml", {"return_period_years"}),
    (risk_metrics_path, "risk_metrics", {"loss_basis", "curve_type"}),
    (source_aal_path, "source_aal", {"source_type"}),
    (component_aal_path, "component_aal", {"component"}),
]:
    table = pd.read_csv(path)
    missing = sorted(required_columns.difference(table.columns))
    append_check(
        validation_rows,
        f"{label}_table_schema_complete",
        not missing and len(table) > 0,
        f"rows={len(table)}; missing={missing}",
    )

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(INPUT_INVENTORY_PATH, index=False)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
]
validation.to_csv(VALIDATION_PATH, index=False)

handoff = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "scope": {
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "input_loss_basis": (
            "sampled total ground-up building repair loss from Notebook 5, including "
            "structural, drift-sensitive nonstructural, and acceleration-sensitive "
            "nonstructural repair"
        ),
        "not_yet_applied": [
            "insurance take-up or coverage selection",
            "policy deductibles",
            "policy limits",
            "coinsurance",
            "reinsurance terms",
        ],
    },
    "annual_catalog": {
        "declared_duration_years": declared_years,
        "occurrences": expected_occurrences,
        "occupied_years": annual_occupied_years,
        "multiple_event_years": annual_multiple_event_years,
        "zero_event_years": annual_zero_event_years,
    },
    "portfolio": {
        "buildings": expected_buildings,
        "replacement_value_2022_usd": exposure_value,
        "occupancy_classes": sorted(exposure_base["occ_type"].astype(str).unique().tolist()),
        "design_levels": sorted(exposure_base["design_level"].astype(str).unique().tolist()),
    },
    "ground_up_loss": {
        "building_rows": total_rows,
        "sampled_total_2022_usd": sampled_total_sum,
        "analytical_expected_total_2022_usd": analytical_total_sum,
        "sampled_aal_2022_usd": annual_sampled_aal,
        "analytical_expected_aal_2022_usd": annual_expected_aal,
        "maximum_sampled_aep_2022_usd": maximum_sampled_aep,
        "maximum_sampled_oep_2022_usd": maximum_sampled_oep,
    },
    "accepted_notebook5_warnings": cell12_open_warnings.to_dict(orient="records"),
    "inputs": {
        row["role"]: {
            "path": row["path"],
            "sha256": row["sha256"],
            "size_bytes": row["size_bytes"],
            "recorded_rows": row["recorded_rows"],
        }
        for row in inventory_rows
    },
    "outputs": {
        "exposure_base": {
            "path": str(EXPOSURE_BASE_PATH),
            "sha256": sha256_file(EXPOSURE_BASE_PATH),
            "rows": int(len(exposure_base)),
        },
        "input_inventory": {
            "path": str(INPUT_INVENTORY_PATH),
            "sha256": sha256_file(INPUT_INVENTORY_PATH),
            "rows": int(len(inventory)),
        },
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
        "handoff": str(HANDOFF_PATH),
    },
    "next_cell": (
        "Cell 2: define transparent baseline insurance coverage assumptions and "
        "create one validated policy-term record for each portfolio building."
    ),
}
write_json(HANDOFF_PATH, handoff)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "critical_checks": int(validation["severity"].astype(str).str.lower().eq("critical").sum()),
    "critical_failures": critical_failures.to_dict(orient="records"),
    "warnings": warnings.to_dict(orient="records"),
    "annual_catalog": handoff["annual_catalog"],
    "portfolio": handoff["portfolio"],
    "ground_up_loss": handoff["ground_up_loss"],
    "outputs": handoff["outputs"],
    "next_cell": handoff["next_cell"],
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 1 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 1 INPUT AND NOTEBOOK 5 HANDOFF VALIDATION COMPLETE")
print("=" * 78)
print(f"Ground-up building rows:       {total_rows:,}")
print(f"Catalog occurrences:           {expected_occurrences:,}")
print(f"Declared catalog years:        {declared_years:,}")
print(f"Portfolio buildings:           {expected_buildings:,}")
print(f"Portfolio replacement value:   ${exposure_value:,.0f} ({DOLLAR_YEAR} USD)")
print(f"Sampled ground-up AAL:         ${annual_sampled_aal:,.2f}")
print(f"Analytical expected AAL:       ${annual_expected_aal:,.2f}")
print(f"Maximum sampled AEP loss:      ${maximum_sampled_aep:,.0f}")
print(f"Maximum sampled OEP loss:      ${maximum_sampled_oep:,.0f}")
print(f"Critical validation checks:    {len(validation.loc[validation['severity'].eq('critical')]):,}")
print(f"Critical failures:             {len(critical_failures):,}")
print(f"Warnings requiring review:     {len(warnings):,}")
print()
print("Insurance exposure base:")
print(f"  {EXPOSURE_BASE_PATH}")
print("Notebook 6 input handoff:")
print(f"  {HANDOFF_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: define transparent baseline policy coverage, deductible, limit, and coinsurance assumptions.")



NOTEBOOK 6 CELL 1 INPUT AND NOTEBOOK 5 HANDOFF VALIDATION COMPLETE
Ground-up building rows:       4,996,100
Catalog occurrences:           10,630
Declared catalog years:        2,000,000
Portfolio buildings:           470
Portfolio replacement value:   $384,236,605 (2022 USD)
Sampled ground-up AAL:         $195,922.45
Analytical expected AAL:       $196,250.58
Maximum sampled AEP loss:      $282,749,468
Maximum sampled OEP loss:      $282,749,468
Critical validation checks:    57
Critical failures:             0
Warnings requiring review:     0

Insurance exposure base:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_insurance_parameters\seaside_w2_insurance_exposure_base.csv
Notebook 6 input handoff:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_6_insurance_terms\notebook_6_input_handoff.json
Validation:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_

In [2]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell2_baseline_policy_terms_v1"
DOLLAR_YEAR = 2022
NUMERIC_TOLERANCE = 1e-10
AGGREGATE_TOLERANCE_USD = 0.01

SCENARIO_ID = "baseline_full_coverage_10pct_deductible_v1"
SCENARIO_NAME = "Full building coverage with 10% replacement-value deductible"
COVERAGE_TAKE_UP = 1.00
COVERED_LOSS_SHARE = 1.00
DEDUCTIBLE_PERCENT_OF_REPLACEMENT = 0.10
POLICY_LIMIT_PERCENT_OF_REPLACEMENT = 1.00
COINSURANCE_SHARE = 1.00


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_input_handoff.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_insurance_parameters"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
PARAMETER_DIR.mkdir(parents=True, exist_ok=True)

CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_1_validation.csv"
CELL1_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_1_summary.json"
CELL1_HANDOFF_PATH = METADATA_DIR / "notebook_6_input_handoff.json"
EXPOSURE_BASE_PATH = PARAMETER_DIR / "seaside_w2_insurance_exposure_base.csv"

POLICY_TERMS_PATH = PARAMETER_DIR / "seaside_w2_baseline_policy_terms.csv"
ASSUMPTION_REGISTRY_PATH = PARAMETER_DIR / "notebook_6_baseline_policy_assumptions.csv"
FORMULA_SPECIFICATION_PATH = (
    METADATA_DIR / "notebook_6_cell_2_policy_formula_specification.json"
)
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_2_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_2_summary.json"

validation_rows: list[dict[str, Any]] = []

required_inputs = [
    CELL1_VALIDATION_PATH,
    CELL1_SUMMARY_PATH,
    CELL1_HANDOFF_PATH,
    EXPOSURE_BASE_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required Notebook 6 inputs are missing: {missing_inputs}")

cell1_validation = pd.read_csv(CELL1_VALIDATION_PATH)
required_validation_columns = {"check_id", "severity", "passed", "detail"}
missing_validation_columns = sorted(
    required_validation_columns.difference(cell1_validation.columns)
)
append_check(
    validation_rows,
    "cell1_validation_schema_complete",
    not missing_validation_columns,
    f"missing={missing_validation_columns}",
)
if missing_validation_columns:
    raise KeyError(
        f"Cell 1 validation is missing columns: {missing_validation_columns}"
    )

cell1_validation["passed"] = parse_bool_series(cell1_validation["passed"])
cell1_failures = cell1_validation.loc[
    cell1_validation["severity"].astype(str).str.lower().eq("critical")
    & ~cell1_validation["passed"]
]
cell1_warnings = cell1_validation.loc[
    cell1_validation["severity"].astype(str).str.lower().eq("warning")
    & ~cell1_validation["passed"]
]
append_check(
    validation_rows,
    "cell1_critical_validation_passed",
    cell1_failures.empty,
    f"critical_failures={len(cell1_failures)}",
)
append_check(
    validation_rows,
    "cell1_has_no_unresolved_warnings",
    cell1_warnings.empty,
    f"warnings={len(cell1_warnings)}",
)

cell1_summary = load_json(CELL1_SUMMARY_PATH)
cell1_handoff = load_json(CELL1_HANDOFF_PATH)
append_check(
    validation_rows,
    "cell1_summary_records_success",
    bool(cell1_summary.get("all_critical_checks_passed")),
    f"all_critical_checks_passed={cell1_summary.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell1_handoff_records_success",
    bool(cell1_handoff.get("all_critical_checks_passed")),
    f"all_critical_checks_passed={cell1_handoff.get('all_critical_checks_passed')}",
)

exposure = pd.read_csv(EXPOSURE_BASE_PATH)
required_exposure_columns = {
    "site_ordinal",
    "site_id",
    "structural_type",
    "design_level",
    "occ_type",
    "year_built",
    "floor_area_sqft",
    "replacement_cost_per_sqft_2022_usd",
    "building_replacement_value_2022_usd",
}
missing_exposure_columns = sorted(required_exposure_columns.difference(exposure.columns))
append_check(
    validation_rows,
    "exposure_schema_complete",
    not missing_exposure_columns,
    f"missing={missing_exposure_columns}",
)
if missing_exposure_columns:
    raise KeyError(f"Exposure base is missing columns: {missing_exposure_columns}")

numeric_columns = [
    "site_ordinal",
    "year_built",
    "floor_area_sqft",
    "replacement_cost_per_sqft_2022_usd",
    "building_replacement_value_2022_usd",
]
for column in numeric_columns:
    exposure[column] = pd.to_numeric(exposure[column], errors="raise")

text_columns = ["site_id", "structural_type", "design_level", "occ_type"]
for column in text_columns:
    exposure[column] = exposure[column].astype(str).str.strip()

expected_buildings = int(
    cell1_handoff.get(
        "portfolio",
        {},
    ).get("buildings", len(exposure))
)
if expected_buildings == len(exposure):
    expected_buildings = int(
        cell1_summary.get("portfolio", {}).get("buildings", expected_buildings)
    )

site_ordinals = exposure["site_ordinal"].to_numpy(dtype=np.int64)
replacement_values = exposure[
    "building_replacement_value_2022_usd"
].to_numpy(dtype=float)

append_check(
    validation_rows,
    "exposure_rows_match_portfolio",
    len(exposure) == expected_buildings,
    f"actual={len(exposure)}; expected={expected_buildings}",
)
append_check(
    validation_rows,
    "site_ordinals_unique",
    not exposure["site_ordinal"].duplicated().any(),
    f"duplicates={int(exposure['site_ordinal'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "site_ids_unique",
    not exposure["site_id"].duplicated().any(),
    f"duplicates={int(exposure['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "site_ordinals_complete",
    np.array_equal(np.sort(site_ordinals), np.arange(len(exposure), dtype=np.int64)),
    (
        f"minimum={int(site_ordinals.min())}; maximum={int(site_ordinals.max())}; "
        f"rows={len(exposure)}"
    ),
)
append_check(
    validation_rows,
    "exposure_text_fields_nonempty",
    all(exposure[column].ne("").all() for column in text_columns),
    "checked=site_id,structural_type,design_level,occ_type",
)
append_check(
    validation_rows,
    "replacement_values_positive_and_finite",
    bool(np.isfinite(replacement_values).all() and np.all(replacement_values > 0.0)),
    (
        f"minimum={replacement_values.min():.6f}; "
        f"maximum={replacement_values.max():.6f}"
    ),
)

recorded_portfolio_value = float(
    cell1_handoff.get("portfolio", {}).get(
        "replacement_value_2022_usd",
        cell1_summary.get("portfolio", {}).get(
            "replacement_value_2022_usd",
            replacement_values.sum(dtype=np.float64),
        ),
    )
)
calculated_portfolio_value = float(replacement_values.sum(dtype=np.float64))
append_check(
    validation_rows,
    "exposure_value_reconciles_with_cell1",
    abs(calculated_portfolio_value - recorded_portfolio_value)
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"calculated={calculated_portfolio_value:.6f}; "
        f"recorded={recorded_portfolio_value:.6f}; "
        f"difference={calculated_portfolio_value - recorded_portfolio_value:.3e}"
    ),
)

assumption_values = {
    "coverage_take_up": COVERAGE_TAKE_UP,
    "covered_loss_share": COVERED_LOSS_SHARE,
    "deductible_percent_of_replacement": DEDUCTIBLE_PERCENT_OF_REPLACEMENT,
    "policy_limit_percent_of_replacement": POLICY_LIMIT_PERCENT_OF_REPLACEMENT,
    "coinsurance_share": COINSURANCE_SHARE,
}
append_check(
    validation_rows,
    "all_policy_shares_are_within_zero_and_one",
    all(0.0 <= value <= 1.0 for value in assumption_values.values()),
    f"values={assumption_values}",
)
append_check(
    validation_rows,
    "baseline_has_full_policy_take_up",
    abs(COVERAGE_TAKE_UP - 1.0) <= NUMERIC_TOLERANCE,
    f"coverage_take_up={COVERAGE_TAKE_UP:.6f}",
)
append_check(
    validation_rows,
    "baseline_has_full_eligible_building_repair_coverage",
    abs(COVERED_LOSS_SHARE - 1.0) <= NUMERIC_TOLERANCE,
    f"covered_loss_share={COVERED_LOSS_SHARE:.6f}",
)
append_check(
    validation_rows,
    "deductible_is_less_than_policy_limit",
    DEDUCTIBLE_PERCENT_OF_REPLACEMENT
    < POLICY_LIMIT_PERCENT_OF_REPLACEMENT - NUMERIC_TOLERANCE,
    (
        f"deductible_percent={DEDUCTIBLE_PERCENT_OF_REPLACEMENT:.6f}; "
        f"limit_percent={POLICY_LIMIT_PERCENT_OF_REPLACEMENT:.6f}"
    ),
)

policy_terms = exposure.copy()
policy_terms.insert(0, "policy_scenario_id", SCENARIO_ID)
policy_terms.insert(1, "policy_scenario_name", SCENARIO_NAME)
policy_terms.insert(
    2,
    "policy_id",
    [f"EQ-BLDG-{ordinal:04d}" for ordinal in site_ordinals],
)
policy_terms["policy_covered"] = True
policy_terms["coverage_take_up_share"] = COVERAGE_TAKE_UP
policy_terms["covered_loss_share"] = COVERED_LOSS_SHARE
policy_terms["insured_value_2022_usd"] = replacement_values * COVERAGE_TAKE_UP
policy_terms["deductible_type"] = "percentage_of_building_replacement_value"
policy_terms["deductible_percent"] = DEDUCTIBLE_PERCENT_OF_REPLACEMENT
policy_terms["deductible_amount_2022_usd"] = (
    replacement_values * DEDUCTIBLE_PERCENT_OF_REPLACEMENT
)
policy_terms["policy_limit_type"] = "maximum_insurer_payment_per_occurrence"
policy_terms["policy_limit_percent"] = POLICY_LIMIT_PERCENT_OF_REPLACEMENT
policy_terms["policy_limit_amount_2022_usd"] = (
    replacement_values * POLICY_LIMIT_PERCENT_OF_REPLACEMENT
)
policy_terms["coinsurance_share"] = COINSURANCE_SHARE
policy_terms["deductible_application"] = "per_building_per_occurrence"
policy_terms["policy_limit_application"] = "per_building_per_occurrence"
policy_terms["currency"] = "USD"
policy_terms["dollar_year"] = DOLLAR_YEAR
policy_terms["assumption_source"] = "synthetic_project_baseline"

policy_ids = policy_terms["policy_id"].astype(str)
deductibles = policy_terms["deductible_amount_2022_usd"].to_numpy(dtype=float)
limits = policy_terms["policy_limit_amount_2022_usd"].to_numpy(dtype=float)
insured_values = policy_terms["insured_value_2022_usd"].to_numpy(dtype=float)
coinsurance = policy_terms["coinsurance_share"].to_numpy(dtype=float)
covered_loss_share = policy_terms["covered_loss_share"].to_numpy(dtype=float)

append_check(
    validation_rows,
    "policy_ids_unique",
    not policy_ids.duplicated().any(),
    f"duplicates={int(policy_ids.duplicated().sum())}",
)
append_check(
    validation_rows,
    "every_building_has_policy_terms",
    len(policy_terms) == len(exposure)
    and policy_terms["policy_covered"].astype(bool).all(),
    f"policy_rows={len(policy_terms)}; exposure_rows={len(exposure)}",
)
append_check(
    validation_rows,
    "policy_numeric_terms_finite",
    bool(
        np.isfinite(deductibles).all()
        and np.isfinite(limits).all()
        and np.isfinite(insured_values).all()
        and np.isfinite(coinsurance).all()
        and np.isfinite(covered_loss_share).all()
    ),
    "checked=deductible,limit,insured_value,coinsurance,covered_loss_share",
)
append_check(
    validation_rows,
    "policy_numeric_terms_nonnegative",
    bool(
        np.all(deductibles >= 0.0)
        and np.all(limits >= 0.0)
        and np.all(insured_values >= 0.0)
    ),
    (
        f"minimum_deductible={deductibles.min():.6f}; "
        f"minimum_limit={limits.min():.6f}"
    ),
)
append_check(
    validation_rows,
    "insured_values_equal_replacement_values",
    float(np.max(np.abs(insured_values - replacement_values), initial=0.0))
    <= NUMERIC_TOLERANCE,
    (
        f"maximum_error={np.max(np.abs(insured_values - replacement_values), initial=0.0):.3e}"
    ),
)
append_check(
    validation_rows,
    "deductible_amounts_follow_assumption",
    float(
        np.max(
            np.abs(
                deductibles
                - replacement_values * DEDUCTIBLE_PERCENT_OF_REPLACEMENT
            ),
            initial=0.0,
        )
    )
    <= NUMERIC_TOLERANCE,
    (
        "maximum_error="
        f"{np.max(np.abs(deductibles - replacement_values * DEDUCTIBLE_PERCENT_OF_REPLACEMENT), initial=0.0):.3e}"
    ),
)
append_check(
    validation_rows,
    "policy_limits_follow_assumption",
    float(
        np.max(
            np.abs(
                limits - replacement_values * POLICY_LIMIT_PERCENT_OF_REPLACEMENT
            ),
            initial=0.0,
        )
    )
    <= NUMERIC_TOLERANCE,
    (
        "maximum_error="
        f"{np.max(np.abs(limits - replacement_values * POLICY_LIMIT_PERCENT_OF_REPLACEMENT), initial=0.0):.3e}"
    ),
)
append_check(
    validation_rows,
    "deductibles_are_below_limits_for_all_buildings",
    bool(np.all(deductibles < limits - NUMERIC_TOLERANCE)),
    (
        f"minimum_margin={np.min(limits - deductibles):.6f}; "
        f"maximum_margin={np.max(limits - deductibles):.6f}"
    ),
)
append_check(
    validation_rows,
    "coinsurance_is_full_for_baseline",
    bool(np.all(np.abs(coinsurance - 1.0) <= NUMERIC_TOLERANCE)),
    f"unique_values={sorted(np.unique(coinsurance).tolist())}",
)
append_check(
    validation_rows,
    "covered_loss_share_is_full_for_baseline",
    bool(np.all(np.abs(covered_loss_share - 1.0) <= NUMERIC_TOLERANCE)),
    f"unique_values={sorted(np.unique(covered_loss_share).tolist())}",
)

portfolio_deductible = float(deductibles.sum(dtype=np.float64))
portfolio_policy_limit = float(limits.sum(dtype=np.float64))
portfolio_insured_value = float(insured_values.sum(dtype=np.float64))
append_check(
    validation_rows,
    "portfolio_insured_value_reconciles",
    abs(portfolio_insured_value - calculated_portfolio_value)
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"insured_value={portfolio_insured_value:.6f}; "
        f"replacement_value={calculated_portfolio_value:.6f}"
    ),
)
append_check(
    validation_rows,
    "portfolio_deductible_reconciles",
    abs(
        portfolio_deductible
        - calculated_portfolio_value * DEDUCTIBLE_PERCENT_OF_REPLACEMENT
    )
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"deductible={portfolio_deductible:.6f}; "
        f"expected={calculated_portfolio_value * DEDUCTIBLE_PERCENT_OF_REPLACEMENT:.6f}"
    ),
)
append_check(
    validation_rows,
    "portfolio_policy_limit_reconciles",
    abs(
        portfolio_policy_limit
        - calculated_portfolio_value * POLICY_LIMIT_PERCENT_OF_REPLACEMENT
    )
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"limit={portfolio_policy_limit:.6f}; "
        f"expected={calculated_portfolio_value * POLICY_LIMIT_PERCENT_OF_REPLACEMENT:.6f}"
    ),
)

assumption_registry = pd.DataFrame(
    [
        {
            "policy_scenario_id": SCENARIO_ID,
            "assumption_name": "policy_take_up_share",
            "value": COVERAGE_TAKE_UP,
            "unit": "fraction",
            "application": "all portfolio buildings",
            "source_type": "synthetic_project_baseline",
            "rationale": (
                "All buildings are treated as insured to isolate policy and reinsurance "
                "mechanics before later take-up sensitivity analysis."
            ),
        },
        {
            "policy_scenario_id": SCENARIO_ID,
            "assumption_name": "covered_loss_share",
            "value": COVERED_LOSS_SHARE,
            "unit": "fraction",
            "application": "sampled total building repair ground-up loss",
            "source_type": "synthetic_project_baseline",
            "rationale": (
                "Structural, drift-sensitive nonstructural, and acceleration-sensitive "
                "nonstructural repair losses are eligible under the baseline building policy."
            ),
        },
        {
            "policy_scenario_id": SCENARIO_ID,
            "assumption_name": "deductible_percent_of_replacement_value",
            "value": DEDUCTIBLE_PERCENT_OF_REPLACEMENT,
            "unit": "fraction",
            "application": "per building per occurrence",
            "source_type": "synthetic_project_baseline",
            "rationale": (
                "A transparent percentage deductible is used as a demonstration assumption, "
                "not as an empirical estimate of the actual Seaside insurance portfolio."
            ),
        },
        {
            "policy_scenario_id": SCENARIO_ID,
            "assumption_name": "policy_limit_percent_of_replacement_value",
            "value": POLICY_LIMIT_PERCENT_OF_REPLACEMENT,
            "unit": "fraction",
            "application": "maximum insurer payment per building per occurrence",
            "source_type": "synthetic_project_baseline",
            "rationale": (
                "The baseline uses full replacement-value limits so the first analysis "
                "emphasizes deductible effects without introducing unsupported underinsurance."
            ),
        },
        {
            "policy_scenario_id": SCENARIO_ID,
            "assumption_name": "coinsurance_share",
            "value": COINSURANCE_SHARE,
            "unit": "fraction",
            "application": "after deductible and policy limit",
            "source_type": "synthetic_project_baseline",
            "rationale": (
                "Full coinsurance is used for the baseline. Alternative shares can be added "
                "later as explicit sensitivity cases."
            ),
        },
    ]
)

formula_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_scenario_id": SCENARIO_ID,
    "policy_scenario_name": SCENARIO_NAME,
    "currency": "USD",
    "dollar_year": DOLLAR_YEAR,
    "loss_basis": (
        "sampled total building repair ground-up loss from Notebook 5, including "
        "structural, drift-sensitive nonstructural, and acceleration-sensitive "
        "nonstructural repair"
    ),
    "terms": {
        "policy_take_up_share": COVERAGE_TAKE_UP,
        "covered_loss_share": COVERED_LOSS_SHARE,
        "deductible_percent_of_replacement_value": (
            DEDUCTIBLE_PERCENT_OF_REPLACEMENT
        ),
        "policy_limit_percent_of_replacement_value": (
            POLICY_LIMIT_PERCENT_OF_REPLACEMENT
        ),
        "coinsurance_share": COINSURANCE_SHARE,
        "deductible_application": "per building per occurrence",
        "policy_limit_application": "per building per occurrence",
    },
    "future_cell_3_equations": {
        "eligible_loss": "ground_up_loss * covered_loss_share if policy_covered else 0",
        "loss_after_deductible": "max(eligible_loss - deductible_amount, 0)",
        "limited_loss": "min(loss_after_deductible, policy_limit_amount)",
        "gross_insured_loss": "limited_loss * coinsurance_share",
        "uninsured_loss": "ground_up_loss - gross_insured_loss",
        "deductible_absorbed_loss": "min(eligible_loss, deductible_amount)",
        "limit_absorbed_loss": "max(loss_after_deductible - policy_limit_amount, 0)",
        "coinsurance_absorbed_loss": "limited_loss * (1 - coinsurance_share)",
    },
    "important_scope_notes": [
        "These terms are transparent synthetic project assumptions, not observed policy data.",
        "The baseline assumes 100% take-up and full replacement-value limits.",
        "The baseline includes building repair loss only and excludes contents, business interruption, additional living expense, demand surge, taxes, and claims adjustment expenses.",
        "No annual aggregate deductible, franchise deductible, waiting period, or policy sublimit is applied.",
        "Reinsurance terms are not introduced in Cell 2.",
        "Because Notebook 5 caps building ground-up loss at replacement value and the baseline limit equals replacement value, the policy limit is not expected to bind in the baseline production run.",
    ],
}

policy_terms.to_csv(POLICY_TERMS_PATH, index=False)
assumption_registry.to_csv(ASSUMPTION_REGISTRY_PATH, index=False)
write_json(FORMULA_SPECIFICATION_PATH, formula_specification)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
]
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "policy_scenario": {
        "policy_scenario_id": SCENARIO_ID,
        "policy_scenario_name": SCENARIO_NAME,
        "assumption_source": "synthetic_project_baseline",
        "policy_take_up_share": COVERAGE_TAKE_UP,
        "covered_loss_share": COVERED_LOSS_SHARE,
        "deductible_percent_of_replacement_value": (
            DEDUCTIBLE_PERCENT_OF_REPLACEMENT
        ),
        "policy_limit_percent_of_replacement_value": (
            POLICY_LIMIT_PERCENT_OF_REPLACEMENT
        ),
        "coinsurance_share": COINSURANCE_SHARE,
    },
    "portfolio": {
        "buildings": int(len(policy_terms)),
        "occupancy_classes": sorted(policy_terms["occ_type"].unique().tolist()),
        "design_levels": sorted(policy_terms["design_level"].unique().tolist()),
        "replacement_value_2022_usd": calculated_portfolio_value,
        "insured_value_2022_usd": portfolio_insured_value,
        "aggregate_building_deductible_2022_usd": portfolio_deductible,
        "aggregate_policy_limit_2022_usd": portfolio_policy_limit,
        "minimum_building_deductible_2022_usd": float(deductibles.min()),
        "maximum_building_deductible_2022_usd": float(deductibles.max()),
        "minimum_building_policy_limit_2022_usd": float(limits.min()),
        "maximum_building_policy_limit_2022_usd": float(limits.max()),
    },
    "validation": {
        "checks": int(len(validation)),
        "critical_failures": int(len(critical_failures)),
        "warnings_requiring_review": int(len(warnings)),
    },
    "outputs": {
        "policy_terms": {
            "path": str(POLICY_TERMS_PATH),
            "sha256": sha256_file(POLICY_TERMS_PATH),
            "rows": int(len(policy_terms)),
        },
        "assumption_registry": {
            "path": str(ASSUMPTION_REGISTRY_PATH),
            "sha256": sha256_file(ASSUMPTION_REGISTRY_PATH),
            "rows": int(len(assumption_registry)),
        },
        "formula_specification": {
            "path": str(FORMULA_SPECIFICATION_PATH),
            "sha256": sha256_file(FORMULA_SPECIFICATION_PATH),
        },
        "validation": {
            "path": str(VALIDATION_PATH),
            "sha256": sha256_file(VALIDATION_PATH),
        },
    },
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 2 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 2 BASELINE POLICY TERMS COMPLETE")
print("=" * 78)
print(f"Policy scenario:                 {SCENARIO_ID}")
print(f"Portfolio buildings:             {len(policy_terms):,}")
print(f"Policy take-up:                   {COVERAGE_TAKE_UP:.1%}")
print(f"Covered building repair share:   {COVERED_LOSS_SHARE:.1%}")
print(f"Deductible:                       {DEDUCTIBLE_PERCENT_OF_REPLACEMENT:.1%} of replacement value")
print(f"Policy limit:                     {POLICY_LIMIT_PERCENT_OF_REPLACEMENT:.1%} of replacement value")
print(f"Coinsurance:                      {COINSURANCE_SHARE:.1%}")
print(f"Portfolio replacement value:      ${calculated_portfolio_value:,.0f}")
print(f"Aggregate building deductibles:   ${portfolio_deductible:,.0f}")
print(f"Aggregate policy limits:          ${portfolio_policy_limit:,.0f}")
print(f"Critical validation checks:       {len(validation):,}")
print(f"Critical failures:                {len(critical_failures):,}")
print(f"Warnings requiring review:        {len(warnings):,}")
print()
print("Building policy terms:")
print(f"  {POLICY_TERMS_PATH}")
print("Assumption registry:")
print(f"  {ASSUMPTION_REGISTRY_PATH}")
print("Policy formula specification:")
print(f"  {FORMULA_SPECIFICATION_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: controlled gross-insured and uninsured loss calculation validation.")



NOTEBOOK 6 CELL 2 BASELINE POLICY TERMS COMPLETE
Policy scenario:                 baseline_full_coverage_10pct_deductible_v1
Portfolio buildings:             470
Policy take-up:                   100.0%
Covered building repair share:   100.0%
Deductible:                       10.0% of replacement value
Policy limit:                     100.0% of replacement value
Coinsurance:                      100.0%
Portfolio replacement value:      $384,236,605
Aggregate building deductibles:   $38,423,660
Aggregate policy limits:          $384,236,605
Critical validation checks:       34
Critical failures:                0
Warnings requiring review:        0

Building policy terms:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_insurance_parameters\seaside_w2_baseline_policy_terms.csv
Assumption registry:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_insurance_parameters\notebook_6_baseline_policy_assu

In [3]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell3_controlled_policy_loss_validation_v1"
DOLLAR_YEAR = 2022
READ_CHUNK_ROWS = 250_000
ROW_TOLERANCE_USD = 1e-6
AGGREGATE_TOLERANCE_USD = 0.01
NUMERIC_TOLERANCE = 1e-10

ACTUAL_TARGET_LOSS_RATIOS = np.array(
    [0.00, 0.025, 0.10, 0.25, 0.50, 0.75, 1.00], dtype=float
)
STRESS_LOSS_RATIOS = np.array(
    [0.00, 0.025, 0.05, 0.099999, 0.10, 0.100001, 0.25, 0.50, 1.00, 1.10, 1.25],
    dtype=float,
)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_input_handoff.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def resolve_path(value: str | Path, project_root: Path) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return project_root / path


def apply_policy_terms(
    ground_up_loss: np.ndarray,
    policy_covered: np.ndarray,
    covered_loss_share: np.ndarray,
    deductible_amount: np.ndarray,
    policy_limit_amount: np.ndarray,
    coinsurance_share: np.ndarray,
) -> dict[str, np.ndarray]:
    ground_up = np.asarray(ground_up_loss, dtype=float)
    covered = np.asarray(policy_covered, dtype=bool)
    covered_share = np.asarray(covered_loss_share, dtype=float)
    deductible = np.asarray(deductible_amount, dtype=float)
    policy_limit = np.asarray(policy_limit_amount, dtype=float)
    coinsurance = np.asarray(coinsurance_share, dtype=float)

    if np.any(~np.isfinite(ground_up)) or np.any(ground_up < 0.0):
        raise ValueError("Ground-up losses must be finite and nonnegative.")

    eligible_loss = np.where(covered, ground_up * covered_share, 0.0)
    uncovered_loss = ground_up - eligible_loss
    deductible_absorbed_loss = np.minimum(eligible_loss, deductible)
    loss_after_deductible = np.maximum(eligible_loss - deductible, 0.0)
    limited_loss = np.minimum(loss_after_deductible, policy_limit)
    policy_limit_absorbed_loss = np.maximum(
        loss_after_deductible - policy_limit, 0.0
    )
    gross_insured_loss = limited_loss * coinsurance
    coinsurance_absorbed_loss = limited_loss * (1.0 - coinsurance)
    uninsured_loss = ground_up - gross_insured_loss
    decomposed_uninsured_loss = (
        uncovered_loss
        + deductible_absorbed_loss
        + policy_limit_absorbed_loss
        + coinsurance_absorbed_loss
    )

    return {
        "eligible_ground_up_loss_2022_usd": eligible_loss,
        "uncovered_ground_up_loss_2022_usd": uncovered_loss,
        "deductible_absorbed_loss_2022_usd": deductible_absorbed_loss,
        "loss_after_deductible_2022_usd": loss_after_deductible,
        "limited_loss_2022_usd": limited_loss,
        "policy_limit_absorbed_loss_2022_usd": policy_limit_absorbed_loss,
        "coinsurance_absorbed_loss_2022_usd": coinsurance_absorbed_loss,
        "gross_insured_loss_2022_usd": gross_insured_loss,
        "uninsured_loss_2022_usd": uninsured_loss,
        "decomposed_uninsured_loss_2022_usd": decomposed_uninsured_loss,
    }


def add_policy_outputs(frame: pd.DataFrame, ground_up_column: str) -> pd.DataFrame:
    result = frame.copy()
    outputs = apply_policy_terms(
        ground_up_loss=result[ground_up_column].to_numpy(dtype=float),
        policy_covered=result["policy_covered"].to_numpy(dtype=bool),
        covered_loss_share=result["covered_loss_share"].to_numpy(dtype=float),
        deductible_amount=result["deductible_amount_2022_usd"].to_numpy(dtype=float),
        policy_limit_amount=result["policy_limit_amount_2022_usd"].to_numpy(dtype=float),
        coinsurance_share=result["coinsurance_share"].to_numpy(dtype=float),
    )
    for column, values in outputs.items():
        result[column] = values
    return result


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_insurance_parameters"
CONTROLLED_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_controlled_insurance_terms"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
CONTROLLED_DIR.mkdir(parents=True, exist_ok=True)

CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_1_validation.csv"
CELL1_HANDOFF_PATH = METADATA_DIR / "notebook_6_input_handoff.json"
CELL2_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_2_validation.csv"
CELL2_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_2_summary.json"
CELL2_FORMULA_PATH = METADATA_DIR / "notebook_6_cell_2_policy_formula_specification.json"
POLICY_TERMS_PATH = PARAMETER_DIR / "seaside_w2_baseline_policy_terms.csv"

ACTUAL_CONTROLS_PATH = CONTROLLED_DIR / "controlled_actual_policy_loss_rows.csv"
STRESS_CASES_PATH = CONTROLLED_DIR / "controlled_policy_formula_stress_cases.csv"
MONOTONICITY_GRID_PATH = CONTROLLED_DIR / "controlled_policy_monotonicity_grid.csv"
FORMULA_SPECIFICATION_PATH = METADATA_DIR / "notebook_6_cell_3_insurance_loss_formula_specification.json"
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_3_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_3_summary.json"

validation_rows: list[dict[str, Any]] = []
required_inputs = [
    CELL1_VALIDATION_PATH,
    CELL1_HANDOFF_PATH,
    CELL2_VALIDATION_PATH,
    CELL2_SUMMARY_PATH,
    CELL2_FORMULA_PATH,
    POLICY_TERMS_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required Notebook 6 inputs are missing: {missing_inputs}")

for validation_path, label in [
    (CELL1_VALIDATION_PATH, "cell1"),
    (CELL2_VALIDATION_PATH, "cell2"),
]:
    upstream = pd.read_csv(validation_path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(upstream.columns))
    append_check(
        validation_rows,
        f"{label}_validation_schema_complete",
        not missing,
        f"missing={missing}",
    )
    if missing:
        raise KeyError(f"{label} validation is missing columns: {missing}")
    upstream["passed"] = parse_bool_series(upstream["passed"])
    failures = upstream.loc[
        upstream["severity"].astype(str).str.lower().eq("critical")
        & ~upstream["passed"]
    ]
    warnings = upstream.loc[
        upstream["severity"].astype(str).str.lower().eq("warning")
        & ~upstream["passed"]
    ]
    append_check(
        validation_rows,
        f"{label}_critical_validation_passed",
        failures.empty,
        f"critical_failures={len(failures)}",
    )
    append_check(
        validation_rows,
        f"{label}_has_no_unresolved_warnings",
        warnings.empty,
        f"warnings={len(warnings)}",
    )

cell1_handoff = load_json(CELL1_HANDOFF_PATH)
cell2_summary = load_json(CELL2_SUMMARY_PATH)
cell2_formula = load_json(CELL2_FORMULA_PATH)
append_check(
    validation_rows,
    "cell1_handoff_records_success",
    bool(cell1_handoff.get("all_critical_checks_passed")),
    f"success={cell1_handoff.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell2_summary_records_success",
    bool(cell2_summary.get("all_critical_checks_passed")),
    f"success={cell2_summary.get('all_critical_checks_passed')}",
)

full_loss_record = cell1_handoff.get("inputs", {}).get(
    "full_building_ground_up_loss", {}
)
full_loss_path_value = full_loss_record.get("path")
if not full_loss_path_value:
    raise KeyError(
        "Notebook 6 Cell 1 handoff does not contain the full building ground-up loss path."
    )
FULL_GROUND_UP_LOSS_PATH = resolve_path(full_loss_path_value, PROJECT_ROOT)
append_check(
    validation_rows,
    "full_ground_up_loss_input_exists",
    FULL_GROUND_UP_LOSS_PATH.exists(),
    str(FULL_GROUND_UP_LOSS_PATH),
)
if not FULL_GROUND_UP_LOSS_PATH.exists():
    raise FileNotFoundError(FULL_GROUND_UP_LOSS_PATH)

recorded_full_loss_hash = str(full_loss_record.get("sha256", ""))
actual_full_loss_hash = sha256_file(FULL_GROUND_UP_LOSS_PATH)
append_check(
    validation_rows,
    "full_ground_up_loss_hash_matches_cell1_handoff",
    not recorded_full_loss_hash or actual_full_loss_hash == recorded_full_loss_hash,
    f"recorded={recorded_full_loss_hash}; actual={actual_full_loss_hash}",
)

policy_terms = pd.read_csv(POLICY_TERMS_PATH)
required_policy_columns = {
    "policy_scenario_id",
    "policy_id",
    "site_ordinal",
    "site_id",
    "policy_covered",
    "covered_loss_share",
    "building_replacement_value_2022_usd",
    "deductible_amount_2022_usd",
    "policy_limit_amount_2022_usd",
    "coinsurance_share",
}
missing_policy_columns = sorted(required_policy_columns.difference(policy_terms.columns))
append_check(
    validation_rows,
    "policy_terms_schema_complete",
    not missing_policy_columns,
    f"missing={missing_policy_columns}",
)
if missing_policy_columns:
    raise KeyError(f"Policy terms are missing columns: {missing_policy_columns}")

numeric_policy_columns = [
    "site_ordinal",
    "covered_loss_share",
    "building_replacement_value_2022_usd",
    "deductible_amount_2022_usd",
    "policy_limit_amount_2022_usd",
    "coinsurance_share",
]
for column in numeric_policy_columns:
    policy_terms[column] = pd.to_numeric(policy_terms[column], errors="raise")
policy_terms["policy_covered"] = parse_bool_series(policy_terms["policy_covered"])
policy_terms["site_id"] = policy_terms["site_id"].astype(str).str.strip()
policy_terms["policy_id"] = policy_terms["policy_id"].astype(str).str.strip()
policy_terms["policy_scenario_id"] = (
    policy_terms["policy_scenario_id"].astype(str).str.strip()
)

append_check(
    validation_rows,
    "policy_site_ordinals_unique",
    not policy_terms["site_ordinal"].duplicated().any(),
    f"duplicates={int(policy_terms['site_ordinal'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "policy_site_ids_unique",
    not policy_terms["site_id"].duplicated().any(),
    f"duplicates={int(policy_terms['site_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "all_baseline_policies_are_covered",
    bool(policy_terms["policy_covered"].all()),
    f"covered={int(policy_terms['policy_covered'].sum())}; rows={len(policy_terms)}",
)

scenario_ids = sorted(policy_terms["policy_scenario_id"].unique().tolist())
append_check(
    validation_rows,
    "one_policy_scenario_is_present",
    len(scenario_ids) == 1,
    f"scenario_ids={scenario_ids}",
)

baseline_terms = cell2_formula.get("terms", {})
append_check(
    validation_rows,
    "cell2_formula_specification_matches_policy_scenario",
    str(cell2_formula.get("policy_scenario_id")) == scenario_ids[0],
    (
        f"formula_scenario={cell2_formula.get('policy_scenario_id')}; "
        f"policy_scenario={scenario_ids[0]}"
    ),
)
append_check(
    validation_rows,
    "baseline_uses_full_coverage_and_coinsurance",
    bool(
        np.all(
            np.abs(policy_terms["covered_loss_share"].to_numpy(dtype=float) - 1.0)
            <= NUMERIC_TOLERANCE
        )
        and np.all(
            np.abs(policy_terms["coinsurance_share"].to_numpy(dtype=float) - 1.0)
            <= NUMERIC_TOLERANCE
        )
    ),
    (
        f"covered_loss_shares={sorted(policy_terms['covered_loss_share'].unique().tolist())}; "
        f"coinsurance_shares={sorted(policy_terms['coinsurance_share'].unique().tolist())}"
    ),
)

replacement_values = policy_terms["building_replacement_value_2022_usd"].to_numpy(dtype=float)
deductibles = policy_terms["deductible_amount_2022_usd"].to_numpy(dtype=float)
limits = policy_terms["policy_limit_amount_2022_usd"].to_numpy(dtype=float)
append_check(
    validation_rows,
    "deductibles_and_limits_are_valid",
    bool(
        np.isfinite(replacement_values).all()
        and np.isfinite(deductibles).all()
        and np.isfinite(limits).all()
        and np.all(replacement_values > 0.0)
        and np.all(deductibles >= 0.0)
        and np.all(limits > 0.0)
        and np.all(deductibles < limits)
    ),
    (
        f"minimum_replacement={replacement_values.min():.6f}; "
        f"minimum_deductible={deductibles.min():.6f}; "
        f"minimum_limit={limits.min():.6f}"
    ),
)

policy_order = policy_terms.sort_values(
    ["building_replacement_value_2022_usd", "site_ordinal"]
).reset_index(drop=True)
selected_policy_positions = sorted({0, len(policy_order) // 2, len(policy_order) - 1})
selected_policies = policy_order.iloc[selected_policy_positions].copy()
selected_policies["policy_size_case"] = ["minimum", "median", "maximum"]

stress_frames: list[pd.DataFrame] = []
for _, policy in selected_policies.iterrows():
    frame = pd.DataFrame(
        {
            "policy_size_case": policy["policy_size_case"],
            "stress_case_type": "formula_stress_test",
            "ground_up_loss_ratio_to_replacement": STRESS_LOSS_RATIOS,
        }
    )
    for column in required_policy_columns:
        frame[column] = policy[column]
    frame["sampled_total_ground_up_loss_2022_usd"] = (
        frame["ground_up_loss_ratio_to_replacement"].to_numpy(dtype=float)
        * float(policy["building_replacement_value_2022_usd"])
    )
    stress_frames.append(frame)

stress_cases = pd.concat(stress_frames, ignore_index=True)
stress_cases = add_policy_outputs(
    stress_cases, "sampled_total_ground_up_loss_2022_usd"
)

stress_ground_up = stress_cases["sampled_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
stress_replacement = stress_cases["building_replacement_value_2022_usd"].to_numpy(dtype=float)
stress_deductible = stress_cases["deductible_amount_2022_usd"].to_numpy(dtype=float)
stress_limit = stress_cases["policy_limit_amount_2022_usd"].to_numpy(dtype=float)
expected_baseline_gross = np.minimum(
    np.maximum(stress_ground_up - stress_deductible, 0.0), stress_limit
)
expected_baseline_uninsured = stress_ground_up - expected_baseline_gross
stress_cases["expected_baseline_gross_insured_loss_2022_usd"] = expected_baseline_gross
stress_cases["expected_baseline_uninsured_loss_2022_usd"] = expected_baseline_uninsured
stress_cases["within_notebook5_physical_loss_cap"] = stress_ground_up <= (
    stress_replacement + ROW_TOLERANCE_USD
)
stress_cases["gross_equation_error_2022_usd"] = (
    stress_cases["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
    - expected_baseline_gross
)
stress_cases["uninsured_equation_error_2022_usd"] = (
    stress_cases["uninsured_loss_2022_usd"].to_numpy(dtype=float)
    - expected_baseline_uninsured
)
stress_cases["uninsured_decomposition_error_2022_usd"] = (
    stress_cases["uninsured_loss_2022_usd"].to_numpy(dtype=float)
    - stress_cases["decomposed_uninsured_loss_2022_usd"].to_numpy(dtype=float)
)
stress_cases["loss_conservation_error_2022_usd"] = (
    stress_ground_up
    - stress_cases["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
    - stress_cases["uninsured_loss_2022_usd"].to_numpy(dtype=float)
)

max_stress_gross_error = float(
    np.max(np.abs(stress_cases["gross_equation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
max_stress_uninsured_error = float(
    np.max(np.abs(stress_cases["uninsured_equation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
max_stress_decomposition_error = float(
    np.max(
        np.abs(stress_cases["uninsured_decomposition_error_2022_usd"].to_numpy(dtype=float)), initial=0.0
    )
)
max_stress_conservation_error = float(
    np.max(np.abs(stress_cases["loss_conservation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
append_check(
    validation_rows,
    "stress_case_gross_equation_reproduced",
    max_stress_gross_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_stress_gross_error:.3e}",
)
append_check(
    validation_rows,
    "stress_case_uninsured_equation_reproduced",
    max_stress_uninsured_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_stress_uninsured_error:.3e}",
)
append_check(
    validation_rows,
    "stress_case_uninsured_decomposition_reproduced",
    max_stress_decomposition_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_stress_decomposition_error:.3e}",
)
append_check(
    validation_rows,
    "stress_case_loss_conservation_reproduced",
    max_stress_conservation_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_stress_conservation_error:.3e}",
)

below_or_at_deductible = stress_ground_up <= stress_deductible + ROW_TOLERANCE_USD
above_limit_threshold = stress_ground_up > (
    stress_deductible + stress_limit + ROW_TOLERANCE_USD
)
append_check(
    validation_rows,
    "gross_insured_is_zero_at_or_below_deductible",
    bool(
        np.all(
            np.abs(
                stress_cases.loc[
                    below_or_at_deductible, "gross_insured_loss_2022_usd"
                ].to_numpy(dtype=float)
            )
            <= ROW_TOLERANCE_USD
        )
    ),
    f"cases={int(below_or_at_deductible.sum())}",
)
append_check(
    validation_rows,
    "gross_insured_is_capped_by_policy_limit",
    bool(
        np.all(
            stress_cases["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
            <= stress_limit + ROW_TOLERANCE_USD
        )
    ),
    (
        "maximum_excess="
        f"{np.max(stress_cases['gross_insured_loss_2022_usd'].to_numpy(dtype=float) - stress_limit):.3e}"
    ),
)
append_check(
    validation_rows,
    "policy_limit_absorbed_loss_activates_in_stress_cases",
    bool(
        above_limit_threshold.any()
        and np.all(
            stress_cases.loc[
                above_limit_threshold, "policy_limit_absorbed_loss_2022_usd"
            ].to_numpy(dtype=float)
            > 0.0
        )
    ),
    f"above_limit_cases={int(above_limit_threshold.sum())}",
)
append_check(
    validation_rows,
    "coinsurance_absorbed_loss_is_zero_for_baseline",
    bool(
        np.all(
            np.abs(
                stress_cases["coinsurance_absorbed_loss_2022_usd"].to_numpy(dtype=float)
            )
            <= ROW_TOLERANCE_USD
        )
    ),
    (
        "maximum="
        f"{np.max(np.abs(stress_cases['coinsurance_absorbed_loss_2022_usd'].to_numpy(dtype=float)), initial=0.0):.3e}"
    ),
)

median_policy = selected_policies.loc[
    selected_policies["policy_size_case"].eq("median")
].iloc[0]
monotonicity_grid = pd.DataFrame(
    {
        "ground_up_loss_ratio_to_replacement": np.linspace(0.0, 1.25, 1_251),
    }
)
for column in required_policy_columns:
    monotonicity_grid[column] = median_policy[column]
monotonicity_grid["sampled_total_ground_up_loss_2022_usd"] = (
    monotonicity_grid["ground_up_loss_ratio_to_replacement"].to_numpy(dtype=float)
    * float(median_policy["building_replacement_value_2022_usd"])
)
monotonicity_grid = add_policy_outputs(
    monotonicity_grid, "sampled_total_ground_up_loss_2022_usd"
)

gross_differences = np.diff(
    monotonicity_grid["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
)
uninsured_differences = np.diff(
    monotonicity_grid["uninsured_loss_2022_usd"].to_numpy(dtype=float)
)
append_check(
    validation_rows,
    "gross_insured_loss_is_monotonic_nondecreasing",
    bool(np.all(gross_differences >= -ROW_TOLERANCE_USD)),
    f"minimum_increment={gross_differences.min():.6f}",
)
append_check(
    validation_rows,
    "uninsured_loss_is_monotonic_nondecreasing",
    bool(np.all(uninsured_differences >= -ROW_TOLERANCE_USD)),
    f"minimum_increment={uninsured_differences.min():.6f}",
)

physical_grid = monotonicity_grid.loc[
    monotonicity_grid["ground_up_loss_ratio_to_replacement"] <= 1.0 + NUMERIC_TOLERANCE
]
append_check(
    validation_rows,
    "baseline_policy_limit_is_nonbinding_within_physical_loss_cap",
    bool(
        np.all(
            np.abs(
                physical_grid["policy_limit_absorbed_loss_2022_usd"].to_numpy(dtype=float)
            )
            <= ROW_TOLERANCE_USD
        )
    ),
    (
        "maximum_limit_absorbed="
        f"{np.max(np.abs(physical_grid['policy_limit_absorbed_loss_2022_usd'].to_numpy(dtype=float)), initial=0.0):.3e}"
    ),
)

required_loss_columns = {
    "catalog_year",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    "site_ordinal",
    "site_id",
    "building_replacement_value_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
}
full_header = pd.read_csv(FULL_GROUND_UP_LOSS_PATH, compression="gzip", nrows=0)
missing_loss_columns = sorted(required_loss_columns.difference(full_header.columns))
append_check(
    validation_rows,
    "ground_up_loss_schema_complete_for_control_selection",
    not missing_loss_columns,
    f"missing={missing_loss_columns}",
)
if missing_loss_columns:
    raise KeyError(f"Ground-up loss file is missing columns: {missing_loss_columns}")

selection_columns = sorted(required_loss_columns)
best_rows: dict[tuple[str, float], tuple[float, dict[str, Any]]] = {}
total_rows_scanned = 0
source_types_seen: set[str] = set()
minimum_production_ratio = np.inf
maximum_production_ratio = -np.inf
maximum_production_loss_cap_error = 0.0

for chunk in pd.read_csv(
    FULL_GROUND_UP_LOSS_PATH,
    compression="gzip",
    usecols=selection_columns,
    chunksize=READ_CHUNK_ROWS,
):
    for column in [
        "catalog_year",
        "occurrence_ordinal",
        "magnitude",
        "site_ordinal",
        "building_replacement_value_2022_usd",
        "sampled_total_ground_up_loss_2022_usd",
    ]:
        chunk[column] = pd.to_numeric(chunk[column], errors="raise")
    chunk["source_type"] = chunk["source_type"].astype(str).str.strip()
    chunk["site_id"] = chunk["site_id"].astype(str).str.strip()
    replacement = chunk["building_replacement_value_2022_usd"].to_numpy(dtype=float)
    losses = chunk["sampled_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
    ratios = np.divide(
        losses,
        replacement,
        out=np.zeros_like(losses, dtype=float),
        where=replacement > 0.0,
    )
    chunk = chunk.assign(_loss_ratio=ratios)
    total_rows_scanned += len(chunk)
    minimum_production_ratio = min(minimum_production_ratio, float(ratios.min()))
    maximum_production_ratio = max(maximum_production_ratio, float(ratios.max()))
    maximum_production_loss_cap_error = max(
        maximum_production_loss_cap_error,
        float(np.max(losses - replacement, initial=0.0)),
    )

    for source_type, source_chunk in chunk.groupby("source_type", sort=False):
        source_types_seen.add(str(source_type))
        source_ratios = source_chunk["_loss_ratio"].to_numpy(dtype=float)
        for target in ACTUAL_TARGET_LOSS_RATIOS:
            local_position = int(np.argmin(np.abs(source_ratios - target)))
            selected = source_chunk.iloc[local_position]
            distance = float(abs(float(selected["_loss_ratio"]) - target))
            key = (str(source_type), float(target))
            current = best_rows.get(key)
            if current is None or distance < current[0]:
                row = selected.drop(labels=["_loss_ratio"]).to_dict()
                row["selection_source_type"] = str(source_type)
                row["selection_target_loss_ratio"] = float(target)
                row["actual_ground_up_loss_ratio"] = float(selected["_loss_ratio"])
                row["selection_absolute_ratio_error"] = distance
                best_rows[key] = (distance, row)

expected_scanned_rows = int(cell1_handoff.get("ground_up_loss", {}).get("building_rows", 0))
append_check(
    validation_rows,
    "control_selection_scanned_all_ground_up_rows",
    expected_scanned_rows <= 0 or total_rows_scanned == expected_scanned_rows,
    f"scanned={total_rows_scanned}; expected={expected_scanned_rows}",
)
append_check(
    validation_rows,
    "production_ground_up_losses_respect_replacement_value_cap",
    maximum_production_loss_cap_error <= ROW_TOLERANCE_USD,
    f"maximum_excess={maximum_production_loss_cap_error:.3e}",
)
append_check(
    validation_rows,
    "actual_control_selection_covers_every_source_and_target",
    len(best_rows) == len(source_types_seen) * len(ACTUAL_TARGET_LOSS_RATIOS),
    (
        f"selected={len(best_rows)}; sources={sorted(source_types_seen)}; "
        f"targets={ACTUAL_TARGET_LOSS_RATIOS.tolist()}"
    ),
)

actual_controls = pd.DataFrame([item[1] for item in best_rows.values()])
actual_controls = actual_controls.sort_values(
    ["selection_source_type", "selection_target_loss_ratio"]
).reset_index(drop=True)

policy_join_columns = [
    "policy_scenario_id",
    "policy_id",
    "site_ordinal",
    "site_id",
    "policy_covered",
    "covered_loss_share",
    "deductible_amount_2022_usd",
    "policy_limit_amount_2022_usd",
    "coinsurance_share",
]
actual_controls = actual_controls.merge(
    policy_terms[policy_join_columns],
    on=["site_ordinal", "site_id"],
    how="left",
    validate="many_to_one",
)
append_check(
    validation_rows,
    "actual_controls_join_to_policy_terms",
    not actual_controls["policy_id"].isna().any(),
    f"missing_policy_rows={int(actual_controls['policy_id'].isna().sum())}",
)
if actual_controls["policy_id"].isna().any():
    raise RuntimeError("One or more controlled production rows lack policy terms.")

actual_controls = add_policy_outputs(
    actual_controls, "sampled_total_ground_up_loss_2022_usd"
)
actual_ground_up = actual_controls["sampled_total_ground_up_loss_2022_usd"].to_numpy(dtype=float)
actual_deductible = actual_controls["deductible_amount_2022_usd"].to_numpy(dtype=float)
actual_limit = actual_controls["policy_limit_amount_2022_usd"].to_numpy(dtype=float)
actual_expected_gross = np.minimum(
    np.maximum(actual_ground_up - actual_deductible, 0.0), actual_limit
)
actual_expected_uninsured = actual_ground_up - actual_expected_gross
actual_controls["expected_baseline_gross_insured_loss_2022_usd"] = actual_expected_gross
actual_controls["expected_baseline_uninsured_loss_2022_usd"] = actual_expected_uninsured
actual_controls["gross_equation_error_2022_usd"] = (
    actual_controls["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
    - actual_expected_gross
)
actual_controls["uninsured_equation_error_2022_usd"] = (
    actual_controls["uninsured_loss_2022_usd"].to_numpy(dtype=float)
    - actual_expected_uninsured
)
actual_controls["uninsured_decomposition_error_2022_usd"] = (
    actual_controls["uninsured_loss_2022_usd"].to_numpy(dtype=float)
    - actual_controls["decomposed_uninsured_loss_2022_usd"].to_numpy(dtype=float)
)
actual_controls["loss_conservation_error_2022_usd"] = (
    actual_ground_up
    - actual_controls["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
    - actual_controls["uninsured_loss_2022_usd"].to_numpy(dtype=float)
)

max_actual_gross_error = float(
    np.max(np.abs(actual_controls["gross_equation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
max_actual_uninsured_error = float(
    np.max(np.abs(actual_controls["uninsured_equation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
max_actual_decomposition_error = float(
    np.max(
        np.abs(actual_controls["uninsured_decomposition_error_2022_usd"].to_numpy(dtype=float)), initial=0.0
    )
)
max_actual_conservation_error = float(
    np.max(np.abs(actual_controls["loss_conservation_error_2022_usd"].to_numpy(dtype=float)), initial=0.0)
)
append_check(
    validation_rows,
    "actual_control_gross_equation_reproduced",
    max_actual_gross_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_actual_gross_error:.3e}",
)
append_check(
    validation_rows,
    "actual_control_uninsured_equation_reproduced",
    max_actual_uninsured_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_actual_uninsured_error:.3e}",
)
append_check(
    validation_rows,
    "actual_control_uninsured_decomposition_reproduced",
    max_actual_decomposition_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_actual_decomposition_error:.3e}",
)
append_check(
    validation_rows,
    "actual_control_loss_conservation_reproduced",
    max_actual_conservation_error <= ROW_TOLERANCE_USD,
    f"maximum_error={max_actual_conservation_error:.3e}",
)
append_check(
    validation_rows,
    "actual_control_gross_insured_is_nonnegative_and_bounded",
    bool(
        np.all(actual_controls["gross_insured_loss_2022_usd"].to_numpy(dtype=float) >= -ROW_TOLERANCE_USD)
        and np.all(
            actual_controls["gross_insured_loss_2022_usd"].to_numpy(dtype=float)
            <= actual_ground_up + ROW_TOLERANCE_USD
        )
    ),
    (
        f"minimum={actual_controls['gross_insured_loss_2022_usd'].min():.6f}; "
        f"maximum={actual_controls['gross_insured_loss_2022_usd'].max():.6f}"
    ),
)
append_check(
    validation_rows,
    "actual_control_policy_limit_is_nonbinding",
    bool(
        np.all(
            np.abs(
                actual_controls["policy_limit_absorbed_loss_2022_usd"].to_numpy(dtype=float)
            )
            <= ROW_TOLERANCE_USD
        )
    ),
    (
        "maximum="
        f"{np.max(np.abs(actual_controls['policy_limit_absorbed_loss_2022_usd'].to_numpy(dtype=float)), initial=0.0):.3e}"
    ),
)
append_check(
    validation_rows,
    "actual_controls_represent_all_source_types",
    set(actual_controls["source_type"].astype(str)) == source_types_seen,
    (
        f"actual={sorted(actual_controls['source_type'].astype(str).unique().tolist())}; "
        f"expected={sorted(source_types_seen)}"
    ),
)

actual_controls.to_csv(ACTUAL_CONTROLS_PATH, index=False)
stress_cases.to_csv(STRESS_CASES_PATH, index=False)
monotonicity_grid.to_csv(MONOTONICITY_GRID_PATH, index=False)

formula_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_scenario_id": scenario_ids[0],
    "currency": "USD",
    "dollar_year": DOLLAR_YEAR,
    "input_loss": "sampled total building ground-up repair loss",
    "equations": {
        "eligible_loss": (
            "policy_covered ? ground_up_loss * covered_loss_share : 0"
        ),
        "uncovered_loss": "ground_up_loss - eligible_loss",
        "deductible_absorbed_loss": "min(eligible_loss, deductible_amount)",
        "loss_after_deductible": "max(eligible_loss - deductible_amount, 0)",
        "limited_loss": "min(loss_after_deductible, policy_limit_amount)",
        "policy_limit_absorbed_loss": (
            "max(loss_after_deductible - policy_limit_amount, 0)"
        ),
        "gross_insured_loss": "limited_loss * coinsurance_share",
        "coinsurance_absorbed_loss": "limited_loss * (1 - coinsurance_share)",
        "uninsured_loss": "ground_up_loss - gross_insured_loss",
        "uninsured_decomposition": (
            "uncovered_loss + deductible_absorbed_loss + "
            "policy_limit_absorbed_loss + coinsurance_absorbed_loss"
        ),
    },
    "baseline_simplification": (
        "With 100% take-up, 100% covered loss share, full coinsurance, and a "
        "replacement-value limit, production gross insured loss equals "
        "max(ground_up_loss - deductible, 0). The limit cannot bind because "
        "Notebook 5 caps building ground-up loss at replacement value."
    ),
    "analytical_expected_loss_note": (
        "The policy function is nonlinear. Applying deductibles or limits to "
        "Notebook 5 analytical expected ground-up loss would not equal expected "
        "insured payout. Production insurance calculations therefore use sampled "
        "occurrence-building ground-up losses only."
    ),
    "stress_test_note": (
        "Stress cases above 100% of replacement value are mathematical equation "
        "tests used to activate the policy-limit branch. They are not production "
        "ground-up losses."
    ),
    "upstream_cell2_terms": baseline_terms,
}
write_json(FORMULA_SPECIFICATION_PATH, formula_specification)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
]
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "policy_scenario_id": scenario_ids[0],
    "production_scan": {
        "rows": int(total_rows_scanned),
        "source_types": sorted(source_types_seen),
        "minimum_ground_up_loss_ratio": float(minimum_production_ratio),
        "maximum_ground_up_loss_ratio": float(maximum_production_ratio),
        "maximum_loss_cap_error_2022_usd": float(maximum_production_loss_cap_error),
    },
    "controlled_validation": {
        "actual_control_rows": int(len(actual_controls)),
        "stress_case_rows": int(len(stress_cases)),
        "monotonicity_grid_rows": int(len(monotonicity_grid)),
        "maximum_actual_gross_equation_error_2022_usd": max_actual_gross_error,
        "maximum_actual_uninsured_equation_error_2022_usd": max_actual_uninsured_error,
        "maximum_actual_decomposition_error_2022_usd": max_actual_decomposition_error,
        "maximum_stress_gross_equation_error_2022_usd": max_stress_gross_error,
        "maximum_stress_uninsured_equation_error_2022_usd": max_stress_uninsured_error,
        "maximum_stress_decomposition_error_2022_usd": max_stress_decomposition_error,
    },
    "validation": {
        "checks": int(len(validation)),
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warnings.to_dict(orient="records"),
    },
    "outputs": {
        "actual_controls": {
            "path": str(ACTUAL_CONTROLS_PATH),
            "sha256": sha256_file(ACTUAL_CONTROLS_PATH),
            "rows": int(len(actual_controls)),
        },
        "stress_cases": {
            "path": str(STRESS_CASES_PATH),
            "sha256": sha256_file(STRESS_CASES_PATH),
            "rows": int(len(stress_cases)),
        },
        "monotonicity_grid": {
            "path": str(MONOTONICITY_GRID_PATH),
            "sha256": sha256_file(MONOTONICITY_GRID_PATH),
            "rows": int(len(monotonicity_grid)),
        },
        "formula_specification": {
            "path": str(FORMULA_SPECIFICATION_PATH),
            "sha256": sha256_file(FORMULA_SPECIFICATION_PATH),
        },
        "validation": {
            "path": str(VALIDATION_PATH),
            "sha256": sha256_file(VALIDATION_PATH),
        },
    },
    "next_cell": (
        "Cell 4: apply validated building policy terms to all 4,996,100 "
        "occurrence-building ground-up loss rows using restartable chunks."
    ),
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 3 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 3 CONTROLLED POLICY LOSS VALIDATION COMPLETE")
print("=" * 78)
print(f"Ground-up rows scanned:             {total_rows_scanned:,}")
print(f"Source types represented:           {len(source_types_seen):,}")
print(f"Controlled production rows:         {len(actual_controls):,}")
print(f"Formula stress-test rows:            {len(stress_cases):,}")
print(f"Monotonicity-grid rows:              {len(monotonicity_grid):,}")
print(f"Maximum production loss ratio:       {maximum_production_ratio:.6f}")
print(f"Maximum actual gross equation error: ${max_actual_gross_error:.3e}")
print(f"Maximum stress gross equation error: ${max_stress_gross_error:.3e}")
print(f"Critical validation checks:          {len(validation.loc[validation['severity'].eq('critical')]):,}")
print(f"Critical failures:                   {len(critical_failures):,}")
print(f"Warnings requiring review:           {len(warnings):,}")
print()
print("Controlled production policy losses:")
print(f"  {ACTUAL_CONTROLS_PATH}")
print("Policy formula stress cases:")
print(f"  {STRESS_CASES_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: full production gross-insured and uninsured loss calculation.")



NOTEBOOK 6 CELL 3 CONTROLLED POLICY LOSS VALIDATION COMPLETE
Ground-up rows scanned:             4,996,100
Source types represented:           2
Controlled production rows:         14
Formula stress-test rows:            33
Monotonicity-grid rows:              1,251
Maximum production loss ratio:       1.000000
Maximum actual gross equation error: $0.000e+00
Maximum stress gross equation error: $0.000e+00
Critical validation checks:          47
Critical failures:                   0
Warnings requiring review:           0

Controlled production policy losses:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_controlled_insurance_terms\controlled_actual_policy_loss_rows.csv
Policy formula stress cases:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_controlled_insurance_terms\controlled_policy_formula_stress_cases.csv
Validation:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\d

In [6]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import shutil
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell4_full_gross_insured_loss_v1"
DOLLAR_YEAR = 2022
ROW_TOLERANCE_USD = 1e-6
AGGREGATE_TOLERANCE_USD = 0.01
NUMERIC_TOLERANCE = 1e-10


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_input_handoff.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def prior_validation_status(path: Path) -> tuple[bool, int, int]:
    table = pd.read_csv(path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")
    table["passed"] = parse_bool_series(table["passed"])
    failures = table.loc[
        table["severity"].astype(str).str.lower().eq("critical")
        & ~table["passed"]
    ]
    warnings = table.loc[
        table["severity"].astype(str).str.lower().eq("warning")
        & ~table["passed"]
    ]
    return failures.empty, int(len(failures)), int(len(warnings))


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    buffer = io.StringIO()
    frame.to_csv(buffer, index=False, lineterminator="\n")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
            compressed.write(buffer.getvalue().encode("utf-8"))
    temporary.replace(path)


def concatenate_gzip_csv_files(chunk_paths: list[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(fileobj=raw_output, mode="wb", mtime=0) as compressed_output:
            for chunk_path in chunk_paths:
                with gzip.open(chunk_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        compressed_output.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            f"Chunk header mismatch while concatenating {chunk_path}."
                        )
                    shutil.copyfileobj(source, compressed_output, length=8 * 1024 * 1024)
    temporary.replace(output_path)


def apply_policy_terms(
    ground_up_loss: np.ndarray,
    policy_covered: np.ndarray,
    covered_loss_share: np.ndarray,
    deductible_amount: np.ndarray,
    policy_limit_amount: np.ndarray,
    coinsurance_share: np.ndarray,
) -> dict[str, np.ndarray]:
    ground_up = np.asarray(ground_up_loss, dtype=np.float64)
    covered = np.asarray(policy_covered, dtype=bool)
    covered_share = np.asarray(covered_loss_share, dtype=np.float64)
    deductible = np.asarray(deductible_amount, dtype=np.float64)
    policy_limit = np.asarray(policy_limit_amount, dtype=np.float64)
    coinsurance = np.asarray(coinsurance_share, dtype=np.float64)

    if np.any(~np.isfinite(ground_up)) or np.any(ground_up < 0.0):
        raise ValueError("Ground-up losses must be finite and nonnegative.")

    eligible_loss = np.where(covered, ground_up * covered_share, 0.0)
    uncovered_loss = ground_up - eligible_loss
    deductible_absorbed_loss = np.minimum(eligible_loss, deductible)
    loss_after_deductible = np.maximum(eligible_loss - deductible, 0.0)
    limited_loss = np.minimum(loss_after_deductible, policy_limit)
    policy_limit_absorbed_loss = np.maximum(
        loss_after_deductible - policy_limit, 0.0
    )
    gross_insured_loss = limited_loss * coinsurance
    coinsurance_absorbed_loss = limited_loss * (1.0 - coinsurance)
    uninsured_loss = ground_up - gross_insured_loss

    return {
        "eligible_ground_up_loss_2022_usd": eligible_loss,
        "uncovered_ground_up_loss_2022_usd": uncovered_loss,
        "deductible_absorbed_loss_2022_usd": deductible_absorbed_loss,
        "loss_after_deductible_2022_usd": loss_after_deductible,
        "limited_loss_2022_usd": limited_loss,
        "policy_limit_absorbed_loss_2022_usd": policy_limit_absorbed_loss,
        "coinsurance_absorbed_loss_2022_usd": coinsurance_absorbed_loss,
        "gross_insured_loss_2022_usd": gross_insured_loss,
        "uninsured_loss_2022_usd": uninsured_loss,
    }


def summarize_occurrences(frame: pd.DataFrame) -> pd.DataFrame:
    event_fields = [
        "catalog_year",
        "catalog_event_id",
        "rupture_ordinal",
        "rupture_template_event_id",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
    ]
    group_columns = [column for column in event_fields if column in frame.columns]

    output = (
        frame.groupby(group_columns, sort=False)
        .agg(
            buildings=("site_id", "size"),
            buildings_with_ground_up_loss=("ground_up_loss_positive_indicator", "sum"),
            buildings_with_gross_insured_loss=("gross_insured_loss_positive_indicator", "sum"),
            buildings_fully_absorbed_by_deductible=(
                "deductible_fully_absorbs_loss_indicator",
                "sum",
            ),
            buildings_hitting_policy_limit=("policy_limit_binding_indicator", "sum"),
            portfolio_replacement_value_2022_usd=(
                "building_replacement_value_2022_usd",
                "sum",
            ),
            sampled_total_ground_up_loss_2022_usd=(
                "sampled_total_ground_up_loss_2022_usd",
                "sum",
            ),
            eligible_ground_up_loss_2022_usd=(
                "eligible_ground_up_loss_2022_usd",
                "sum",
            ),
            uncovered_ground_up_loss_2022_usd=(
                "uncovered_ground_up_loss_2022_usd",
                "sum",
            ),
            deductible_absorbed_loss_2022_usd=(
                "deductible_absorbed_loss_2022_usd",
                "sum",
            ),
            loss_after_deductible_2022_usd=(
                "loss_after_deductible_2022_usd",
                "sum",
            ),
            limited_loss_2022_usd=("limited_loss_2022_usd", "sum"),
            policy_limit_absorbed_loss_2022_usd=(
                "policy_limit_absorbed_loss_2022_usd",
                "sum",
            ),
            coinsurance_absorbed_loss_2022_usd=(
                "coinsurance_absorbed_loss_2022_usd",
                "sum",
            ),
            gross_insured_loss_2022_usd=("gross_insured_loss_2022_usd", "sum"),
            uninsured_loss_2022_usd=("uninsured_loss_2022_usd", "sum"),
        )
        .reset_index()
    )

    ground_up = output["sampled_total_ground_up_loss_2022_usd"].to_numpy(
        dtype=np.float64
    )
    replacement = output["portfolio_replacement_value_2022_usd"].to_numpy(
        dtype=np.float64
    )
    gross = output["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
    uninsured = output["uninsured_loss_2022_usd"].to_numpy(dtype=np.float64)

    output["ground_up_loss_ratio_to_portfolio_replacement"] = np.divide(
        ground_up,
        replacement,
        out=np.zeros_like(ground_up),
        where=replacement > 0.0,
    )
    output["gross_insured_loss_ratio_to_portfolio_replacement"] = np.divide(
        gross,
        replacement,
        out=np.zeros_like(gross),
        where=replacement > 0.0,
    )
    output["insurance_recovery_share_of_ground_up_loss"] = np.divide(
        gross,
        ground_up,
        out=np.zeros_like(gross),
        where=ground_up > 0.0,
    )
    output["uninsured_share_of_ground_up_loss"] = np.divide(
        uninsured,
        ground_up,
        out=np.zeros_like(uninsured),
        where=ground_up > 0.0,
    )
    return output


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_insurance_parameters"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_baseline_insured_loss"
CHUNK_DIR = OUTPUT_DIR / "chunks"
WORK_DIR = OUTPUT_DIR / "work"
MARKER_DIR = WORK_DIR / "markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validations"
for directory in [METADATA_DIR, OUTPUT_DIR, CHUNK_DIR, MARKER_DIR, CHUNK_VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CELL1_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_1_validation.csv"
CELL1_HANDOFF_PATH = METADATA_DIR / "notebook_6_input_handoff.json"
CELL2_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_2_validation.csv"
CELL2_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_2_summary.json"
CELL2_FORMULA_PATH = METADATA_DIR / "notebook_6_cell_2_policy_formula_specification.json"
CELL3_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_3_validation.csv"
CELL3_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_3_summary.json"
CELL3_FORMULA_PATH = METADATA_DIR / "notebook_6_cell_3_insurance_loss_formula_specification.json"
POLICY_TERMS_PATH = PARAMETER_DIR / "seaside_w2_baseline_policy_terms.csv"
CONTROLLED_ACTUAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_6_controlled_insurance_terms"
    / "controlled_actual_policy_loss_rows.csv"
)

CELL11_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_5_damage_loss"
    / "notebook_5_cell_11_chunk_manifest.csv"
)

FINAL_BUILDING_LOSS_PATH = OUTPUT_DIR / "full_baseline_gross_insured_loss.csv.gz"
FINAL_EVENT_LOSS_PATH = OUTPUT_DIR / "baseline_gross_insured_loss_event_summary.csv.gz"
CHUNK_MANIFEST_PATH = METADATA_DIR / "notebook_6_cell_4_chunk_manifest.csv"
LOSS_RECONCILIATION_PATH = METADATA_DIR / "notebook_6_cell_4_loss_reconciliation.csv"
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_4_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_4_summary.json"
FORMULA_SPECIFICATION_PATH = (
    METADATA_DIR / "notebook_6_cell_4_production_insurance_formula_specification.json"
)

validation_rows: list[dict[str, Any]] = []
required_inputs = [
    CELL1_VALIDATION_PATH,
    CELL1_HANDOFF_PATH,
    CELL2_VALIDATION_PATH,
    CELL2_SUMMARY_PATH,
    CELL2_FORMULA_PATH,
    CELL3_VALIDATION_PATH,
    CELL3_SUMMARY_PATH,
    CELL3_FORMULA_PATH,
    POLICY_TERMS_PATH,
    CONTROLLED_ACTUAL_PATH,
    CELL11_MANIFEST_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")

for upstream_path, label in [
    (CELL1_VALIDATION_PATH, "cell1"),
    (CELL2_VALIDATION_PATH, "cell2"),
    (CELL3_VALIDATION_PATH, "cell3"),
]:
    passed, failures, warnings = prior_validation_status(upstream_path)
    append_check(
        validation_rows,
        f"{label}_critical_validation_passed",
        passed,
        f"critical_failures={failures}",
    )
    append_check(
        validation_rows,
        f"{label}_has_no_unresolved_warnings",
        warnings == 0,
        f"warnings={warnings}",
    )

cell1_handoff = load_json(CELL1_HANDOFF_PATH)
cell2_summary = load_json(CELL2_SUMMARY_PATH)
cell3_summary = load_json(CELL3_SUMMARY_PATH)
cell3_formula = load_json(CELL3_FORMULA_PATH)

append_check(
    validation_rows,
    "cell1_handoff_declares_success",
    bool(cell1_handoff.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell1_handoff.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell2_summary_declares_success",
    bool(cell2_summary.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell2_summary.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell3_summary_declares_success",
    bool(cell3_summary.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell3_summary.get('all_critical_checks_passed')}",
)

policy_terms = pd.read_csv(POLICY_TERMS_PATH)
required_policy_columns = {
    "policy_scenario_id",
    "policy_scenario_name",
    "policy_id",
    "site_ordinal",
    "site_id",
    "building_replacement_value_2022_usd",
    "policy_covered",
    "covered_loss_share",
    "deductible_amount_2022_usd",
    "policy_limit_amount_2022_usd",
    "coinsurance_share",
}
missing_policy_columns = sorted(required_policy_columns.difference(policy_terms.columns))
append_check(
    validation_rows,
    "policy_terms_schema_complete",
    not missing_policy_columns,
    f"missing={missing_policy_columns}",
)
if missing_policy_columns:
    raise KeyError(f"Policy terms are missing columns: {missing_policy_columns}")

numeric_policy_columns = [
    "site_ordinal",
    "building_replacement_value_2022_usd",
    "covered_loss_share",
    "deductible_amount_2022_usd",
    "policy_limit_amount_2022_usd",
    "coinsurance_share",
]
for column in numeric_policy_columns:
    policy_terms[column] = pd.to_numeric(policy_terms[column], errors="raise")
policy_terms["site_ordinal"] = policy_terms["site_ordinal"].astype(np.int64)
policy_terms["site_id"] = policy_terms["site_id"].astype(str).str.strip()
policy_terms["policy_id"] = policy_terms["policy_id"].astype(str).str.strip()
policy_terms["policy_scenario_id"] = (
    policy_terms["policy_scenario_id"].astype(str).str.strip()
)
policy_terms["policy_covered"] = parse_bool_series(policy_terms["policy_covered"])

append_check(
    validation_rows,
    "policy_site_ordinals_unique",
    not policy_terms["site_ordinal"].duplicated().any(),
    f"duplicates={int(policy_terms['site_ordinal'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "policy_site_ids_unique",
    not policy_terms["site_id"].duplicated().any(),
    f"duplicates={int(policy_terms['site_id'].duplicated().sum())}",
)
scenario_ids = sorted(policy_terms["policy_scenario_id"].unique().tolist())
append_check(
    validation_rows,
    "exactly_one_policy_scenario",
    len(scenario_ids) == 1,
    f"scenario_ids={scenario_ids}",
)
if len(scenario_ids) != 1:
    raise RuntimeError(f"Expected one policy scenario, found {scenario_ids}")
policy_scenario_id = scenario_ids[0]
policy_scenario_name = str(policy_terms["policy_scenario_name"].iloc[0])

cell2_policy_output = cell2_summary.get("outputs", {}).get("policy_terms", {})
recorded_policy_hash = str(cell2_policy_output.get("sha256", ""))
actual_policy_hash = sha256_file(POLICY_TERMS_PATH)
append_check(
    validation_rows,
    "policy_terms_hash_matches_cell2_summary",
    not recorded_policy_hash or actual_policy_hash == recorded_policy_hash,
    f"actual={actual_policy_hash}; recorded={recorded_policy_hash}",
)

expected_buildings = int(
    cell1_handoff.get("portfolio", {}).get("buildings", len(policy_terms))
)
expected_occurrences = int(
    cell1_handoff.get("annual_catalog", {}).get("occurrences", 0)
)
expected_rows = int(
    cell1_handoff.get("ground_up_loss", {}).get("building_rows", 0)
)
declared_years = int(
    cell1_handoff.get("annual_catalog", {}).get("declared_duration_years", 0)
)
if expected_occurrences <= 0 and expected_rows > 0 and expected_buildings > 0:
    expected_occurrences = expected_rows // expected_buildings
if expected_rows <= 0 and expected_occurrences > 0 and expected_buildings > 0:
    expected_rows = expected_occurrences * expected_buildings
if declared_years <= 0:
    declared_years = int(
        cell1_handoff.get("annual_catalog", {}).get("catalog_years", 0)
    )

append_check(
    validation_rows,
    "policy_terms_cover_expected_buildings",
    len(policy_terms) == expected_buildings,
    f"policy_rows={len(policy_terms)}; expected={expected_buildings}",
)

manifest = pd.read_csv(CELL11_MANIFEST_PATH)
required_manifest_columns = {
    "chunk_id",
    "rows",
    "occurrences",
    "minimum_occurrence_ordinal",
    "maximum_occurrence_ordinal",
    "sampled_total_loss_sum_2022_usd",
    "output_path",
    "output_sha256",
}
missing_manifest_columns = sorted(required_manifest_columns.difference(manifest.columns))
append_check(
    validation_rows,
    "cell11_chunk_manifest_schema_complete",
    not missing_manifest_columns,
    f"missing={missing_manifest_columns}",
)
if missing_manifest_columns:
    raise KeyError(f"Cell 11 chunk manifest is missing columns: {missing_manifest_columns}")

for column in [
    "chunk_id",
    "rows",
    "occurrences",
    "minimum_occurrence_ordinal",
    "maximum_occurrence_ordinal",
    "sampled_total_loss_sum_2022_usd",
]:
    manifest[column] = pd.to_numeric(manifest[column], errors="raise")
manifest["chunk_id"] = manifest["chunk_id"].astype(np.int64)
manifest["rows"] = manifest["rows"].astype(np.int64)
manifest["occurrences"] = manifest["occurrences"].astype(np.int64)
manifest["minimum_occurrence_ordinal"] = manifest[
    "minimum_occurrence_ordinal"
].astype(np.int64)
manifest["maximum_occurrence_ordinal"] = manifest[
    "maximum_occurrence_ordinal"
].astype(np.int64)
manifest = manifest.sort_values("minimum_occurrence_ordinal").reset_index(drop=True)

append_check(
    validation_rows,
    "cell11_chunk_ids_unique",
    not manifest["chunk_id"].duplicated().any(),
    f"duplicates={int(manifest['chunk_id'].duplicated().sum())}",
)
append_check(
    validation_rows,
    "cell11_manifest_rows_reconcile",
    int(manifest["rows"].sum()) == expected_rows,
    f"manifest_rows={int(manifest['rows'].sum())}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "cell11_manifest_occurrences_reconcile",
    int(manifest["occurrences"].sum()) == expected_occurrences,
    (
        f"manifest_occurrences={int(manifest['occurrences'].sum())}; "
        f"expected={expected_occurrences}"
    ),
)

manifest_minimum_occurrence = int(manifest["minimum_occurrence_ordinal"].min())
manifest_maximum_occurrence = int(manifest["maximum_occurrence_ordinal"].max())
manifest_spans_catalog = (
    manifest_minimum_occurrence == 0
    and manifest_maximum_occurrence == expected_occurrences - 1
)
append_check(
    validation_rows,
    "cell11_manifest_occurrence_domain_spans_catalog",
    manifest_spans_catalog,
    (
        f"minimum={manifest_minimum_occurrence}; "
        f"maximum={manifest_maximum_occurrence}; "
        f"expected_minimum=0; expected_maximum={expected_occurrences - 1}; "
        "chunk ranges need not be mutually contiguous because source chunks may "
        "contain nonconsecutive occurrence ordinals"
    ),
)

controlled_actual = pd.read_csv(CONTROLLED_ACTUAL_PATH)
controlled_key_columns = ["occurrence_id", "site_id"]
controlled_result_columns = [
    "eligible_ground_up_loss_2022_usd",
    "uncovered_ground_up_loss_2022_usd",
    "deductible_absorbed_loss_2022_usd",
    "loss_after_deductible_2022_usd",
    "limited_loss_2022_usd",
    "policy_limit_absorbed_loss_2022_usd",
    "coinsurance_absorbed_loss_2022_usd",
    "gross_insured_loss_2022_usd",
    "uninsured_loss_2022_usd",
]
missing_control_columns = sorted(
    set(controlled_key_columns + controlled_result_columns).difference(
        controlled_actual.columns
    )
)
append_check(
    validation_rows,
    "cell3_controlled_output_schema_complete",
    not missing_control_columns,
    f"missing={missing_control_columns}",
)
if missing_control_columns:
    raise KeyError(
        f"Controlled policy-loss output is missing columns: {missing_control_columns}"
    )
controlled_actual["occurrence_id"] = (
    controlled_actual["occurrence_id"].astype(str).str.strip()
)
controlled_actual["site_id"] = controlled_actual["site_id"].astype(str).str.strip()
controlled_keys = set(
    zip(controlled_actual["occurrence_id"], controlled_actual["site_id"])
)

formula_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_scenario_id": policy_scenario_id,
    "policy_scenario_name": policy_scenario_name,
    "currency": "USD",
    "dollar_year": DOLLAR_YEAR,
    "input_loss_basis": (
        "sampled total building repair ground-up loss from Notebook 5 Cell 11"
    ),
    "equations": {
        "eligible_ground_up_loss": (
            "policy_covered ? ground_up_loss * covered_loss_share : 0"
        ),
        "uncovered_ground_up_loss": "ground_up_loss - eligible_ground_up_loss",
        "deductible_absorbed_loss": (
            "min(eligible_ground_up_loss, deductible_amount)"
        ),
        "loss_after_deductible": (
            "max(eligible_ground_up_loss - deductible_amount, 0)"
        ),
        "limited_loss": "min(loss_after_deductible, policy_limit_amount)",
        "policy_limit_absorbed_loss": (
            "max(loss_after_deductible - policy_limit_amount, 0)"
        ),
        "gross_insured_loss": "limited_loss * coinsurance_share",
        "coinsurance_absorbed_loss": (
            "limited_loss * (1 - coinsurance_share)"
        ),
        "uninsured_loss": "ground_up_loss - gross_insured_loss",
        "loss_conservation": (
            "ground_up_loss = gross_insured_loss + uninsured_loss"
        ),
        "uninsured_decomposition": (
            "uninsured_loss = uncovered_ground_up_loss + "
            "deductible_absorbed_loss + policy_limit_absorbed_loss + "
            "coinsurance_absorbed_loss"
        ),
    },
    "important_scope_notes": [
        "Production insurance calculations use sampled occurrence-building losses.",
        "Analytical expected ground-up losses are not passed through nonlinear policy terms.",
        "The baseline policy is synthetic and is not observed Seaside insurance data.",
        "Reinsurance terms are not applied in Cell 4.",
    ],
    "upstream_cell3_formula_sha256": sha256_file(CELL3_FORMULA_PATH),
}
write_json(FORMULA_SPECIFICATION_PATH, formula_specification)

basis_payload = {
    "pipeline_version": PIPELINE_VERSION,
    "policy_terms_sha256": actual_policy_hash,
    "cell3_formula_sha256": sha256_file(CELL3_FORMULA_PATH),
    "cell3_controlled_output_sha256": sha256_file(CONTROLLED_ACTUAL_PATH),
    "policy_scenario_id": policy_scenario_id,
    "row_tolerance_usd": ROW_TOLERANCE_USD,
}
basis_hash = hashlib.sha256(
    json.dumps(basis_payload, sort_keys=True).encode("utf-8")
).hexdigest()

policy_join_columns = [
    column
    for column in [
        "site_ordinal",
        "site_id",
        "policy_scenario_id",
        "policy_scenario_name",
        "policy_id",
        "policy_covered",
        "coverage_take_up_share",
        "covered_loss_share",
        "insured_value_2022_usd",
        "deductible_type",
        "deductible_percent",
        "deductible_amount_2022_usd",
        "policy_limit_type",
        "policy_limit_percent",
        "policy_limit_amount_2022_usd",
        "coinsurance_share",
        "deductible_application",
        "policy_limit_application",
        "assumption_source",
    ]
    if column in policy_terms.columns
]
policy_lookup = policy_terms[policy_join_columns].copy()

print()
print("=" * 78)
print("FULL ANNUAL-CATALOG BASELINE GROSS-INSURED LOSS")
print("=" * 78)
print(f"Ground-up rows:            {expected_rows:,}")
print(f"Catalog occurrences:       {expected_occurrences:,}")
print(f"Portfolio buildings:       {expected_buildings:,}")
print(f"Production chunks:         {len(manifest):,}")
print(f"Policy scenario:           {policy_scenario_id}")
print("Deductible application:    per building, per occurrence")
print("Loss basis:                sampled total ground-up repair loss")

seen_pairs = np.zeros(expected_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int64)
site_counts = np.zeros(expected_buildings, dtype=np.int64)
chunk_paths: list[Path] = []
chunk_manifest_rows: list[dict[str, Any]] = []
event_summary_frames: list[pd.DataFrame] = []
controlled_production_frames: list[pd.DataFrame] = []
source_totals: Counter[str] = Counter()

aggregate_totals = {
    "ground_up": 0.0,
    "eligible": 0.0,
    "uncovered": 0.0,
    "deductible": 0.0,
    "after_deductible": 0.0,
    "limited": 0.0,
    "limit_absorbed": 0.0,
    "coinsurance_absorbed": 0.0,
    "gross_insured": 0.0,
    "uninsured": 0.0,
}
maximum_errors = {
    "eligible": 0.0,
    "uncovered": 0.0,
    "deductible": 0.0,
    "after_deductible": 0.0,
    "limited": 0.0,
    "limit_absorbed": 0.0,
    "coinsurance": 0.0,
    "gross": 0.0,
    "uninsured": 0.0,
    "uninsured_decomposition": 0.0,
    "loss_conservation": 0.0,
}
maximum_ground_up_loss_ratio = 0.0
maximum_gross_insured_loss_ratio = 0.0
maximum_event_ground_up_loss = 0.0
maximum_event_gross_insured_loss = 0.0
maximum_event_uninsured_loss = 0.0
rows_with_ground_up_loss = 0
rows_with_gross_insured_loss = 0
rows_fully_absorbed_by_deductible = 0
rows_hitting_policy_limit = 0
total_rows = 0
seen_occurrences: set[str] = set()

for sequence_index, row in manifest.iterrows():
    chunk_id = int(row["chunk_id"])
    input_path = resolve_recorded_path(PROJECT_ROOT, row["output_path"])
    input_hash = sha256_file(input_path)
    recorded_input_hash = str(row["output_sha256"])
    if input_hash != recorded_input_hash:
        raise RuntimeError(
            f"Cell 11 chunk {chunk_id:04d} hash mismatch: "
            f"actual={input_hash}, recorded={recorded_input_hash}"
        )

    chunk_output_path = CHUNK_DIR / f"baseline_insured_loss_chunk_{chunk_id:04d}.csv.gz"
    marker_path = MARKER_DIR / f"baseline_insured_loss_chunk_{chunk_id:04d}.json"
    chunk_validation_path = (
        CHUNK_VALIDATION_DIR
        / f"baseline_insured_loss_chunk_{chunk_id:04d}_validation.csv"
    )

    print()
    print("-" * 78)
    print(
        f"BASELINE INSURED LOSS CHUNK {sequence_index + 1} OF {len(manifest)} "
        f"[ID {chunk_id:04d}]"
    )

    marker_valid = False
    if marker_path.exists() and chunk_output_path.exists() and chunk_validation_path.exists():
        try:
            marker = load_json(marker_path)
            chunk_validation_existing = pd.read_csv(chunk_validation_path)
            chunk_validation_existing["passed"] = parse_bool_series(
                chunk_validation_existing["passed"]
            )
            existing_failures = chunk_validation_existing.loc[
                chunk_validation_existing["severity"].astype(str).str.lower().eq(
                    "critical"
                )
                & ~chunk_validation_existing["passed"]
            ]
            marker_valid = (
                marker.get("pipeline_version") == PIPELINE_VERSION
                and marker.get("basis_hash") == basis_hash
                and marker.get("input_chunk_sha256") == input_hash
                and int(marker.get("rows", -1)) == int(row["rows"])
                and marker.get("output_sha256") == sha256_file(chunk_output_path)
                and marker.get("validation_sha256")
                == sha256_file(chunk_validation_path)
                and existing_failures.empty
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        output = pd.read_csv(chunk_output_path, compression="gzip")
        print(
            f"Reused validated insured-loss chunk with {len(output):,} rows."
        )
    else:
        ground_up = pd.read_csv(input_path, compression="gzip")
        required_ground_up_columns = {
            "catalog_year",
            "occurrence_ordinal",
            "occurrence_id",
            "rupture_id",
            "source_type",
            "magnitude",
            "site_ordinal",
            "site_id",
            "building_replacement_value_2022_usd",
            "sampled_total_ground_up_loss_2022_usd",
        }
        missing_ground_up_columns = sorted(
            required_ground_up_columns.difference(ground_up.columns)
        )
        if missing_ground_up_columns:
            raise KeyError(
                f"Cell 11 chunk {chunk_id:04d} is missing columns: "
                f"{missing_ground_up_columns}"
            )

        ground_up["site_ordinal"] = pd.to_numeric(
            ground_up["site_ordinal"], errors="raise"
        ).astype(np.int64)
        ground_up["site_id"] = ground_up["site_id"].astype(str).str.strip()
        ground_up["occurrence_ordinal"] = pd.to_numeric(
            ground_up["occurrence_ordinal"], errors="raise"
        ).astype(np.int64)
        ground_up["occurrence_id"] = (
            ground_up["occurrence_id"].astype(str).str.strip()
        )

        output = ground_up.merge(
            policy_lookup,
            on=["site_ordinal", "site_id"],
            how="left",
            validate="many_to_one",
            indicator=True,
        )
        unmatched = int((output["_merge"] != "both").sum())
        output = output.drop(columns="_merge")
        if unmatched:
            raise RuntimeError(
                f"Chunk {chunk_id:04d} has {unmatched} rows without policy terms."
            )

        ground_up_losses = pd.to_numeric(
            output["sampled_total_ground_up_loss_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)
        replacement_values = pd.to_numeric(
            output["building_replacement_value_2022_usd"], errors="raise"
        ).to_numpy(dtype=np.float64)
        outputs = apply_policy_terms(
            ground_up_loss=ground_up_losses,
            policy_covered=output["policy_covered"].to_numpy(dtype=bool),
            covered_loss_share=pd.to_numeric(
                output["covered_loss_share"], errors="raise"
            ).to_numpy(dtype=np.float64),
            deductible_amount=pd.to_numeric(
                output["deductible_amount_2022_usd"], errors="raise"
            ).to_numpy(dtype=np.float64),
            policy_limit_amount=pd.to_numeric(
                output["policy_limit_amount_2022_usd"], errors="raise"
            ).to_numpy(dtype=np.float64),
            coinsurance_share=pd.to_numeric(
                output["coinsurance_share"], errors="raise"
            ).to_numpy(dtype=np.float64),
        )
        for column, values in outputs.items():
            output[column] = values

        gross = output["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
        uninsured = output["uninsured_loss_2022_usd"].to_numpy(dtype=np.float64)
        deductible = output["deductible_amount_2022_usd"].to_numpy(dtype=np.float64)
        eligible = output["eligible_ground_up_loss_2022_usd"].to_numpy(
            dtype=np.float64
        )
        after_deductible = output["loss_after_deductible_2022_usd"].to_numpy(
            dtype=np.float64
        )
        policy_limit = output["policy_limit_amount_2022_usd"].to_numpy(
            dtype=np.float64
        )

        output["ground_up_loss_ratio_to_replacement"] = np.divide(
            ground_up_losses,
            replacement_values,
            out=np.zeros_like(ground_up_losses),
            where=replacement_values > 0.0,
        )
        output["gross_insured_loss_ratio_to_replacement"] = np.divide(
            gross,
            replacement_values,
            out=np.zeros_like(gross),
            where=replacement_values > 0.0,
        )
        output["insurance_recovery_share_of_ground_up_loss"] = np.divide(
            gross,
            ground_up_losses,
            out=np.zeros_like(gross),
            where=ground_up_losses > 0.0,
        )
        output["uninsured_share_of_ground_up_loss"] = np.divide(
            uninsured,
            ground_up_losses,
            out=np.zeros_like(uninsured),
            where=ground_up_losses > 0.0,
        )
        output["ground_up_loss_positive_indicator"] = (
            ground_up_losses > ROW_TOLERANCE_USD
        ).astype(np.int8)
        output["gross_insured_loss_positive_indicator"] = (
            gross > ROW_TOLERANCE_USD
        ).astype(np.int8)
        output["deductible_fully_absorbs_loss_indicator"] = (
            (eligible > ROW_TOLERANCE_USD)
            & (after_deductible <= ROW_TOLERANCE_USD)
        ).astype(np.int8)
        output["policy_limit_binding_indicator"] = (
            after_deductible > policy_limit + ROW_TOLERANCE_USD
        ).astype(np.int8)

        chunk_checks: list[dict[str, Any]] = []
        expected_chunk_rows = int(row["rows"])
        expected_chunk_occurrences = int(row["occurrences"])
        append_check(
            chunk_checks,
            "output_rows_match_manifest",
            len(output) == expected_chunk_rows,
            f"rows={len(output)}; expected={expected_chunk_rows}",
        )
        append_check(
            chunk_checks,
            "output_occurrences_match_manifest",
            output["occurrence_id"].nunique() == expected_chunk_occurrences,
            (
                f"occurrences={output['occurrence_id'].nunique()}; "
                f"expected={expected_chunk_occurrences}"
            ),
        )
        append_check(
            chunk_checks,
            "occurrence_site_pairs_unique",
            not output.duplicated(["occurrence_id", "site_id"]).any(),
            (
                f"duplicates={int(output.duplicated(['occurrence_id', 'site_id']).sum())}"
            ),
        )
        occurrence_sizes = output.groupby("occurrence_id", sort=False).size()
        append_check(
            chunk_checks,
            "occurrences_have_complete_portfolios",
            occurrence_sizes.eq(expected_buildings).all(),
            (
                f"minimum={int(occurrence_sizes.min())}; "
                f"maximum={int(occurrence_sizes.max())}; expected={expected_buildings}"
            ),
        )
        append_check(
            chunk_checks,
            "ground_up_losses_nonnegative",
            bool(np.all(ground_up_losses >= -ROW_TOLERANCE_USD)),
            f"minimum={float(ground_up_losses.min()):.6f}",
        )
        append_check(
            chunk_checks,
            "ground_up_losses_respect_replacement_value",
            bool(
                np.all(
                    ground_up_losses
                    <= replacement_values + ROW_TOLERANCE_USD
                )
            ),
            (
                "maximum_excess="
                f"{float(np.max(ground_up_losses - replacement_values)):.6e}"
            ),
        )
        append_check(
            chunk_checks,
            "gross_insured_losses_bounded",
            bool(
                np.all(gross >= -ROW_TOLERANCE_USD)
                and np.all(gross <= ground_up_losses + ROW_TOLERANCE_USD)
            ),
            f"minimum={float(gross.min()):.6f}; maximum={float(gross.max()):.6f}",
        )
        append_check(
            chunk_checks,
            "uninsured_losses_bounded",
            bool(
                np.all(uninsured >= -ROW_TOLERANCE_USD)
                and np.all(uninsured <= ground_up_losses + ROW_TOLERANCE_USD)
            ),
            (
                f"minimum={float(uninsured.min()):.6f}; "
                f"maximum={float(uninsured.max()):.6f}"
            ),
        )
        append_check(
            chunk_checks,
            "loss_conservation_holds",
            float(np.max(np.abs(ground_up_losses - gross - uninsured)))
            <= ROW_TOLERANCE_USD,
            (
                "maximum_error="
                f"{float(np.max(np.abs(ground_up_losses - gross - uninsured))):.3e}"
            ),
        )
        uninsured_decomposition = (
            output["uncovered_ground_up_loss_2022_usd"].to_numpy(dtype=np.float64)
            + output["deductible_absorbed_loss_2022_usd"].to_numpy(dtype=np.float64)
            + output["policy_limit_absorbed_loss_2022_usd"].to_numpy(dtype=np.float64)
            + output["coinsurance_absorbed_loss_2022_usd"].to_numpy(dtype=np.float64)
        )
        append_check(
            chunk_checks,
            "uninsured_decomposition_holds",
            float(np.max(np.abs(uninsured - uninsured_decomposition)))
            <= ROW_TOLERANCE_USD,
            (
                "maximum_error="
                f"{float(np.max(np.abs(uninsured - uninsured_decomposition))):.3e}"
            ),
        )
        append_check(
            chunk_checks,
            "baseline_policy_limit_does_not_bind",
            int(output["policy_limit_binding_indicator"].sum()) == 0,
            f"binding_rows={int(output['policy_limit_binding_indicator'].sum())}",
        )

        chunk_validation = pd.DataFrame(chunk_checks)
        chunk_failures = chunk_validation.loc[
            chunk_validation["severity"].eq("critical")
            & ~chunk_validation["passed"]
        ]
        chunk_validation.to_csv(chunk_validation_path, index=False)
        if not chunk_failures.empty:
            raise RuntimeError(
                f"Insured-loss chunk {chunk_id:04d} failed validation. "
                f"Review {chunk_validation_path}."
            )

        write_gzip_csv_deterministic(output, chunk_output_path)
        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "input_chunk_path": str(input_path),
            "input_chunk_sha256": input_hash,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "output_path": str(chunk_output_path),
            "output_sha256": sha256_file(chunk_output_path),
            "validation_path": str(chunk_validation_path),
            "validation_sha256": sha256_file(chunk_validation_path),
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        write_json(marker_path, marker)
        print(
            f"Calculated {len(output):,} rows for "
            f"{output['occurrence_id'].nunique():,} occurrences."
        )

    required_output_columns = {
        "catalog_year",
        "occurrence_ordinal",
        "occurrence_id",
        "rupture_id",
        "source_type",
        "magnitude",
        "site_ordinal",
        "site_id",
        "building_replacement_value_2022_usd",
        "sampled_total_ground_up_loss_2022_usd",
        "policy_scenario_id",
        "policy_id",
        "policy_covered",
        "covered_loss_share",
        "deductible_amount_2022_usd",
        "policy_limit_amount_2022_usd",
        "coinsurance_share",
        "eligible_ground_up_loss_2022_usd",
        "uncovered_ground_up_loss_2022_usd",
        "deductible_absorbed_loss_2022_usd",
        "loss_after_deductible_2022_usd",
        "limited_loss_2022_usd",
        "policy_limit_absorbed_loss_2022_usd",
        "coinsurance_absorbed_loss_2022_usd",
        "gross_insured_loss_2022_usd",
        "uninsured_loss_2022_usd",
        "ground_up_loss_positive_indicator",
        "gross_insured_loss_positive_indicator",
        "deductible_fully_absorbs_loss_indicator",
        "policy_limit_binding_indicator",
    }
    missing_output_columns = sorted(required_output_columns.difference(output.columns))
    if missing_output_columns:
        raise KeyError(
            f"Insured-loss chunk {chunk_id:04d} is missing columns: "
            f"{missing_output_columns}"
        )

    output["occurrence_ordinal"] = pd.to_numeric(
        output["occurrence_ordinal"], errors="raise"
    ).astype(np.int64)
    output["site_ordinal"] = pd.to_numeric(
        output["site_ordinal"], errors="raise"
    ).astype(np.int64)
    output["occurrence_id"] = output["occurrence_id"].astype(str).str.strip()
    output["site_id"] = output["site_id"].astype(str).str.strip()

    occurrence_ordinals = output["occurrence_ordinal"].to_numpy(dtype=np.int64)
    site_ordinals = output["site_ordinal"].to_numpy(dtype=np.int64)
    if np.any(occurrence_ordinals < 0) or np.any(
        occurrence_ordinals >= expected_occurrences
    ):
        raise RuntimeError(
            f"Chunk {chunk_id:04d} has occurrence ordinals outside expected range."
        )
    if np.any(site_ordinals < 0) or np.any(site_ordinals >= expected_buildings):
        raise RuntimeError(
            f"Chunk {chunk_id:04d} has site ordinals outside expected range."
        )

    pair_indices = occurrence_ordinals * expected_buildings + site_ordinals
    if len(np.unique(pair_indices)) != len(pair_indices):
        raise RuntimeError(f"Chunk {chunk_id:04d} contains duplicate ordinal pairs.")
    if seen_pairs[pair_indices].any():
        raise RuntimeError(
            f"Chunk {chunk_id:04d} repeats previously processed occurrence-site pairs."
        )
    seen_pairs[pair_indices] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    ground_up_losses = pd.to_numeric(
        output["sampled_total_ground_up_loss_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    replacement_values = pd.to_numeric(
        output["building_replacement_value_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    policy_covered = parse_bool_series(output["policy_covered"]).to_numpy(dtype=bool)
    covered_share = pd.to_numeric(
        output["covered_loss_share"], errors="raise"
    ).to_numpy(dtype=np.float64)
    deductible_amount = pd.to_numeric(
        output["deductible_amount_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    policy_limit_amount = pd.to_numeric(
        output["policy_limit_amount_2022_usd"], errors="raise"
    ).to_numpy(dtype=np.float64)
    coinsurance_share = pd.to_numeric(
        output["coinsurance_share"], errors="raise"
    ).to_numpy(dtype=np.float64)

    expected_outputs = apply_policy_terms(
        ground_up_loss=ground_up_losses,
        policy_covered=policy_covered,
        covered_loss_share=covered_share,
        deductible_amount=deductible_amount,
        policy_limit_amount=policy_limit_amount,
        coinsurance_share=coinsurance_share,
    )
    for column, expected_values in expected_outputs.items():
        actual_values = pd.to_numeric(output[column], errors="raise").to_numpy(
            dtype=np.float64
        )
        error_name = {
            "eligible_ground_up_loss_2022_usd": "eligible",
            "uncovered_ground_up_loss_2022_usd": "uncovered",
            "deductible_absorbed_loss_2022_usd": "deductible",
            "loss_after_deductible_2022_usd": "after_deductible",
            "limited_loss_2022_usd": "limited",
            "policy_limit_absorbed_loss_2022_usd": "limit_absorbed",
            "coinsurance_absorbed_loss_2022_usd": "coinsurance",
            "gross_insured_loss_2022_usd": "gross",
            "uninsured_loss_2022_usd": "uninsured",
        }[column]
        maximum_errors[error_name] = max(
            maximum_errors[error_name],
            float(np.max(np.abs(actual_values - expected_values))),
        )

    eligible = output["eligible_ground_up_loss_2022_usd"].to_numpy(dtype=np.float64)
    uncovered = output["uncovered_ground_up_loss_2022_usd"].to_numpy(dtype=np.float64)
    deductible_absorbed = output[
        "deductible_absorbed_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    after_deductible = output["loss_after_deductible_2022_usd"].to_numpy(
        dtype=np.float64
    )
    limited = output["limited_loss_2022_usd"].to_numpy(dtype=np.float64)
    limit_absorbed = output[
        "policy_limit_absorbed_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    coinsurance_absorbed = output[
        "coinsurance_absorbed_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    gross = output["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
    uninsured = output["uninsured_loss_2022_usd"].to_numpy(dtype=np.float64)

    uninsured_decomposition = (
        uncovered + deductible_absorbed + limit_absorbed + coinsurance_absorbed
    )
    maximum_errors["uninsured_decomposition"] = max(
        maximum_errors["uninsured_decomposition"],
        float(np.max(np.abs(uninsured - uninsured_decomposition))),
    )
    maximum_errors["loss_conservation"] = max(
        maximum_errors["loss_conservation"],
        float(np.max(np.abs(ground_up_losses - gross - uninsured))),
    )

    aggregate_totals["ground_up"] += float(ground_up_losses.sum(dtype=np.float64))
    aggregate_totals["eligible"] += float(eligible.sum(dtype=np.float64))
    aggregate_totals["uncovered"] += float(uncovered.sum(dtype=np.float64))
    aggregate_totals["deductible"] += float(
        deductible_absorbed.sum(dtype=np.float64)
    )
    aggregate_totals["after_deductible"] += float(
        after_deductible.sum(dtype=np.float64)
    )
    aggregate_totals["limited"] += float(limited.sum(dtype=np.float64))
    aggregate_totals["limit_absorbed"] += float(
        limit_absorbed.sum(dtype=np.float64)
    )
    aggregate_totals["coinsurance_absorbed"] += float(
        coinsurance_absorbed.sum(dtype=np.float64)
    )
    aggregate_totals["gross_insured"] += float(gross.sum(dtype=np.float64))
    aggregate_totals["uninsured"] += float(uninsured.sum(dtype=np.float64))

    maximum_ground_up_loss_ratio = max(
        maximum_ground_up_loss_ratio,
        float(
            np.max(
                np.divide(
                    ground_up_losses,
                    replacement_values,
                    out=np.zeros_like(ground_up_losses),
                    where=replacement_values > 0.0,
                )
            )
        ),
    )
    maximum_gross_insured_loss_ratio = max(
        maximum_gross_insured_loss_ratio,
        float(
            np.max(
                np.divide(
                    gross,
                    replacement_values,
                    out=np.zeros_like(gross),
                    where=replacement_values > 0.0,
                )
            )
        ),
    )

    rows_with_ground_up_loss += int(
        output["ground_up_loss_positive_indicator"].sum()
    )
    rows_with_gross_insured_loss += int(
        output["gross_insured_loss_positive_indicator"].sum()
    )
    rows_fully_absorbed_by_deductible += int(
        output["deductible_fully_absorbs_loss_indicator"].sum()
    )
    rows_hitting_policy_limit += int(output["policy_limit_binding_indicator"].sum())

    source_chunk = output.groupby("source_type", sort=False)[
        [
            "sampled_total_ground_up_loss_2022_usd",
            "gross_insured_loss_2022_usd",
            "uninsured_loss_2022_usd",
        ]
    ].sum()
    for source_type, source_row in source_chunk.iterrows():
        source_totals[f"{source_type}|ground_up"] += float(
            source_row["sampled_total_ground_up_loss_2022_usd"]
        )
        source_totals[f"{source_type}|gross_insured"] += float(
            source_row["gross_insured_loss_2022_usd"]
        )
        source_totals[f"{source_type}|uninsured"] += float(
            source_row["uninsured_loss_2022_usd"]
        )

    event_summary_chunk = summarize_occurrences(output)
    maximum_event_ground_up_loss = max(
        maximum_event_ground_up_loss,
        float(
            event_summary_chunk["sampled_total_ground_up_loss_2022_usd"].max()
        ),
    )
    maximum_event_gross_insured_loss = max(
        maximum_event_gross_insured_loss,
        float(event_summary_chunk["gross_insured_loss_2022_usd"].max()),
    )
    maximum_event_uninsured_loss = max(
        maximum_event_uninsured_loss,
        float(event_summary_chunk["uninsured_loss_2022_usd"].max()),
    )
    event_summary_frames.append(event_summary_chunk)

    keys = list(zip(output["occurrence_id"], output["site_id"]))
    controlled_mask = np.fromiter(
        (key in controlled_keys for key in keys),
        dtype=bool,
        count=len(keys),
    )
    if controlled_mask.any():
        controlled_production_frames.append(output.loc[controlled_mask].copy())

    chunk_paths.append(chunk_output_path)
    total_rows += len(output)
    seen_occurrences.update(output["occurrence_id"].unique().tolist())
    chunk_manifest_rows.append(
        {
            "chunk_sequence": int(sequence_index),
            "chunk_id": chunk_id,
            "rows": int(len(output)),
            "occurrences": int(output["occurrence_id"].nunique()),
            "minimum_occurrence_ordinal": int(output["occurrence_ordinal"].min()),
            "maximum_occurrence_ordinal": int(output["occurrence_ordinal"].max()),
            "sampled_ground_up_loss_sum_2022_usd": float(
                ground_up_losses.sum(dtype=np.float64)
            ),
            "gross_insured_loss_sum_2022_usd": float(gross.sum(dtype=np.float64)),
            "uninsured_loss_sum_2022_usd": float(
                uninsured.sum(dtype=np.float64)
            ),
            "input_path": str(input_path),
            "input_sha256": input_hash,
            "output_path": str(chunk_output_path),
            "output_size_bytes": int(chunk_output_path.stat().st_size),
            "output_sha256": sha256_file(chunk_output_path),
            "marker_path": str(marker_path),
            "validation_path": str(chunk_validation_path),
            "reused": bool(marker_valid),
        }
    )

if len(chunk_paths) != len(manifest):
    raise RuntimeError(
        f"Processed {len(chunk_paths)} chunks but expected {len(manifest)}."
    )

concatenate_gzip_csv_files(chunk_paths, FINAL_BUILDING_LOSS_PATH)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(drop=True)
write_gzip_csv_deterministic(event_summary, FINAL_EVENT_LOSS_PATH)

chunk_manifest = pd.DataFrame(chunk_manifest_rows)
chunk_manifest.to_csv(CHUNK_MANIFEST_PATH, index=False)

if controlled_production_frames:
    controlled_production = pd.concat(controlled_production_frames, ignore_index=True)
else:
    controlled_production = pd.DataFrame()

control_merge = controlled_actual[
    controlled_key_columns + controlled_result_columns
].merge(
    controlled_production[
        controlled_key_columns + controlled_result_columns
    ],
    on=controlled_key_columns,
    how="left",
    suffixes=("_cell3", "_cell4"),
    validate="one_to_one",
    indicator=True,
)
missing_control_rows = int((control_merge["_merge"] != "both").sum())
maximum_control_difference = 0.0
for column in controlled_result_columns:
    cell3_values = pd.to_numeric(
        control_merge[f"{column}_cell3"], errors="coerce"
    ).to_numpy(dtype=np.float64)
    cell4_values = pd.to_numeric(
        control_merge[f"{column}_cell4"], errors="coerce"
    ).to_numpy(dtype=np.float64)
    valid = np.isfinite(cell3_values) & np.isfinite(cell4_values)
    if valid.any():
        maximum_control_difference = max(
            maximum_control_difference,
            float(np.max(np.abs(cell3_values[valid] - cell4_values[valid]))),
        )

expected_ground_up_total = float(
    cell1_handoff.get("ground_up_loss", {}).get("sampled_total_2022_usd", np.nan)
)
if not np.isfinite(expected_ground_up_total):
    expected_ground_up_total = float(manifest["sampled_total_loss_sum_2022_usd"].sum())

portfolio_replacement_value = float(
    cell1_handoff.get("portfolio", {}).get(
        "replacement_value_2022_usd",
        policy_terms["building_replacement_value_2022_usd"].sum(),
    )
)

ground_up_aal = (
    aggregate_totals["ground_up"] / declared_years if declared_years > 0 else np.nan
)
gross_insured_aal = (
    aggregate_totals["gross_insured"] / declared_years
    if declared_years > 0
    else np.nan
)
uninsured_aal = (
    aggregate_totals["uninsured"] / declared_years
    if declared_years > 0
    else np.nan
)
insurance_recovery_share = (
    aggregate_totals["gross_insured"] / aggregate_totals["ground_up"]
    if aggregate_totals["ground_up"] > 0.0
    else 0.0
)
uninsured_share = (
    aggregate_totals["uninsured"] / aggregate_totals["ground_up"]
    if aggregate_totals["ground_up"] > 0.0
    else 0.0
)

reconciliation = pd.DataFrame(
    [
        {
            "loss_measure": "sampled_total_ground_up_loss",
            "building_total_2022_usd": aggregate_totals["ground_up"],
            "event_total_2022_usd": float(
                event_summary["sampled_total_ground_up_loss_2022_usd"].sum()
            ),
            "difference_2022_usd": float(
                aggregate_totals["ground_up"]
                - event_summary["sampled_total_ground_up_loss_2022_usd"].sum()
            ),
        },
        {
            "loss_measure": "gross_insured_loss",
            "building_total_2022_usd": aggregate_totals["gross_insured"],
            "event_total_2022_usd": float(
                event_summary["gross_insured_loss_2022_usd"].sum()
            ),
            "difference_2022_usd": float(
                aggregate_totals["gross_insured"]
                - event_summary["gross_insured_loss_2022_usd"].sum()
            ),
        },
        {
            "loss_measure": "uninsured_loss",
            "building_total_2022_usd": aggregate_totals["uninsured"],
            "event_total_2022_usd": float(
                event_summary["uninsured_loss_2022_usd"].sum()
            ),
            "difference_2022_usd": float(
                aggregate_totals["uninsured"]
                - event_summary["uninsured_loss_2022_usd"].sum()
            ),
        },
    ]
)
reconciliation.to_csv(LOSS_RECONCILIATION_PATH, index=False)

append_check(
    validation_rows,
    "production_rows_match_cell1_handoff",
    total_rows == expected_rows,
    f"rows={total_rows}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "all_occurrence_site_pairs_processed_once",
    bool(seen_pairs.all()),
    f"processed={int(seen_pairs.sum())}; expected={expected_rows}",
)
append_check(
    validation_rows,
    "all_occurrences_have_complete_portfolios",
    bool(np.all(occurrence_counts == expected_buildings)),
    (
        f"minimum={int(occurrence_counts.min())}; "
        f"maximum={int(occurrence_counts.max())}; expected={expected_buildings}"
    ),
)
append_check(
    validation_rows,
    "all_buildings_appear_in_every_occurrence",
    bool(np.all(site_counts == expected_occurrences)),
    (
        f"minimum={int(site_counts.min())}; "
        f"maximum={int(site_counts.max())}; expected={expected_occurrences}"
    ),
)
append_check(
    validation_rows,
    "occurrence_count_matches_handoff",
    len(seen_occurrences) == expected_occurrences,
    f"occurrences={len(seen_occurrences)}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "ground_up_total_reconciles_with_notebook5_handoff",
    abs(aggregate_totals["ground_up"] - expected_ground_up_total)
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"calculated={aggregate_totals['ground_up']:.6f}; "
        f"recorded={expected_ground_up_total:.6f}; "
        f"difference={aggregate_totals['ground_up'] - expected_ground_up_total:.3e}"
    ),
)
append_check(
    validation_rows,
    "gross_and_uninsured_losses_conserve_ground_up_total",
    abs(
        aggregate_totals["ground_up"]
        - aggregate_totals["gross_insured"]
        - aggregate_totals["uninsured"]
    )
    <= AGGREGATE_TOLERANCE_USD,
    (
        "difference="
        f"{aggregate_totals['ground_up'] - aggregate_totals['gross_insured'] - aggregate_totals['uninsured']:.3e}"
    ),
)
append_check(
    validation_rows,
    "uninsured_components_reconcile_to_uninsured_total",
    abs(
        aggregate_totals["uninsured"]
        - aggregate_totals["uncovered"]
        - aggregate_totals["deductible"]
        - aggregate_totals["limit_absorbed"]
        - aggregate_totals["coinsurance_absorbed"]
    )
    <= AGGREGATE_TOLERANCE_USD,
    (
        "difference="
        f"{aggregate_totals['uninsured'] - aggregate_totals['uncovered'] - aggregate_totals['deductible'] - aggregate_totals['limit_absorbed'] - aggregate_totals['coinsurance_absorbed']:.3e}"
    ),
)
append_check(
    validation_rows,
    "event_and_building_totals_reconcile",
    bool(
        np.all(
            np.abs(reconciliation["difference_2022_usd"].to_numpy(dtype=float))
            <= AGGREGATE_TOLERANCE_USD
        )
    ),
    (
        "maximum_difference="
        f"{float(np.abs(reconciliation['difference_2022_usd']).max()):.3e}"
    ),
)
append_check(
    validation_rows,
    "all_row_level_policy_equations_reproduce",
    max(maximum_errors.values()) <= ROW_TOLERANCE_USD,
    (
        f"maximum_error={max(maximum_errors.values()):.3e}; "
        f"errors={maximum_errors}"
    ),
)
append_check(
    validation_rows,
    "cell3_controlled_rows_all_reproduced",
    missing_control_rows == 0
    and len(control_merge) == len(controlled_actual)
    and maximum_control_difference <= ROW_TOLERANCE_USD,
    (
        f"controlled_rows={len(control_merge)}/{len(controlled_actual)}; "
        f"missing={missing_control_rows}; "
        f"maximum_difference={maximum_control_difference:.3e}"
    ),
)
append_check(
    validation_rows,
    "baseline_uncovered_loss_is_zero",
    abs(aggregate_totals["uncovered"]) <= AGGREGATE_TOLERANCE_USD,
    f"uncovered_total={aggregate_totals['uncovered']:.6f}",
)
append_check(
    validation_rows,
    "baseline_policy_limit_absorbed_loss_is_zero",
    abs(aggregate_totals["limit_absorbed"]) <= AGGREGATE_TOLERANCE_USD
    and rows_hitting_policy_limit == 0,
    (
        f"limit_absorbed={aggregate_totals['limit_absorbed']:.6f}; "
        f"binding_rows={rows_hitting_policy_limit}"
    ),
)
append_check(
    validation_rows,
    "baseline_coinsurance_absorbed_loss_is_zero",
    abs(aggregate_totals["coinsurance_absorbed"]) <= AGGREGATE_TOLERANCE_USD,
    (
        "coinsurance_absorbed="
        f"{aggregate_totals['coinsurance_absorbed']:.6f}"
    ),
)
append_check(
    validation_rows,
    "gross_insured_loss_not_greater_than_ground_up_loss",
    aggregate_totals["gross_insured"]
    <= aggregate_totals["ground_up"] + AGGREGATE_TOLERANCE_USD,
    (
        f"gross={aggregate_totals['gross_insured']:.6f}; "
        f"ground_up={aggregate_totals['ground_up']:.6f}"
    ),
)
append_check(
    validation_rows,
    "maximum_ground_up_loss_ratio_respects_physical_cap",
    maximum_ground_up_loss_ratio <= 1.0 + NUMERIC_TOLERANCE,
    f"maximum_ratio={maximum_ground_up_loss_ratio:.12f}",
)
append_check(
    validation_rows,
    "maximum_gross_insured_loss_ratio_respects_policy_cap",
    maximum_gross_insured_loss_ratio <= 1.0 + NUMERIC_TOLERANCE,
    f"maximum_ratio={maximum_gross_insured_loss_ratio:.12f}",
)
append_check(
    validation_rows,
    "final_building_output_exists_and_has_expected_rows",
    FINAL_BUILDING_LOSS_PATH.exists() and total_rows == expected_rows,
    f"path={FINAL_BUILDING_LOSS_PATH}; rows={total_rows}",
)
append_check(
    validation_rows,
    "event_output_has_one_row_per_occurrence",
    len(event_summary) == expected_occurrences
    and not event_summary["occurrence_id"].duplicated().any(),
    (
        f"rows={len(event_summary)}; expected={expected_occurrences}; "
        f"duplicates={int(event_summary['occurrence_id'].duplicated().sum())}"
    ),
)
append_check(
    validation_rows,
    "chunk_manifest_complete",
    len(chunk_manifest) == len(manifest)
    and int(chunk_manifest["rows"].sum()) == expected_rows,
    (
        f"chunks={len(chunk_manifest)}/{len(manifest)}; "
        f"rows={int(chunk_manifest['rows'].sum())}/{expected_rows}"
    ),
)
append_check(
    validation_rows,
    "insurance_recovery_and_uninsured_shares_sum_to_one",
    abs(insurance_recovery_share + uninsured_share - 1.0) <= NUMERIC_TOLERANCE
    if aggregate_totals["ground_up"] > 0.0
    else True,
    (
        f"recovery_share={insurance_recovery_share:.12f}; "
        f"uninsured_share={uninsured_share:.12f}"
    ),
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
]
warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
]
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "policy_scenario": {
        "policy_scenario_id": policy_scenario_id,
        "policy_scenario_name": policy_scenario_name,
    },
    "annual_catalog": {
        "declared_duration_years": declared_years,
        "occurrences": expected_occurrences,
        "rows": total_rows,
        "buildings": expected_buildings,
    },
    "portfolio": {
        "replacement_value_2022_usd": portfolio_replacement_value,
    },
    "production": {
        "chunks": int(len(chunk_manifest)),
        "ground_up_loss_total_2022_usd": aggregate_totals["ground_up"],
        "eligible_ground_up_loss_total_2022_usd": aggregate_totals["eligible"],
        "uncovered_ground_up_loss_total_2022_usd": aggregate_totals["uncovered"],
        "deductible_absorbed_loss_total_2022_usd": aggregate_totals["deductible"],
        "loss_after_deductible_total_2022_usd": aggregate_totals[
            "after_deductible"
        ],
        "policy_limit_absorbed_loss_total_2022_usd": aggregate_totals[
            "limit_absorbed"
        ],
        "coinsurance_absorbed_loss_total_2022_usd": aggregate_totals[
            "coinsurance_absorbed"
        ],
        "gross_insured_loss_total_2022_usd": aggregate_totals["gross_insured"],
        "uninsured_loss_total_2022_usd": aggregate_totals["uninsured"],
        "ground_up_aal_2022_usd": ground_up_aal,
        "gross_insured_preliminary_aal_2022_usd": gross_insured_aal,
        "uninsured_preliminary_aal_2022_usd": uninsured_aal,
        "insurance_recovery_share_of_ground_up_loss": insurance_recovery_share,
        "uninsured_share_of_ground_up_loss": uninsured_share,
        "maximum_ground_up_loss_ratio_to_replacement": maximum_ground_up_loss_ratio,
        "maximum_gross_insured_loss_ratio_to_replacement": maximum_gross_insured_loss_ratio,
        "maximum_occurrence_ground_up_loss_2022_usd": maximum_event_ground_up_loss,
        "maximum_occurrence_gross_insured_loss_2022_usd": maximum_event_gross_insured_loss,
        "maximum_occurrence_uninsured_loss_2022_usd": maximum_event_uninsured_loss,
        "rows_with_ground_up_loss": rows_with_ground_up_loss,
        "rows_with_gross_insured_loss": rows_with_gross_insured_loss,
        "rows_fully_absorbed_by_deductible": rows_fully_absorbed_by_deductible,
        "rows_hitting_policy_limit": rows_hitting_policy_limit,
        "maximum_row_equation_error_2022_usd": max(maximum_errors.values()),
        "maximum_cell3_control_difference_2022_usd": maximum_control_difference,
        "source_totals_2022_usd": dict(source_totals),
    },
    "outputs": {
        "full_building_insured_loss": {
            "path": str(FINAL_BUILDING_LOSS_PATH),
            "rows": total_rows,
            "sha256": sha256_file(FINAL_BUILDING_LOSS_PATH),
            "size_bytes": int(FINAL_BUILDING_LOSS_PATH.stat().st_size),
            "row_granularity": (
                "one annual-catalog occurrence and one portfolio building"
            ),
        },
        "event_insured_loss": {
            "path": str(FINAL_EVENT_LOSS_PATH),
            "rows": int(len(event_summary)),
            "sha256": sha256_file(FINAL_EVENT_LOSS_PATH),
            "size_bytes": int(FINAL_EVENT_LOSS_PATH.stat().st_size),
        },
        "chunk_manifest": str(CHUNK_MANIFEST_PATH),
        "loss_reconciliation": str(LOSS_RECONCILIATION_PATH),
        "formula_specification": str(FORMULA_SPECIFICATION_PATH),
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "validation": {
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warnings.to_dict(orient="records"),
    },
    "next_cell": (
        "Cell 5: construct complete annual gross-insured and uninsured loss series, "
        "insert zero-event years, and calculate insured AAL, OEP, AEP, and PML."
    ),
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 4 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 4 FULL GROSS-INSURED LOSS COMPLETE")
print("=" * 78)
print(f"Production loss rows:              {total_rows:,}")
print(f"Catalog occurrences:               {expected_occurrences:,}")
print(f"Portfolio buildings:               {expected_buildings:,}")
print(f"Production chunks:                 {len(chunk_manifest):,}")
print(f"Sampled ground-up loss total:      ${aggregate_totals['ground_up']:,.0f}")
print(f"Gross insured loss total:          ${aggregate_totals['gross_insured']:,.0f}")
print(f"Uninsured loss total:              ${aggregate_totals['uninsured']:,.0f}")
print(f"Deductible-absorbed loss total:    ${aggregate_totals['deductible']:,.0f}")
print(f"Gross insured preliminary AAL:     ${gross_insured_aal:,.2f}")
print(f"Uninsured preliminary AAL:         ${uninsured_aal:,.2f}")
print(f"Insurance recovery share:          {insurance_recovery_share:.4%}")
print(f"Maximum gross insured occurrence:  ${maximum_event_gross_insured_loss:,.0f}")
print(f"Maximum row equation error:        ${max(maximum_errors.values()):.3e}")
print(f"Maximum Cell 3 control difference: ${maximum_control_difference:.3e}")
print(f"Critical validation checks:        {len(validation.loc[validation['severity'].eq('critical')]):,}")
print(f"Critical failures:                 {len(critical_failures):,}")
print(f"Warnings requiring review:         {len(warnings):,}")
print()
print("Full building-level insured loss:")
print(f"  {FINAL_BUILDING_LOSS_PATH}")
print("Occurrence-level insured loss:")
print(f"  {FINAL_EVENT_LOSS_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: calculate annual gross-insured and uninsured risk metrics.")



FULL ANNUAL-CATALOG BASELINE GROSS-INSURED LOSS
Ground-up rows:            4,996,100
Catalog occurrences:       10,630
Portfolio buildings:       470
Production chunks:         43
Policy scenario:           baseline_full_coverage_10pct_deductible_v1
Deductible application:    per building, per occurrence
Loss basis:                sampled total ground-up repair loss

------------------------------------------------------------------------------
BASELINE INSURED LOSS CHUNK 1 OF 43 [ID 0000]
Reused validated insured-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
BASELINE INSURED LOSS CHUNK 2 OF 43 [ID 0001]
Reused validated insured-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
BASELINE INSURED LOSS CHUNK 3 OF 43 [ID 0021]
Reused validated insured-loss chunk with 117,500 rows.

------------------------------------------------------------------------------
BASELIN

In [7]:
# from pathlib import Path
# import json
# import pandas as pd

# pd.set_option("display.max_columns", None)
# pd.set_option("display.max_colwidth", None)
# pd.set_option("display.width", 240)

# PROJECT_ROOT = Path(
#     r"."
# )

# METADATA_DIR = (
#     PROJECT_ROOT
#     / "data"
#     / "metadata"
#     / "notebook_6_insurance_terms"
# )

# validation_path = (
#     METADATA_DIR
#     / "notebook_6_cell_4_validation.csv"
# )

# summary_path = (
#     METADATA_DIR
#     / "notebook_6_cell_4_summary.json"
# )

# reconciliation_path = (
#     METADATA_DIR
#     / "notebook_6_cell_4_loss_reconciliation.csv"
# )

# chunk_manifest_path = (
#     METADATA_DIR
#     / "notebook_6_cell_4_chunk_manifest.csv"
# )

# if not validation_path.exists():
#     raise FileNotFoundError(validation_path)

# validation = pd.read_csv(validation_path)

# def parse_passed(series):
#     if series.dtype == bool:
#         return series

#     return (
#         series.astype(str)
#         .str.strip()
#         .str.lower()
#         .map(
#             {
#                 "true": True,
#                 "1": True,
#                 "yes": True,
#                 "false": False,
#                 "0": False,
#                 "no": False,
#             }
#         )
#     )

# validation["passed"] = parse_passed(validation["passed"])

# critical_failures = validation.loc[
#     validation["severity"].astype(str).str.lower().eq("critical")
#     & validation["passed"].eq(False)
# ].copy()

# warnings = validation.loc[
#     validation["severity"].astype(str).str.lower().eq("warning")
#     & validation["passed"].eq(False)
# ].copy()

# print("=" * 78)
# print("NOTEBOOK 6 CELL 4 CRITICAL FAILURES")
# print("=" * 78)

# if critical_failures.empty:
#     print("No critical failures found in the validation file.")
# else:
#     display(
#         critical_failures[
#             ["check_id", "severity", "passed", "detail"]
#         ]
#     )

# print()
# print("=" * 78)
# print("NOTEBOOK 6 CELL 4 WARNINGS")
# print("=" * 78)

# if warnings.empty:
#     print("No warnings.")
# else:
#     display(
#         warnings[
#             ["check_id", "severity", "passed", "detail"]
#         ]
#     )

# if reconciliation_path.exists():
#     print()
#     print("=" * 78)
#     print("LOSS RECONCILIATION")
#     print("=" * 78)

#     reconciliation = pd.read_csv(reconciliation_path)
#     display(reconciliation)

# if chunk_manifest_path.exists():
#     manifest = pd.read_csv(chunk_manifest_path)

#     print()
#     print("=" * 78)
#     print("CHUNK MANIFEST SUMMARY")
#     print("=" * 78)

#     print(f"Manifest rows: {len(manifest):,}")
#     print(f"Columns: {manifest.columns.tolist()}")

#     status_columns = [
#         column
#         for column in manifest.columns
#         if any(
#             token in column.lower()
#             for token in ["status", "valid", "passed", "complete"]
#         )
#     ]

#     for column in status_columns:
#         print()
#         print(f"{column}:")
#         print(manifest[column].value_counts(dropna=False))

# if summary_path.exists():
#     summary = json.loads(summary_path.read_text(encoding="utf-8"))

#     print()
#     print("=" * 78)
#     print("SUMMARY-RECORDED FAILURES")
#     print("=" * 78)

#     summary_failures = summary.get("critical_failures", [])

#     if summary_failures:
#         display(pd.DataFrame(summary_failures))
#     else:
#         print("No critical failures recorded in the summary.")

In [8]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell5_annual_insured_risk_metrics_v1"
DOLLAR_YEAR = 2022
RETURN_PERIODS_YEARS = [
    50,
    100,
    200,
    250,
    500,
    1_000,
    2_000,
    2_500,
    5_000,
    10_000,
    20_000,
    50_000,
    100_000,
    200_000,
    500_000,
    1_000_000,
    2_000_000,
]
NUMERIC_ATOL_USD = 1e-4
AGGREGATE_ATOL_USD = 0.01
TAIL_SUPPORT_WARNING_RANK = 20


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_cell_4_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def prior_validation_status(path: Path) -> tuple[bool, int, int]:
    table = pd.read_csv(path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")

    table["passed"] = parse_bool_series(table["passed"])
    severity = table["severity"].astype(str).str.strip().str.lower()
    failures = table.loc[severity.eq("critical") & ~table["passed"]]
    warnings = table.loc[severity.eq("warning") & ~table["passed"]]
    return failures.empty, int(len(failures)), int(len(warnings))


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_output,
            compresslevel=6,
            mtime=0,
        ) as compressed_output:
            with io.TextIOWrapper(
                compressed_output,
                encoding="utf-8",
                newline="",
            ) as text_output:
                frame.to_csv(text_output, index=False, lineterminator="\n")
    temporary.replace(path)


def empirical_pml(
    losses: np.ndarray,
    return_period_years: int,
) -> tuple[float, int, float]:
    n_years = int(losses.size)
    if return_period_years <= 0 or return_period_years > n_years:
        raise ValueError(
            f"Return period must be in [1, {n_years}], "
            f"received {return_period_years}."
        )

    descending_rank = max(1, int(math.ceil(n_years / return_period_years)))
    ordered = np.sort(np.asarray(losses, dtype=np.float64))[::-1]
    loss = float(ordered[descending_rank - 1])
    empirical_aep = descending_rank / n_years
    return loss, descending_rank, empirical_aep


def build_ep_curve(
    losses: np.ndarray,
    loss_measure: str,
    curve_type: str,
    n_years: int,
) -> pd.DataFrame:
    positive = np.asarray(losses, dtype=np.float64)
    positive = positive[positive > 0.0]
    if positive.size == 0:
        return pd.DataFrame(
            columns=[
                "loss_measure",
                "curve_type",
                "descending_rank",
                "annual_exceedance_probability",
                "return_period_years",
                "loss_2022_usd",
            ]
        )

    ordered = np.sort(positive)[::-1]
    ranks = np.arange(1, ordered.size + 1, dtype=np.int64)
    return pd.DataFrame(
        {
            "loss_measure": loss_measure,
            "curve_type": curve_type,
            "descending_rank": ranks,
            "annual_exceedance_probability": ranks / n_years,
            "return_period_years": n_years / ranks,
            "loss_2022_usd": ordered,
        }
    )


def annual_statistics(
    losses: np.ndarray,
    loss_measure: str,
    curve_type: str,
) -> dict[str, Any]:
    values = np.asarray(losses, dtype=np.float64)
    positive = values[values > 0.0]
    mean_loss = float(np.mean(values))
    standard_deviation = float(np.std(values, ddof=0))
    return {
        "loss_measure": loss_measure,
        "curve_type": curve_type,
        "years": int(values.size),
        "positive_loss_years": int(positive.size),
        "zero_loss_years": int(values.size - positive.size),
        "annual_probability_of_positive_loss": float(positive.size / values.size),
        "mean_annual_loss_2022_usd": mean_loss,
        "annual_loss_standard_deviation_2022_usd": standard_deviation,
        "annual_loss_cov": (
            float(standard_deviation / mean_loss) if mean_loss > 0.0 else 0.0
        ),
        "mean_positive_year_loss_2022_usd": (
            float(np.mean(positive)) if positive.size else 0.0
        ),
        "median_positive_year_loss_2022_usd": (
            float(np.median(positive)) if positive.size else 0.0
        ),
        "maximum_annual_loss_2022_usd": float(np.max(values)),
    }


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_baseline_insured_risk"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CELL1_HANDOFF_PATH = METADATA_DIR / "notebook_6_input_handoff.json"
CELL4_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_4_validation.csv"
CELL4_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_4_summary.json"

ANNUAL_SERIES_PATH = OUTPUT_DIR / "baseline_insurance_annual_loss_series.csv.gz"
EP_CURVE_PATH = OUTPUT_DIR / "baseline_insurance_exceedance_probability_curve.csv.gz"
PML_TABLE_PATH = OUTPUT_DIR / "baseline_insurance_pml_table.csv"
RISK_METRICS_PATH = OUTPUT_DIR / "baseline_insurance_risk_metrics.csv"
SOURCE_AAL_PATH = OUTPUT_DIR / "baseline_insurance_source_aal_summary.csv"
RECOVERY_BY_RETURN_PERIOD_PATH = (
    OUTPUT_DIR / "baseline_insurance_recovery_by_return_period.csv"
)
AEP_PLOT_PNG_PATH = PLOT_DIR / "baseline_insurance_aep_comparison.png"
AEP_PLOT_PDF_PATH = PLOT_DIR / "baseline_insurance_aep_comparison.pdf"
OEP_PLOT_PNG_PATH = PLOT_DIR / "baseline_insurance_oep_comparison.png"
OEP_PLOT_PDF_PATH = PLOT_DIR / "baseline_insurance_oep_comparison.pdf"
RECOVERY_PLOT_PNG_PATH = PLOT_DIR / "baseline_insurance_recovery_by_return_period.png"
RECOVERY_PLOT_PDF_PATH = PLOT_DIR / "baseline_insurance_recovery_by_return_period.pdf"
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_5_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_5_summary.json"

validation_rows: list[dict[str, Any]] = []
required_inputs = [
    CELL1_HANDOFF_PATH,
    CELL4_VALIDATION_PATH,
    CELL4_SUMMARY_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")

cell4_passed, cell4_failures, cell4_warnings = prior_validation_status(
    CELL4_VALIDATION_PATH
)
append_check(
    validation_rows,
    "cell4_critical_validation_passed",
    cell4_passed,
    f"critical_failures={cell4_failures}",
)
append_check(
    validation_rows,
    "cell4_has_no_unresolved_warnings",
    cell4_warnings == 0,
    f"warnings={cell4_warnings}",
)

cell1_handoff = load_json(CELL1_HANDOFF_PATH)
cell4_summary = load_json(CELL4_SUMMARY_PATH)
append_check(
    validation_rows,
    "cell1_handoff_declares_success",
    bool(cell1_handoff.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell1_handoff.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell4_summary_declares_success",
    bool(cell4_summary.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell4_summary.get('all_critical_checks_passed')}",
)

annual_catalog = cell1_handoff["annual_catalog"]
declared_years = int(annual_catalog["declared_duration_years"])
expected_occurrences = int(annual_catalog["occurrences"])
expected_occupied_years = int(annual_catalog["occupied_years"])
expected_multiple_event_years = int(annual_catalog["multiple_event_years"])
expected_zero_event_years = int(annual_catalog["zero_event_years"])
expected_buildings = int(cell1_handoff["portfolio"]["buildings"])
portfolio_replacement_value = float(
    cell1_handoff["portfolio"]["replacement_value_2022_usd"]
)
policy_scenario_id = str(
    cell4_summary["policy_scenario"]["policy_scenario_id"]
)
policy_scenario_name = str(
    cell4_summary["policy_scenario"]["policy_scenario_name"]
)

append_check(
    validation_rows,
    "declared_catalog_duration_positive",
    declared_years > 0,
    f"years={declared_years}",
)
append_check(
    validation_rows,
    "return_periods_supported_by_catalog",
    max(RETURN_PERIODS_YEARS) <= declared_years,
    (
        f"maximum_return_period={max(RETURN_PERIODS_YEARS)}; "
        f"catalog_years={declared_years}"
    ),
)

cell4_event_record = cell4_summary["outputs"]["event_insured_loss"]
event_loss_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(cell4_event_record["path"]),
)
actual_event_hash = sha256_file(event_loss_path)
append_check(
    validation_rows,
    "cell4_event_loss_hash_matches",
    actual_event_hash == str(cell4_event_record["sha256"]),
    f"expected={cell4_event_record['sha256']}; actual={actual_event_hash}",
)

notebook5_annual_record = cell1_handoff["inputs"]["annual_ground_up_loss_series"]
notebook5_annual_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(notebook5_annual_record["path"]),
)
notebook5_annual_hash = sha256_file(notebook5_annual_path)
append_check(
    validation_rows,
    "notebook5_annual_series_hash_matches_handoff",
    notebook5_annual_hash == str(notebook5_annual_record["sha256"]),
    f"expected={notebook5_annual_record['sha256']}; actual={notebook5_annual_hash}",
)

notebook5_pml_record = cell1_handoff["inputs"]["ground_up_pml_table"]
notebook5_pml_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(notebook5_pml_record["path"]),
)
notebook5_pml_hash = sha256_file(notebook5_pml_path)
append_check(
    validation_rows,
    "notebook5_pml_hash_matches_handoff",
    notebook5_pml_hash == str(notebook5_pml_record["sha256"]),
    f"expected={notebook5_pml_record['sha256']}; actual={notebook5_pml_hash}",
)

events = pd.read_csv(event_loss_path, compression="gzip")
required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "buildings",
    "portfolio_replacement_value_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "deductible_absorbed_loss_2022_usd",
    "gross_insured_loss_2022_usd",
    "uninsured_loss_2022_usd",
    "policy_limit_absorbed_loss_2022_usd",
    "coinsurance_absorbed_loss_2022_usd",
}
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "event_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise KeyError(f"Cell 4 event-loss file is missing columns: {missing_event_columns}")

numeric_event_columns = [
    "catalog_year",
    "buildings",
    "portfolio_replacement_value_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
    "deductible_absorbed_loss_2022_usd",
    "gross_insured_loss_2022_usd",
    "uninsured_loss_2022_usd",
    "policy_limit_absorbed_loss_2022_usd",
    "coinsurance_absorbed_loss_2022_usd",
]
for column in numeric_event_columns:
    events[column] = pd.to_numeric(events[column], errors="raise")
events["catalog_year"] = events["catalog_year"].astype(np.int64)
events["buildings"] = events["buildings"].astype(np.int64)
events["occurrence_id"] = events["occurrence_id"].astype(str).str.strip()
events["source_type"] = events["source_type"].astype(str).str.strip()

append_check(
    validation_rows,
    "event_summary_occurrence_count_matches",
    len(events) == expected_occurrences and events["occurrence_id"].is_unique,
    (
        f"rows={len(events)}; expected={expected_occurrences}; "
        f"unique={events['occurrence_id'].nunique()}"
    ),
)
append_check(
    validation_rows,
    "event_summary_has_all_buildings",
    events["buildings"].eq(expected_buildings).all(),
    (
        f"minimum={events['buildings'].min()}; "
        f"maximum={events['buildings'].max()}; expected={expected_buildings}"
    ),
)
append_check(
    validation_rows,
    "catalog_year_labels_within_declared_duration",
    events["catalog_year"].between(1, declared_years).all(),
    (
        f"minimum={events['catalog_year'].min()}; "
        f"maximum={events['catalog_year'].max()}"
    ),
)

loss_columns = [
    "sampled_total_ground_up_loss_2022_usd",
    "deductible_absorbed_loss_2022_usd",
    "gross_insured_loss_2022_usd",
    "uninsured_loss_2022_usd",
    "policy_limit_absorbed_loss_2022_usd",
    "coinsurance_absorbed_loss_2022_usd",
]
loss_matrix = events[loss_columns].to_numpy(dtype=np.float64)
append_check(
    validation_rows,
    "event_losses_finite_and_nonnegative",
    np.isfinite(loss_matrix).all() and np.all(loss_matrix >= 0.0),
    "all occurrence-level insurance loss measures checked",
)

event_ground_up = events["sampled_total_ground_up_loss_2022_usd"].to_numpy(
    dtype=np.float64
)
event_gross = events["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
event_uninsured = events["uninsured_loss_2022_usd"].to_numpy(dtype=np.float64)
event_deductible = events["deductible_absorbed_loss_2022_usd"].to_numpy(
    dtype=np.float64
)
event_conservation_error = np.abs(event_ground_up - event_gross - event_uninsured)
append_check(
    validation_rows,
    "event_ground_up_equals_gross_plus_uninsured",
    float(event_conservation_error.max()) <= NUMERIC_ATOL_USD,
    f"maximum_error={event_conservation_error.max():.3e}",
)
append_check(
    validation_rows,
    "baseline_uninsured_equals_deductible_absorbed",
    np.allclose(
        event_uninsured,
        event_deductible,
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    f"maximum_error={np.max(np.abs(event_uninsured - event_deductible)):.3e}",
)
append_check(
    validation_rows,
    "baseline_limit_and_coinsurance_absorption_zero",
    (
        events["policy_limit_absorbed_loss_2022_usd"].max()
        <= NUMERIC_ATOL_USD
        and events["coinsurance_absorbed_loss_2022_usd"].max()
        <= NUMERIC_ATOL_USD
    ),
    (
        "maximum_limit_absorbed="
        f"{events['policy_limit_absorbed_loss_2022_usd'].max():.3e}; "
        "maximum_coinsurance_absorbed="
        f"{events['coinsurance_absorbed_loss_2022_usd'].max():.3e}"
    ),
)

annual_occupied = (
    events.groupby("catalog_year", sort=True)
    .agg(
        event_count=("occurrence_id", "size"),
        ground_up_aep_2022_usd=(
            "sampled_total_ground_up_loss_2022_usd",
            "sum",
        ),
        ground_up_oep_2022_usd=(
            "sampled_total_ground_up_loss_2022_usd",
            "max",
        ),
        gross_insured_aep_2022_usd=("gross_insured_loss_2022_usd", "sum"),
        gross_insured_oep_2022_usd=("gross_insured_loss_2022_usd", "max"),
        uninsured_aep_2022_usd=("uninsured_loss_2022_usd", "sum"),
        uninsured_oep_2022_usd=("uninsured_loss_2022_usd", "max"),
    )
    .reset_index()
)

catalog_year = np.arange(1, declared_years + 1, dtype=np.int64)
event_count = np.zeros(declared_years, dtype=np.int16)
ground_up_aep = np.zeros(declared_years, dtype=np.float64)
ground_up_oep = np.zeros(declared_years, dtype=np.float64)
gross_aep = np.zeros(declared_years, dtype=np.float64)
gross_oep = np.zeros(declared_years, dtype=np.float64)
uninsured_aep = np.zeros(declared_years, dtype=np.float64)
uninsured_oep = np.zeros(declared_years, dtype=np.float64)

positions = annual_occupied["catalog_year"].to_numpy(dtype=np.int64) - 1
event_count[positions] = annual_occupied["event_count"].to_numpy(dtype=np.int16)
ground_up_aep[positions] = annual_occupied["ground_up_aep_2022_usd"].to_numpy(
    dtype=np.float64
)
ground_up_oep[positions] = annual_occupied["ground_up_oep_2022_usd"].to_numpy(
    dtype=np.float64
)
gross_aep[positions] = annual_occupied["gross_insured_aep_2022_usd"].to_numpy(
    dtype=np.float64
)
gross_oep[positions] = annual_occupied["gross_insured_oep_2022_usd"].to_numpy(
    dtype=np.float64
)
uninsured_aep[positions] = annual_occupied["uninsured_aep_2022_usd"].to_numpy(
    dtype=np.float64
)
uninsured_oep[positions] = annual_occupied["uninsured_oep_2022_usd"].to_numpy(
    dtype=np.float64
)

annual_series = pd.DataFrame(
    {
        "catalog_year": catalog_year,
        "event_count": event_count,
        "zero_event_year": event_count == 0,
        "ground_up_aep_2022_usd": ground_up_aep,
        "ground_up_oep_2022_usd": ground_up_oep,
        "gross_insured_aep_2022_usd": gross_aep,
        "gross_insured_oep_2022_usd": gross_oep,
        "uninsured_aep_2022_usd": uninsured_aep,
        "uninsured_oep_2022_usd": uninsured_oep,
    }
)
write_gzip_csv_deterministic(annual_series, ANNUAL_SERIES_PATH)

occupied_years = int(np.count_nonzero(event_count))
zero_event_years = int(np.count_nonzero(event_count == 0))
multiple_event_years = int(np.count_nonzero(event_count > 1))
maximum_events_in_year = int(event_count.max())

append_check(
    validation_rows,
    "annual_series_contains_all_catalog_years",
    len(annual_series) == declared_years
    and int(annual_series["catalog_year"].iloc[0]) == 1
    and int(annual_series["catalog_year"].iloc[-1]) == declared_years,
    (
        f"rows={len(annual_series)}; first={annual_series['catalog_year'].iloc[0]}; "
        f"last={annual_series['catalog_year'].iloc[-1]}"
    ),
)
append_check(
    validation_rows,
    "event_counts_reconcile",
    int(event_count.sum()) == expected_occurrences,
    f"annual_sum={int(event_count.sum())}; expected={expected_occurrences}",
)
append_check(
    validation_rows,
    "occupied_year_count_matches_handoff",
    occupied_years == expected_occupied_years,
    f"actual={occupied_years}; expected={expected_occupied_years}",
)
append_check(
    validation_rows,
    "zero_event_year_count_matches_handoff",
    zero_event_years == expected_zero_event_years,
    f"actual={zero_event_years}; expected={expected_zero_event_years}",
)
append_check(
    validation_rows,
    "multiple_event_year_count_matches_handoff",
    multiple_event_years == expected_multiple_event_years,
    f"actual={multiple_event_years}; expected={expected_multiple_event_years}",
)

annual_ground_up_conservation_error = np.abs(
    ground_up_aep - gross_aep - uninsured_aep
)
append_check(
    validation_rows,
    "annual_aep_ground_up_equals_gross_plus_uninsured",
    float(annual_ground_up_conservation_error.max()) <= NUMERIC_ATOL_USD,
    f"maximum_error={annual_ground_up_conservation_error.max():.3e}",
)

for measure_name, aep_values, oep_values in [
    ("ground_up", ground_up_aep, ground_up_oep),
    ("gross_insured", gross_aep, gross_oep),
    ("uninsured", uninsured_aep, uninsured_oep),
]:
    append_check(
        validation_rows,
        f"{measure_name}_aep_not_less_than_oep",
        np.all(aep_values + NUMERIC_ATOL_USD >= oep_values),
        f"minimum_difference={np.min(aep_values - oep_values):.6f}",
    )
    append_check(
        validation_rows,
        f"{measure_name}_single_and_zero_event_years_have_equal_aep_oep",
        np.allclose(
            aep_values[event_count <= 1],
            oep_values[event_count <= 1],
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        ),
        f"years_checked={int(np.count_nonzero(event_count <= 1))}",
    )

append_check(
    validation_rows,
    "annual_ground_up_total_matches_event_total",
    math.isclose(
        float(ground_up_aep.sum()),
        float(event_ground_up.sum()),
        rel_tol=1e-12,
        abs_tol=AGGREGATE_ATOL_USD,
    ),
    f"annual={ground_up_aep.sum():.6f}; event={event_ground_up.sum():.6f}",
)
append_check(
    validation_rows,
    "annual_gross_total_matches_event_total",
    math.isclose(
        float(gross_aep.sum()),
        float(event_gross.sum()),
        rel_tol=1e-12,
        abs_tol=AGGREGATE_ATOL_USD,
    ),
    f"annual={gross_aep.sum():.6f}; event={event_gross.sum():.6f}",
)
append_check(
    validation_rows,
    "annual_uninsured_total_matches_event_total",
    math.isclose(
        float(uninsured_aep.sum()),
        float(event_uninsured.sum()),
        rel_tol=1e-12,
        abs_tol=AGGREGATE_ATOL_USD,
    ),
    f"annual={uninsured_aep.sum():.6f}; event={event_uninsured.sum():.6f}",
)

notebook5_annual = pd.read_csv(
    notebook5_annual_path,
    compression="gzip",
    usecols=[
        "catalog_year",
        "event_count",
        "sampled_aep_total_ground_up_loss_2022_usd",
        "sampled_oep_total_ground_up_loss_2022_usd",
    ],
)
notebook5_annual["catalog_year"] = pd.to_numeric(
    notebook5_annual["catalog_year"], errors="raise"
).astype(np.int64)
notebook5_annual["event_count"] = pd.to_numeric(
    notebook5_annual["event_count"], errors="raise"
).astype(np.int16)
append_check(
    validation_rows,
    "notebook5_annual_series_rows_match",
    len(notebook5_annual) == declared_years,
    f"rows={len(notebook5_annual)}; expected={declared_years}",
)
append_check(
    validation_rows,
    "ground_up_annual_series_reproduces_notebook5",
    (
        np.array_equal(
            notebook5_annual["catalog_year"].to_numpy(dtype=np.int64),
            catalog_year,
        )
        and np.array_equal(
            notebook5_annual["event_count"].to_numpy(dtype=np.int16),
            event_count,
        )
        and np.allclose(
            notebook5_annual[
                "sampled_aep_total_ground_up_loss_2022_usd"
            ].to_numpy(dtype=np.float64),
            ground_up_aep,
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        )
        and np.allclose(
            notebook5_annual[
                "sampled_oep_total_ground_up_loss_2022_usd"
            ].to_numpy(dtype=np.float64),
            ground_up_oep,
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        )
    ),
    (
        "maximum_aep_difference="
        f"{np.max(np.abs(notebook5_annual['sampled_aep_total_ground_up_loss_2022_usd'].to_numpy(dtype=np.float64) - ground_up_aep)):.3e}; "
        "maximum_oep_difference="
        f"{np.max(np.abs(notebook5_annual['sampled_oep_total_ground_up_loss_2022_usd'].to_numpy(dtype=np.float64) - ground_up_oep)):.3e}"
    ),
)
del notebook5_annual

calculated_aal = {
    "ground_up": float(ground_up_aep.mean()),
    "gross_insured": float(gross_aep.mean()),
    "uninsured": float(uninsured_aep.mean()),
}
cell4_production = cell4_summary["production"]
append_check(
    validation_rows,
    "ground_up_aal_matches_notebook5_handoff",
    math.isclose(
        calculated_aal["ground_up"],
        float(cell1_handoff["ground_up_loss"]["sampled_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell5={calculated_aal['ground_up']:.9f}; "
        f"handoff={float(cell1_handoff['ground_up_loss']['sampled_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "gross_insured_aal_matches_cell4",
    math.isclose(
        calculated_aal["gross_insured"],
        float(cell4_production["gross_insured_preliminary_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell5={calculated_aal['gross_insured']:.9f}; "
        f"cell4={float(cell4_production['gross_insured_preliminary_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "uninsured_aal_matches_cell4",
    math.isclose(
        calculated_aal["uninsured"],
        float(cell4_production["uninsured_preliminary_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell5={calculated_aal['uninsured']:.9f}; "
        f"cell4={float(cell4_production['uninsured_preliminary_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "aal_conservation",
    abs(
        calculated_aal["ground_up"]
        - calculated_aal["gross_insured"]
        - calculated_aal["uninsured"]
    )
    <= NUMERIC_ATOL_USD,
    (
        f"ground_up={calculated_aal['ground_up']:.9f}; "
        f"gross={calculated_aal['gross_insured']:.9f}; "
        f"uninsured={calculated_aal['uninsured']:.9f}"
    ),
)

loss_vectors = {
    ("ground_up", "AEP"): ground_up_aep,
    ("ground_up", "OEP"): ground_up_oep,
    ("gross_insured", "AEP"): gross_aep,
    ("gross_insured", "OEP"): gross_oep,
    ("uninsured", "AEP"): uninsured_aep,
    ("uninsured", "OEP"): uninsured_oep,
}

ep_curve = pd.concat(
    [
        build_ep_curve(values, measure, curve_type, declared_years)
        for (measure, curve_type), values in loss_vectors.items()
    ],
    ignore_index=True,
)
write_gzip_csv_deterministic(ep_curve, EP_CURVE_PATH)

pml_rows: list[dict[str, Any]] = []
for return_period in RETURN_PERIODS_YEARS:
    row: dict[str, Any] = {
        "return_period_years": int(return_period),
        "target_annual_exceedance_probability": 1.0 / return_period,
    }
    supporting_ranks: list[int] = []
    for (measure, curve_type), values in loss_vectors.items():
        loss, rank, empirical_aep = empirical_pml(values, return_period)
        prefix = f"{measure}_{curve_type.lower()}"
        row[f"{prefix}_pml_2022_usd"] = loss
        row[f"{prefix}_supporting_descending_rank"] = rank
        row[f"{prefix}_empirical_aep"] = empirical_aep
        supporting_ranks.append(rank)

    for curve_type in ["aep", "oep"]:
        ground_up_loss = float(row[f"ground_up_{curve_type}_pml_2022_usd"])
        gross_loss = float(row[f"gross_insured_{curve_type}_pml_2022_usd"])
        uninsured_loss = float(row[f"uninsured_{curve_type}_pml_2022_usd"])
        row[f"insurance_recovery_share_{curve_type}"] = (
            gross_loss / ground_up_loss if ground_up_loss > 0.0 else 0.0
        )
        row[f"uninsured_share_{curve_type}"] = (
            uninsured_loss / ground_up_loss if ground_up_loss > 0.0 else 0.0
        )

    row["minimum_supporting_descending_rank"] = min(supporting_ranks)
    row["tail_support_flag"] = (
        "thin_tail_support"
        if min(supporting_ranks) < TAIL_SUPPORT_WARNING_RANK
        else "adequate_order_statistic_support"
    )
    pml_rows.append(row)

pml_table = pd.DataFrame(pml_rows)
pml_table.to_csv(PML_TABLE_PATH, index=False)

for measure in ["ground_up", "gross_insured", "uninsured"]:
    for curve_type in ["aep", "oep"]:
        column = f"{measure}_{curve_type}_pml_2022_usd"
        differences = np.diff(pml_table[column].to_numpy(dtype=np.float64))
        append_check(
            validation_rows,
            f"{measure}_{curve_type}_pml_nondecreasing_with_return_period",
            np.all(differences >= -NUMERIC_ATOL_USD),
            (
                f"minimum_difference={float(differences.min()) if differences.size else 0.0:.6f}"
            ),
        )

for measure in ["ground_up", "gross_insured", "uninsured"]:
    append_check(
        validation_rows,
        f"{measure}_aep_pml_not_less_than_oep_pml",
        np.all(
            pml_table[f"{measure}_aep_pml_2022_usd"].to_numpy(dtype=float)
            + NUMERIC_ATOL_USD
            >= pml_table[f"{measure}_oep_pml_2022_usd"].to_numpy(dtype=float)
        ),
        "checked all return periods",
    )

for curve_type in ["aep", "oep"]:
    ground_up_column = f"ground_up_{curve_type}_pml_2022_usd"
    gross_column = f"gross_insured_{curve_type}_pml_2022_usd"
    uninsured_column = f"uninsured_{curve_type}_pml_2022_usd"
    append_check(
        validation_rows,
        f"gross_{curve_type}_pml_not_greater_than_ground_up",
        np.all(
            pml_table[gross_column].to_numpy(dtype=float)
            <= pml_table[ground_up_column].to_numpy(dtype=float)
            + NUMERIC_ATOL_USD
        ),
        "checked all return periods",
    )
    append_check(
        validation_rows,
        f"uninsured_{curve_type}_pml_not_greater_than_ground_up",
        np.all(
            pml_table[uninsured_column].to_numpy(dtype=float)
            <= pml_table[ground_up_column].to_numpy(dtype=float)
            + NUMERIC_ATOL_USD
        ),
        "checked all return periods",
    )
    recovery = pml_table[f"insurance_recovery_share_{curve_type}"].to_numpy(
        dtype=float
    )
    uninsured_share = pml_table[f"uninsured_share_{curve_type}"].to_numpy(
        dtype=float
    )
    append_check(
        validation_rows,
        f"{curve_type}_pml_recovery_shares_bounded",
        (
            np.all((recovery >= -1e-12) & (recovery <= 1.0 + 1e-12))
            and np.all(
                (uninsured_share >= -1e-12)
                & (uninsured_share <= 1.0 + 1e-12)
            )
        ),
        (
            f"recovery_range=({recovery.min():.6f}, {recovery.max():.6f}); "
            f"uninsured_range=({uninsured_share.min():.6f}, {uninsured_share.max():.6f})"
        ),
    )

notebook5_pml = pd.read_csv(notebook5_pml_path)
notebook5_pml["return_period_years"] = pd.to_numeric(
    notebook5_pml["return_period_years"], errors="raise"
).astype(np.int64)
comparison = pml_table.merge(
    notebook5_pml[
        [
            "return_period_years",
            "sampled_aep_pml_2022_usd",
            "sampled_oep_pml_2022_usd",
        ]
    ],
    on="return_period_years",
    how="left",
    validate="one_to_one",
)
append_check(
    validation_rows,
    "ground_up_pml_reproduces_notebook5",
    (
        comparison["sampled_aep_pml_2022_usd"].notna().all()
        and comparison["sampled_oep_pml_2022_usd"].notna().all()
        and np.allclose(
            comparison["ground_up_aep_pml_2022_usd"],
            comparison["sampled_aep_pml_2022_usd"],
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        )
        and np.allclose(
            comparison["ground_up_oep_pml_2022_usd"],
            comparison["sampled_oep_pml_2022_usd"],
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        )
    ),
    (
        "maximum_aep_difference="
        f"{np.max(np.abs(comparison['ground_up_aep_pml_2022_usd'] - comparison['sampled_aep_pml_2022_usd'])):.3e}; "
        "maximum_oep_difference="
        f"{np.max(np.abs(comparison['ground_up_oep_pml_2022_usd'] - comparison['sampled_oep_pml_2022_usd'])):.3e}"
    ),
)

risk_metrics = pd.DataFrame(
    [
        annual_statistics(values, measure, curve_type)
        for (measure, curve_type), values in loss_vectors.items()
    ]
)
risk_metrics["aal_2022_usd"] = np.where(
    risk_metrics["curve_type"].eq("AEP"),
    risk_metrics["mean_annual_loss_2022_usd"],
    np.nan,
)
risk_metrics["portfolio_replacement_value_2022_usd"] = (
    portfolio_replacement_value
)
risk_metrics["maximum_annual_loss_ratio"] = (
    risk_metrics["maximum_annual_loss_2022_usd"]
    / portfolio_replacement_value
)
risk_metrics.to_csv(RISK_METRICS_PATH, index=False)

source_rows: list[dict[str, Any]] = []
for source_type, group in events.groupby("source_type", sort=True):
    ground_up_total = float(
        group["sampled_total_ground_up_loss_2022_usd"].sum()
    )
    gross_total = float(group["gross_insured_loss_2022_usd"].sum())
    uninsured_total = float(group["uninsured_loss_2022_usd"].sum())
    source_rows.append(
        {
            "source_type": source_type,
            "occurrences": int(len(group)),
            "ground_up_loss_total_2022_usd": ground_up_total,
            "gross_insured_loss_total_2022_usd": gross_total,
            "uninsured_loss_total_2022_usd": uninsured_total,
            "ground_up_aal_2022_usd": ground_up_total / declared_years,
            "gross_insured_aal_2022_usd": gross_total / declared_years,
            "uninsured_aal_2022_usd": uninsured_total / declared_years,
            "insurance_recovery_share": (
                gross_total / ground_up_total if ground_up_total > 0.0 else 0.0
            ),
            "share_of_portfolio_ground_up_aal": (
                ground_up_total / float(event_ground_up.sum())
                if event_ground_up.sum() > 0.0
                else 0.0
            ),
            "share_of_portfolio_gross_insured_aal": (
                gross_total / float(event_gross.sum())
                if event_gross.sum() > 0.0
                else 0.0
            ),
        }
    )
source_aal = pd.DataFrame(source_rows)
source_aal.to_csv(SOURCE_AAL_PATH, index=False)
append_check(
    validation_rows,
    "source_ground_up_aal_reconciles",
    math.isclose(
        float(source_aal["ground_up_aal_2022_usd"].sum()),
        calculated_aal["ground_up"],
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"source_sum={source_aal['ground_up_aal_2022_usd'].sum():.9f}; "
        f"portfolio={calculated_aal['ground_up']:.9f}"
    ),
)
append_check(
    validation_rows,
    "source_gross_insured_aal_reconciles",
    math.isclose(
        float(source_aal["gross_insured_aal_2022_usd"].sum()),
        calculated_aal["gross_insured"],
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"source_sum={source_aal['gross_insured_aal_2022_usd'].sum():.9f}; "
        f"portfolio={calculated_aal['gross_insured']:.9f}"
    ),
)
append_check(
    validation_rows,
    "source_uninsured_aal_reconciles",
    math.isclose(
        float(source_aal["uninsured_aal_2022_usd"].sum()),
        calculated_aal["uninsured"],
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"source_sum={source_aal['uninsured_aal_2022_usd'].sum():.9f}; "
        f"portfolio={calculated_aal['uninsured']:.9f}"
    ),
)

recovery_by_return_period = pml_table[
    [
        "return_period_years",
        "target_annual_exceedance_probability",
        "ground_up_aep_pml_2022_usd",
        "gross_insured_aep_pml_2022_usd",
        "uninsured_aep_pml_2022_usd",
        "insurance_recovery_share_aep",
        "uninsured_share_aep",
        "ground_up_oep_pml_2022_usd",
        "gross_insured_oep_pml_2022_usd",
        "uninsured_oep_pml_2022_usd",
        "insurance_recovery_share_oep",
        "uninsured_share_oep",
        "minimum_supporting_descending_rank",
        "tail_support_flag",
    ]
].copy()
recovery_by_return_period.to_csv(RECOVERY_BY_RETURN_PERIOD_PATH, index=False)

for curve_type, output_png, output_pdf, title in [
    (
        "AEP",
        AEP_PLOT_PNG_PATH,
        AEP_PLOT_PDF_PATH,
        "Baseline annual aggregate loss exceedance curves",
    ),
    (
        "OEP",
        OEP_PLOT_PNG_PATH,
        OEP_PLOT_PDF_PATH,
        "Baseline annual occurrence loss exceedance curves",
    ),
]:
    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    for measure, label in [
        ("ground_up", "Ground-up"),
        ("gross_insured", "Gross insured"),
        ("uninsured", "Uninsured"),
    ]:
        subset = ep_curve.loc[
            ep_curve["loss_measure"].eq(measure)
            & ep_curve["curve_type"].eq(curve_type)
        ]
        ax.plot(
            subset["return_period_years"],
            subset["loss_2022_usd"] / 1_000_000.0,
            linewidth=1.8,
            label=label,
        )
    ax.set_xscale("log")
    ax.set_xlabel("Return period (years)")
    ax.set_ylabel("Loss (million 2022 USD)")
    ax.set_title(title)
    ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_png, dpi=300)
    fig.savefig(output_pdf)
    plt.close(fig)

positive_recovery = recovery_by_return_period.loc[
    recovery_by_return_period["ground_up_aep_pml_2022_usd"] > 0.0
].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.plot(
    positive_recovery["return_period_years"],
    positive_recovery["insurance_recovery_share_aep"],
    linewidth=1.8,
    label="AEP recovery share",
)
ax.plot(
    positive_recovery["return_period_years"],
    positive_recovery["insurance_recovery_share_oep"],
    linewidth=1.8,
    label="OEP recovery share",
)
ax.set_xscale("log")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Gross insured loss / ground-up loss")
ax.set_title("Baseline insurance recovery by return period")
ax.set_ylim(0.0, 1.05)
ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
ax.legend()
fig.tight_layout()
fig.savefig(RECOVERY_PLOT_PNG_PATH, dpi=300)
fig.savefig(RECOVERY_PLOT_PDF_PATH)
plt.close(fig)

for output_path, check_id in [
    (ANNUAL_SERIES_PATH, "annual_series_file_exists"),
    (EP_CURVE_PATH, "ep_curve_file_exists"),
    (PML_TABLE_PATH, "pml_table_file_exists"),
    (RISK_METRICS_PATH, "risk_metrics_file_exists"),
    (SOURCE_AAL_PATH, "source_aal_file_exists"),
    (RECOVERY_BY_RETURN_PERIOD_PATH, "recovery_table_file_exists"),
    (AEP_PLOT_PNG_PATH, "aep_plot_png_exists"),
    (AEP_PLOT_PDF_PATH, "aep_plot_pdf_exists"),
    (OEP_PLOT_PNG_PATH, "oep_plot_png_exists"),
    (OEP_PLOT_PDF_PATH, "oep_plot_pdf_exists"),
    (RECOVERY_PLOT_PNG_PATH, "recovery_plot_png_exists"),
    (RECOVERY_PLOT_PDF_PATH, "recovery_plot_pdf_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

thin_tail_return_periods = pml_table.loc[
    pml_table["tail_support_flag"].eq("thin_tail_support"),
    "return_period_years",
].astype(int).tolist()
append_check(
    validation_rows,
    "extreme_return_period_tail_support_documented",
    len(thin_tail_return_periods) == 0,
    (
        f"thin_tail_return_periods={thin_tail_return_periods}; "
        f"threshold_rank={TAIL_SUPPORT_WARNING_RANK}"
    ),
    severity="warning",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
].copy()
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "policy_scenario": {
        "policy_scenario_id": policy_scenario_id,
        "policy_scenario_name": policy_scenario_name,
    },
    "scope": {
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
        "ground_up_loss_basis": "sampled total building repair loss",
        "gross_insured_loss_basis": (
            "sampled building repair loss after building-level policy terms"
        ),
        "uninsured_loss_definition": (
            "ground-up loss minus gross insured loss; under the baseline it is "
            "entirely deductible-absorbed loss"
        ),
        "deductible_application": "per building, per occurrence",
    },
    "annual_catalog": {
        "declared_duration_years": declared_years,
        "occurrences": expected_occurrences,
        "occupied_years": occupied_years,
        "multiple_event_years": multiple_event_years,
        "zero_event_years": zero_event_years,
        "maximum_events_in_one_year": maximum_events_in_year,
    },
    "portfolio": {
        "buildings": expected_buildings,
        "replacement_value_2022_usd": portfolio_replacement_value,
    },
    "risk_metrics": {
        "ground_up_aal_2022_usd": calculated_aal["ground_up"],
        "gross_insured_aal_2022_usd": calculated_aal["gross_insured"],
        "uninsured_aal_2022_usd": calculated_aal["uninsured"],
        "aal_insurance_recovery_share": (
            calculated_aal["gross_insured"] / calculated_aal["ground_up"]
            if calculated_aal["ground_up"] > 0.0
            else 0.0
        ),
        "aal_uninsured_share": (
            calculated_aal["uninsured"] / calculated_aal["ground_up"]
            if calculated_aal["ground_up"] > 0.0
            else 0.0
        ),
        "maximum_ground_up_aep_2022_usd": float(ground_up_aep.max()),
        "maximum_ground_up_oep_2022_usd": float(ground_up_oep.max()),
        "maximum_gross_insured_aep_2022_usd": float(gross_aep.max()),
        "maximum_gross_insured_oep_2022_usd": float(gross_oep.max()),
        "maximum_uninsured_aep_2022_usd": float(uninsured_aep.max()),
        "maximum_uninsured_oep_2022_usd": float(uninsured_oep.max()),
        "gross_insured_positive_aep_years": int(np.count_nonzero(gross_aep > 0.0)),
        "gross_insured_positive_oep_years": int(np.count_nonzero(gross_oep > 0.0)),
        "uninsured_positive_aep_years": int(np.count_nonzero(uninsured_aep > 0.0)),
        "uninsured_positive_oep_years": int(np.count_nonzero(uninsured_oep > 0.0)),
    },
    "pml_method": {
        "definition": (
            "For return period R, sort all declared annual losses in descending "
            "order and select rank ceil(N/R), where N is the declared catalog "
            "duration."
        ),
        "return_periods_years": RETURN_PERIODS_YEARS,
        "tail_support_warning_rank": TAIL_SUPPORT_WARNING_RANK,
        "thin_tail_return_periods": thin_tail_return_periods,
    },
    "source_files": {
        "cell4_event_loss_path": str(event_loss_path),
        "cell4_event_loss_sha256": actual_event_hash,
        "cell4_summary_path": str(CELL4_SUMMARY_PATH),
        "cell4_summary_sha256": sha256_file(CELL4_SUMMARY_PATH),
        "notebook6_input_handoff_path": str(CELL1_HANDOFF_PATH),
        "notebook6_input_handoff_sha256": sha256_file(CELL1_HANDOFF_PATH),
        "notebook5_annual_series_path": str(notebook5_annual_path),
        "notebook5_annual_series_sha256": notebook5_annual_hash,
        "notebook5_pml_path": str(notebook5_pml_path),
        "notebook5_pml_sha256": notebook5_pml_hash,
    },
    "outputs": {
        "annual_loss_series": {
            "path": str(ANNUAL_SERIES_PATH),
            "sha256": sha256_file(ANNUAL_SERIES_PATH),
            "rows": declared_years,
        },
        "exceedance_probability_curve": {
            "path": str(EP_CURVE_PATH),
            "sha256": sha256_file(EP_CURVE_PATH),
            "rows": int(len(ep_curve)),
        },
        "pml_table": {
            "path": str(PML_TABLE_PATH),
            "sha256": sha256_file(PML_TABLE_PATH),
            "rows": int(len(pml_table)),
        },
        "risk_metrics": {
            "path": str(RISK_METRICS_PATH),
            "sha256": sha256_file(RISK_METRICS_PATH),
            "rows": int(len(risk_metrics)),
        },
        "source_aal_summary": {
            "path": str(SOURCE_AAL_PATH),
            "sha256": sha256_file(SOURCE_AAL_PATH),
            "rows": int(len(source_aal)),
        },
        "recovery_by_return_period": {
            "path": str(RECOVERY_BY_RETURN_PERIOD_PATH),
            "sha256": sha256_file(RECOVERY_BY_RETURN_PERIOD_PATH),
            "rows": int(len(recovery_by_return_period)),
        },
        "aep_plot_png": str(AEP_PLOT_PNG_PATH),
        "aep_plot_pdf": str(AEP_PLOT_PDF_PATH),
        "oep_plot_png": str(OEP_PLOT_PNG_PATH),
        "oep_plot_pdf": str(OEP_PLOT_PDF_PATH),
        "recovery_plot_png": str(RECOVERY_PLOT_PNG_PATH),
        "recovery_plot_pdf": str(RECOVERY_PLOT_PDF_PATH),
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "validation": {
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warning_failures.to_dict(orient="records"),
    },
    "next_cell": (
        "Cell 6: define and validate a transparent baseline occurrence excess-of-loss "
        "reinsurance layer applied to occurrence-level gross insured loss."
    ),
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 5 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 5 ANNUAL INSURED RISK METRICS COMPLETE")
print("=" * 78)
print(f"Policy scenario:                 {policy_scenario_id}")
print(f"Declared catalog years:          {declared_years:,}")
print(f"Catalog occurrences:             {expected_occurrences:,}")
print(f"Occupied years:                  {occupied_years:,}")
print(f"Multiple-event years:            {multiple_event_years:,}")
print(f"Zero-event years:                {zero_event_years:,}")
print(f"Ground-up AAL:                   ${calculated_aal['ground_up']:,.2f}")
print(f"Gross insured AAL:               ${calculated_aal['gross_insured']:,.2f}")
print(f"Uninsured AAL:                   ${calculated_aal['uninsured']:,.2f}")
print(
    "AAL insurance recovery share:    "
    f"{calculated_aal['gross_insured'] / calculated_aal['ground_up']:.4%}"
)
print(f"Maximum gross insured AEP loss:  ${gross_aep.max():,.0f}")
print(f"Maximum gross insured OEP loss:  ${gross_oep.max():,.0f}")
print(f"Critical validation checks:      {int(validation['severity'].eq('critical').sum())}")
print(f"Critical failures:               {len(critical_failures)}")
print(f"Warnings requiring review:       {len(warning_failures)}")
print()
print("Annual loss series:")
print(f"  {ANNUAL_SERIES_PATH}")
print("PML table:")
print(f"  {PML_TABLE_PATH}")
print("Exceedance curve:")
print(f"  {EP_CURVE_PATH}")
print("Risk metrics:")
print(f"  {RISK_METRICS_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: define and validate the baseline occurrence reinsurance layer.")



NOTEBOOK 6 CELL 5 ANNUAL INSURED RISK METRICS COMPLETE
Policy scenario:                 baseline_full_coverage_10pct_deductible_v1
Declared catalog years:          2,000,000
Catalog occurrences:             10,630
Occupied years:                  10,593
Multiple-event years:            36
Zero-event years:                1,989,407
Ground-up AAL:                   $195,922.45
Gross insured AAL:               $122,979.56
Uninsured AAL:                   $72,942.89
AAL insurance recovery share:    62.7695%
Maximum gross insured AEP loss:  $244,325,807
Maximum gross insured OEP loss:  $244,325,807
Critical validation checks:      72
Critical failures:               0
Warnings requiring review:       1

Annual loss series:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_baseline_insured_risk\baseline_insurance_annual_loss_series.csv.gz
PML table:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_basel

In [9]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path(
    r"."
)

validation_path = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_6_insurance_terms"
    / "notebook_6_cell_5_validation.csv"
)

validation = pd.read_csv(validation_path)

warnings = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"].astype(bool)
]

display(warnings[["check_id", "severity", "passed", "detail"]])

,check_id,severity,passed,detail
72,extreme_return_period_tail_support_documented,warning,False,"thin_tail_return_periods=[200000, 500000, 1000000, 2000000]; threshold_rank=20"


In [10]:
from __future__ import annotations

import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell6_baseline_occurrence_xol_layer_v1"
DOLLAR_YEAR = 2022

LAYER_SCENARIO_ID = "baseline_occurrence_xol_500yr_to_2500yr_oep_v1"
LAYER_SCENARIO_NAME = (
    "Synthetic baseline occurrence excess-of-loss layer spanning the empirical "
    "500-year to 2,500-year gross-insured OEP PML"
)
ATTACHMENT_RETURN_PERIOD_YEARS = 500
EXHAUSTION_RETURN_PERIOD_YEARS = 2_500
REINSURER_PARTICIPATION = 1.0
SUBJECT_LOSS = "occurrence-level gross insured building repair loss"
ANNUAL_AGGREGATE_LIMIT = None
REINSTATEMENT_ASSUMPTION = (
    "No annual aggregate cap or reinstatement constraint is modeled in the "
    "baseline. Each catalog occurrence is evaluated independently."
)

NUMERIC_ATOL_USD = 1e-6
AGGREGATE_ATOL_USD = 0.01
TAIL_SUPPORT_WARNING_RANK = 20
EXPECTED_CELL5_WARNING_IDS = {"extreme_return_period_tail_support_documented"}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_cell_5_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def inspect_prior_validation(
    path: Path,
    allowed_warning_ids: set[str] | None = None,
) -> dict[str, Any]:
    table = pd.read_csv(path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")

    table["passed"] = parse_bool_series(table["passed"])
    severity = table["severity"].astype(str).str.strip().str.lower()
    failures = table.loc[severity.eq("critical") & ~table["passed"]].copy()
    warnings = table.loc[severity.eq("warning") & ~table["passed"]].copy()

    allowed = allowed_warning_ids or set()
    warning_ids = set(warnings["check_id"].astype(str).tolist())
    unexpected_warning_ids = sorted(warning_ids.difference(allowed))

    return {
        "critical_failures": failures,
        "warnings": warnings,
        "warning_ids": sorted(warning_ids),
        "unexpected_warning_ids": unexpected_warning_ids,
    }


def apply_occurrence_xol(
    gross_insured_loss: np.ndarray | pd.Series | float,
    attachment: float,
    limit: float,
    participation: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    gross = np.asarray(gross_insured_loss, dtype=np.float64)
    layer_loss_before_participation = np.minimum(
        np.maximum(gross - attachment, 0.0),
        limit,
    )
    ceded = participation * layer_loss_before_participation
    retained = gross - ceded
    return layer_loss_before_participation, ceded, retained


def select_actual_control_cases(
    events: pd.DataFrame,
    attachment: float,
    exhaustion: float,
) -> pd.DataFrame:
    gross_column = "gross_insured_loss_2022_usd"
    candidates: list[tuple[str, int]] = []

    def add_case(label: str, subset: pd.DataFrame, mode: str) -> None:
        if subset.empty:
            return
        if mode == "min":
            index = int(subset[gross_column].idxmin())
        elif mode == "max":
            index = int(subset[gross_column].idxmax())
        elif mode == "closest_attachment":
            index = int((subset[gross_column] - attachment).abs().idxmin())
        elif mode == "closest_exhaustion":
            index = int((subset[gross_column] - exhaustion).abs().idxmin())
        elif mode == "closest_midpoint":
            midpoint = 0.5 * (attachment + exhaustion)
            index = int((subset[gross_column] - midpoint).abs().idxmin())
        else:
            raise ValueError(f"Unknown selection mode: {mode}")
        candidates.append((label, index))

    add_case("minimum_gross_loss", events, "min")
    add_case(
        "largest_below_or_at_attachment",
        events.loc[events[gross_column] <= attachment],
        "max",
    )
    add_case(
        "smallest_above_attachment",
        events.loc[events[gross_column] > attachment],
        "min",
    )
    add_case(
        "closest_to_attachment",
        events,
        "closest_attachment",
    )
    add_case(
        "closest_to_layer_midpoint",
        events,
        "closest_midpoint",
    )
    add_case(
        "largest_below_exhaustion",
        events.loc[events[gross_column] < exhaustion],
        "max",
    )
    add_case(
        "smallest_at_or_above_exhaustion",
        events.loc[events[gross_column] >= exhaustion],
        "min",
    )
    add_case(
        "closest_to_exhaustion",
        events,
        "closest_exhaustion",
    )
    add_case("maximum_gross_loss", events, "max")

    for source_type, group in events.groupby("source_type", sort=True):
        add_case(f"maximum_{source_type}_gross_loss", group, "max")

    records: list[pd.Series] = []
    seen_occurrences: set[str] = set()
    for label, index in candidates:
        row = events.loc[index].copy()
        occurrence_id = str(row["occurrence_id"])
        if occurrence_id in seen_occurrences:
            continue
        seen_occurrences.add(occurrence_id)
        row["control_case"] = label
        records.append(row)

    if not records:
        raise RuntimeError("No actual control cases could be selected.")

    controlled = pd.DataFrame(records).reset_index(drop=True)
    first_columns = [
        "control_case",
        "catalog_year",
        "occurrence_id",
        "source_type",
        "gross_insured_loss_2022_usd",
    ]
    remaining = [column for column in controlled.columns if column not in first_columns]
    return controlled[first_columns + remaining]


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_reinsurance_parameters"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

CELL4_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_4_validation.csv"
CELL4_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_4_summary.json"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_5_validation.csv"
CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_5_summary.json"

LAYER_TERMS_PATH = OUTPUT_DIR / "baseline_occurrence_xol_layer_terms.csv"
CONTROLLED_CASES_PATH = OUTPUT_DIR / "controlled_occurrence_xol_validation_cases.csv"
RESPONSE_GRID_PATH = OUTPUT_DIR / "baseline_occurrence_xol_response_grid.csv"
SOURCE_DIAGNOSTICS_PATH = OUTPUT_DIR / "baseline_occurrence_xol_source_diagnostics.csv"
ASSUMPTION_REGISTRY_PATH = OUTPUT_DIR / "baseline_occurrence_xol_assumptions.csv"
FORMULA_SPECIFICATION_PATH = (
    METADATA_DIR / "notebook_6_cell_6_reinsurance_formula_specification.json"
)
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_6_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_6_summary.json"

validation_rows: list[dict[str, Any]] = []
required_inputs = [
    CELL4_VALIDATION_PATH,
    CELL4_SUMMARY_PATH,
    CELL5_VALIDATION_PATH,
    CELL5_SUMMARY_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")

cell4_validation = inspect_prior_validation(CELL4_VALIDATION_PATH)
cell5_validation = inspect_prior_validation(
    CELL5_VALIDATION_PATH,
    allowed_warning_ids=EXPECTED_CELL5_WARNING_IDS,
)
append_check(
    validation_rows,
    "cell4_critical_validation_passed",
    cell4_validation["critical_failures"].empty,
    f"critical_failures={len(cell4_validation['critical_failures'])}",
)
append_check(
    validation_rows,
    "cell4_has_no_unresolved_warnings",
    len(cell4_validation["warning_ids"]) == 0,
    f"warning_ids={cell4_validation['warning_ids']}",
)
append_check(
    validation_rows,
    "cell5_critical_validation_passed",
    cell5_validation["critical_failures"].empty,
    f"critical_failures={len(cell5_validation['critical_failures'])}",
)
append_check(
    validation_rows,
    "cell5_warnings_are_expected",
    len(cell5_validation["unexpected_warning_ids"]) == 0,
    (
        f"warning_ids={cell5_validation['warning_ids']}; "
        f"unexpected={cell5_validation['unexpected_warning_ids']}"
    ),
)

cell4_summary = load_json(CELL4_SUMMARY_PATH)
cell5_summary = load_json(CELL5_SUMMARY_PATH)
append_check(
    validation_rows,
    "cell4_summary_declares_success",
    bool(cell4_summary.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell4_summary.get('all_critical_checks_passed')}",
)
append_check(
    validation_rows,
    "cell5_summary_declares_success",
    bool(cell5_summary.get("all_critical_checks_passed", False)),
    f"all_critical_checks_passed={cell5_summary.get('all_critical_checks_passed')}",
)

cell4_policy = cell4_summary["policy_scenario"]
cell5_policy = cell5_summary["policy_scenario"]
policy_scenario_id = str(cell4_policy["policy_scenario_id"])
policy_scenario_name = str(cell4_policy["policy_scenario_name"])
append_check(
    validation_rows,
    "cell4_and_cell5_policy_scenarios_match",
    policy_scenario_id == str(cell5_policy["policy_scenario_id"]),
    (
        f"cell4={policy_scenario_id}; "
        f"cell5={cell5_policy['policy_scenario_id']}"
    ),
)

cell4_catalog = cell4_summary["annual_catalog"]
cell5_catalog = cell5_summary["annual_catalog"]
declared_years = int(cell5_catalog["declared_duration_years"])
expected_occurrences = int(cell5_catalog["occurrences"])
portfolio_replacement_value = float(
    cell5_summary["portfolio"]["replacement_value_2022_usd"]
)
append_check(
    validation_rows,
    "cell4_and_cell5_catalog_occurrences_match",
    int(cell4_catalog["occurrences"]) == expected_occurrences,
    (
        f"cell4={cell4_catalog['occurrences']}; "
        f"cell5={expected_occurrences}"
    ),
)
append_check(
    validation_rows,
    "declared_catalog_duration_positive",
    declared_years > 0,
    f"years={declared_years}",
)
append_check(
    validation_rows,
    "layer_return_periods_supported_by_catalog",
    (
        ATTACHMENT_RETURN_PERIOD_YEARS < EXHAUSTION_RETURN_PERIOD_YEARS
        <= declared_years
    ),
    (
        f"attachment_rp={ATTACHMENT_RETURN_PERIOD_YEARS}; "
        f"exhaustion_rp={EXHAUSTION_RETURN_PERIOD_YEARS}; "
        f"catalog_years={declared_years}"
    ),
)

cell4_event_record = cell4_summary["outputs"]["event_insured_loss"]
event_loss_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(cell4_event_record["path"]),
)
actual_event_hash = sha256_file(event_loss_path)
append_check(
    validation_rows,
    "cell4_event_loss_hash_matches",
    actual_event_hash == str(cell4_event_record["sha256"]),
    f"expected={cell4_event_record['sha256']}; actual={actual_event_hash}",
)

cell5_pml_record = cell5_summary["outputs"]["pml_table"]
pml_path = resolve_recorded_path(PROJECT_ROOT, str(cell5_pml_record["path"]))
actual_pml_hash = sha256_file(pml_path)
append_check(
    validation_rows,
    "cell5_pml_hash_matches",
    actual_pml_hash == str(cell5_pml_record["sha256"]),
    f"expected={cell5_pml_record['sha256']}; actual={actual_pml_hash}",
)

events = pd.read_csv(event_loss_path, compression="gzip")
required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "gross_insured_loss_2022_usd",
}
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "event_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise KeyError(f"Cell 4 event-loss file is missing columns: {missing_event_columns}")

for column in ["catalog_year", "gross_insured_loss_2022_usd"]:
    events[column] = pd.to_numeric(events[column], errors="raise")
events["catalog_year"] = events["catalog_year"].astype(np.int64)
events["occurrence_id"] = events["occurrence_id"].astype(str).str.strip()
events["source_type"] = events["source_type"].astype(str).str.strip()

gross = events["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
append_check(
    validation_rows,
    "event_summary_occurrence_count_matches",
    len(events) == expected_occurrences and events["occurrence_id"].is_unique,
    (
        f"rows={len(events)}; expected={expected_occurrences}; "
        f"unique_occurrences={events['occurrence_id'].nunique()}"
    ),
)
append_check(
    validation_rows,
    "event_gross_losses_finite_and_nonnegative",
    np.isfinite(gross).all() and np.all(gross >= 0.0),
    f"minimum={gross.min():.6f}; maximum={gross.max():.6f}",
)
append_check(
    validation_rows,
    "event_gross_losses_not_above_portfolio_replacement_value",
    np.all(gross <= portfolio_replacement_value + AGGREGATE_ATOL_USD),
    (
        f"maximum_gross={gross.max():.6f}; "
        f"portfolio_replacement_value={portfolio_replacement_value:.6f}"
    ),
)
append_check(
    validation_rows,
    "event_catalog_years_within_declared_range",
    events["catalog_year"].between(0, declared_years - 1).all(),
    (
        f"minimum_year={events['catalog_year'].min()}; "
        f"maximum_year={events['catalog_year'].max()}"
    ),
)
append_check(
    validation_rows,
    "event_source_types_nonempty",
    events["source_type"].ne("").all(),
    f"source_types={sorted(events['source_type'].unique().tolist())}",
)
cell4_gross_total = float(
    cell4_summary["production"]["gross_insured_loss_total_2022_usd"]
)
append_check(
    validation_rows,
    "event_gross_total_reconciles_with_cell4",
    abs(float(gross.sum()) - cell4_gross_total) <= AGGREGATE_ATOL_USD,
    f"events={gross.sum():.6f}; cell4={cell4_gross_total:.6f}",
)

pml = pd.read_csv(pml_path)
required_pml_columns = {
    "return_period_years",
    "gross_insured_oep_pml_2022_usd",
    "gross_insured_oep_supporting_descending_rank",
    "gross_insured_oep_empirical_aep",
}
missing_pml_columns = sorted(required_pml_columns.difference(pml.columns))
append_check(
    validation_rows,
    "pml_schema_complete",
    not missing_pml_columns,
    f"missing={missing_pml_columns}",
)
if missing_pml_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise KeyError(f"Cell 5 PML table is missing columns: {missing_pml_columns}")

for column in required_pml_columns:
    pml[column] = pd.to_numeric(pml[column], errors="raise")
pml["return_period_years"] = pml["return_period_years"].astype(np.int64)
append_check(
    validation_rows,
    "pml_return_periods_unique",
    pml["return_period_years"].is_unique,
    f"rows={len(pml)}; unique={pml['return_period_years'].nunique()}",
)
append_check(
    validation_rows,
    "required_layer_return_periods_present",
    {
        ATTACHMENT_RETURN_PERIOD_YEARS,
        EXHAUSTION_RETURN_PERIOD_YEARS,
    }.issubset(set(pml["return_period_years"].tolist())),
    f"available={sorted(pml['return_period_years'].tolist())}",
)

attachment_row = pml.loc[
    pml["return_period_years"].eq(ATTACHMENT_RETURN_PERIOD_YEARS)
].iloc[0]
exhaustion_row = pml.loc[
    pml["return_period_years"].eq(EXHAUSTION_RETURN_PERIOD_YEARS)
].iloc[0]
attachment = float(attachment_row["gross_insured_oep_pml_2022_usd"])
exhaustion = float(exhaustion_row["gross_insured_oep_pml_2022_usd"])
limit = exhaustion - attachment
attachment_rank = int(
    attachment_row["gross_insured_oep_supporting_descending_rank"]
)
exhaustion_rank = int(
    exhaustion_row["gross_insured_oep_supporting_descending_rank"]
)

append_check(
    validation_rows,
    "attachment_is_finite_and_positive",
    math.isfinite(attachment) and attachment > 0.0,
    f"attachment={attachment:.6f}",
)
append_check(
    validation_rows,
    "exhaustion_is_finite_and_above_attachment",
    math.isfinite(exhaustion) and exhaustion > attachment,
    f"attachment={attachment:.6f}; exhaustion={exhaustion:.6f}",
)
append_check(
    validation_rows,
    "layer_limit_is_positive",
    math.isfinite(limit) and limit > 0.0,
    f"limit={limit:.6f}",
)
append_check(
    validation_rows,
    "layer_attachment_and_exhaustion_have_adequate_tail_support",
    attachment_rank >= TAIL_SUPPORT_WARNING_RANK
    and exhaustion_rank >= TAIL_SUPPORT_WARNING_RANK,
    (
        f"attachment_rank={attachment_rank}; "
        f"exhaustion_rank={exhaustion_rank}; "
        f"threshold={TAIL_SUPPORT_WARNING_RANK}"
    ),
)
append_check(
    validation_rows,
    "reinsurer_participation_valid",
    0.0 < REINSURER_PARTICIPATION <= 1.0,
    f"participation={REINSURER_PARTICIPATION}",
)
append_check(
    validation_rows,
    "layer_exhaustion_below_observed_maximum_gross_loss",
    exhaustion < float(gross.max()),
    f"exhaustion={exhaustion:.6f}; maximum_gross={gross.max():.6f}",
)

layer_before, ceded, retained = apply_occurrence_xol(
    gross,
    attachment,
    limit,
    REINSURER_PARTICIPATION,
)

actual_controls = select_actual_control_cases(events, attachment, exhaustion)
actual_layer_before, actual_ceded, actual_retained = apply_occurrence_xol(
    actual_controls["gross_insured_loss_2022_usd"].to_numpy(dtype=float),
    attachment,
    limit,
    REINSURER_PARTICIPATION,
)
actual_controls["case_type"] = "actual_catalog_occurrence"
actual_controls["attachment_2022_usd"] = attachment
actual_controls["limit_2022_usd"] = limit
actual_controls["exhaustion_2022_usd"] = exhaustion
actual_controls["reinsurer_participation"] = REINSURER_PARTICIPATION
actual_controls["layer_loss_before_participation_2022_usd"] = actual_layer_before
actual_controls["ceded_loss_2022_usd"] = actual_ceded
actual_controls["net_retained_loss_2022_usd"] = actual_retained

synthetic_values = np.array(
    sorted(
        set(
            [
                0.0,
                0.25 * attachment,
                0.50 * attachment,
                max(0.0, attachment - 1.0),
                attachment,
                attachment + 1.0,
                0.5 * (attachment + exhaustion),
                max(0.0, exhaustion - 1.0),
                exhaustion,
                exhaustion + 1.0,
                float(gross.max()),
                1.25 * float(gross.max()),
            ]
        )
    ),
    dtype=np.float64,
)
synthetic_layer_before, synthetic_ceded, synthetic_retained = apply_occurrence_xol(
    synthetic_values,
    attachment,
    limit,
    REINSURER_PARTICIPATION,
)
synthetic_controls = pd.DataFrame(
    {
        "control_case": [f"synthetic_case_{index:02d}" for index in range(len(synthetic_values))],
        "catalog_year": np.nan,
        "occurrence_id": "synthetic",
        "source_type": "synthetic_formula_test",
        "gross_insured_loss_2022_usd": synthetic_values,
        "case_type": "synthetic_formula_stress_test",
        "attachment_2022_usd": attachment,
        "limit_2022_usd": limit,
        "exhaustion_2022_usd": exhaustion,
        "reinsurer_participation": REINSURER_PARTICIPATION,
        "layer_loss_before_participation_2022_usd": synthetic_layer_before,
        "ceded_loss_2022_usd": synthetic_ceded,
        "net_retained_loss_2022_usd": synthetic_retained,
    }
)

controlled_cases = pd.concat(
    [actual_controls, synthetic_controls],
    ignore_index=True,
    sort=False,
)
controlled_cases["conservation_error_2022_usd"] = (
    controlled_cases["gross_insured_loss_2022_usd"]
    - controlled_cases["ceded_loss_2022_usd"]
    - controlled_cases["net_retained_loss_2022_usd"]
)
controlled_cases.to_csv(CONTROLLED_CASES_PATH, index=False)

maximum_grid_loss = max(float(gross.max()), exhaustion + 0.25 * limit)
grid_values = np.unique(
    np.concatenate(
        [
            np.linspace(0.0, maximum_grid_loss, 2_001, dtype=np.float64),
            np.array(
                [
                    0.0,
                    attachment,
                    exhaustion,
                    maximum_grid_loss,
                ],
                dtype=np.float64,
            ),
        ]
    )
)
grid_layer_before, grid_ceded, grid_retained = apply_occurrence_xol(
    grid_values,
    attachment,
    limit,
    REINSURER_PARTICIPATION,
)
response_grid = pd.DataFrame(
    {
        "gross_insured_loss_2022_usd": grid_values,
        "layer_loss_before_participation_2022_usd": grid_layer_before,
        "ceded_loss_2022_usd": grid_ceded,
        "net_retained_loss_2022_usd": grid_retained,
    }
)
response_grid["layer_regime"] = np.select(
    [
        response_grid["gross_insured_loss_2022_usd"] <= attachment,
        response_grid["gross_insured_loss_2022_usd"] < exhaustion,
    ],
    ["below_or_at_attachment", "within_layer"],
    default="at_or_above_exhaustion",
)
response_grid.to_csv(RESPONSE_GRID_PATH, index=False)

below_attachment = gross <= attachment + NUMERIC_ATOL_USD
within_layer = (gross > attachment + NUMERIC_ATOL_USD) & (
    gross < exhaustion - NUMERIC_ATOL_USD
)
at_or_above_exhaustion = gross >= exhaustion - NUMERIC_ATOL_USD

source_rows: list[dict[str, Any]] = []
for source_type, indices in events.groupby("source_type", sort=True).groups.items():
    positions = np.asarray(list(indices), dtype=np.int64)
    source_gross = gross[positions]
    source_ceded = ceded[positions]
    source_retained = retained[positions]
    source_rows.append(
        {
            "source_type": source_type,
            "occurrences": int(len(positions)),
            "occurrences_above_attachment": int(
                np.count_nonzero(source_gross > attachment)
            ),
            "occurrences_at_or_above_exhaustion": int(
                np.count_nonzero(source_gross >= exhaustion)
            ),
            "gross_insured_loss_total_2022_usd": float(source_gross.sum()),
            "ceded_loss_total_2022_usd": float(source_ceded.sum()),
            "net_retained_loss_total_2022_usd": float(source_retained.sum()),
            "gross_insured_aal_2022_usd": float(source_gross.sum() / declared_years),
            "ceded_aal_2022_usd": float(source_ceded.sum() / declared_years),
            "net_retained_aal_2022_usd": float(
                source_retained.sum() / declared_years
            ),
            "ceded_share_of_source_gross": (
                float(source_ceded.sum() / source_gross.sum())
                if source_gross.sum() > 0.0
                else 0.0
            ),
        }
    )
source_diagnostics = pd.DataFrame(source_rows)
source_diagnostics.to_csv(SOURCE_DIAGNOSTICS_PATH, index=False)

layer_terms = pd.DataFrame(
    [
        {
            "layer_scenario_id": LAYER_SCENARIO_ID,
            "layer_scenario_name": LAYER_SCENARIO_NAME,
            "underlying_policy_scenario_id": policy_scenario_id,
            "currency": "USD",
            "dollar_year": DOLLAR_YEAR,
            "contract_type": "occurrence excess of loss",
            "subject_loss": SUBJECT_LOSS,
            "attachment_selection_basis": "gross insured OEP PML",
            "attachment_return_period_years": ATTACHMENT_RETURN_PERIOD_YEARS,
            "attachment_2022_usd": attachment,
            "exhaustion_selection_basis": "gross insured OEP PML",
            "exhaustion_return_period_years": EXHAUSTION_RETURN_PERIOD_YEARS,
            "exhaustion_2022_usd": exhaustion,
            "limit_2022_usd": limit,
            "reinsurer_participation": REINSURER_PARTICIPATION,
            "annual_aggregate_limit_2022_usd": ANNUAL_AGGREGATE_LIMIT,
            "reinstatement_assumption": REINSTATEMENT_ASSUMPTION,
            "attachment_supporting_descending_rank": attachment_rank,
            "exhaustion_supporting_descending_rank": exhaustion_rank,
        }
    ]
)
layer_terms.to_csv(LAYER_TERMS_PATH, index=False)

assumption_rows = [
    {
        "assumption_id": "layer_selection",
        "value": (
            f"attachment at {ATTACHMENT_RETURN_PERIOD_YEARS}-year gross-insured "
            f"OEP PML and exhaustion at {EXHAUSTION_RETURN_PERIOD_YEARS}-year "
            "gross-insured OEP PML"
        ),
        "purpose": (
            "Create a transparent portfolio-specific synthetic layer with adequate "
            "empirical tail support."
        ),
        "observed_data": False,
    },
    {
        "assumption_id": "reinsurer_participation",
        "value": f"{REINSURER_PARTICIPATION:.6f}",
        "purpose": "Model a fully placed baseline layer before placement sensitivity.",
        "observed_data": False,
    },
    {
        "assumption_id": "occurrence_application",
        "value": "per catalog occurrence",
        "purpose": "Apply the layer to occurrence-level gross insured loss.",
        "observed_data": False,
    },
    {
        "assumption_id": "annual_aggregate_and_reinstatements",
        "value": REINSTATEMENT_ASSUMPTION,
        "purpose": "Keep the baseline contract transparent and isolate occurrence XoL mechanics.",
        "observed_data": False,
    },
    {
        "assumption_id": "premium_and_expenses",
        "value": "not modeled",
        "purpose": "Report loss transfer only; pricing is outside this baseline cell.",
        "observed_data": False,
    },
]
pd.DataFrame(assumption_rows).to_csv(ASSUMPTION_REGISTRY_PATH, index=False)

synthetic_gross = synthetic_controls["gross_insured_loss_2022_usd"].to_numpy(
    dtype=float
)
synthetic_ceded_array = synthetic_controls["ceded_loss_2022_usd"].to_numpy(
    dtype=float
)
synthetic_retained_array = synthetic_controls[
    "net_retained_loss_2022_usd"
].to_numpy(dtype=float)
expected_layer_before = np.minimum(
    np.maximum(synthetic_gross - attachment, 0.0),
    limit,
)
expected_ceded = REINSURER_PARTICIPATION * expected_layer_before
expected_retained = synthetic_gross - expected_ceded

append_check(
    validation_rows,
    "synthetic_ceded_equation_exact",
    np.max(np.abs(synthetic_ceded_array - expected_ceded)) <= NUMERIC_ATOL_USD,
    f"maximum_error={np.max(np.abs(synthetic_ceded_array - expected_ceded)):.3e}",
)
append_check(
    validation_rows,
    "synthetic_retained_equation_exact",
    np.max(np.abs(synthetic_retained_array - expected_retained))
    <= NUMERIC_ATOL_USD,
    f"maximum_error={np.max(np.abs(synthetic_retained_array - expected_retained)):.3e}",
)
append_check(
    validation_rows,
    "synthetic_loss_conservation_exact",
    np.max(
        np.abs(synthetic_gross - synthetic_ceded_array - synthetic_retained_array)
    )
    <= NUMERIC_ATOL_USD,
    (
        "maximum_error="
        f"{np.max(np.abs(synthetic_gross - synthetic_ceded_array - synthetic_retained_array)):.3e}"
    ),
)
append_check(
    validation_rows,
    "synthetic_ceded_bounds_valid",
    np.all(synthetic_ceded_array >= -NUMERIC_ATOL_USD)
    and np.all(
        synthetic_ceded_array
        <= REINSURER_PARTICIPATION * limit + NUMERIC_ATOL_USD
    ),
    (
        f"minimum={synthetic_ceded_array.min():.6f}; "
        f"maximum={synthetic_ceded_array.max():.6f}"
    ),
)
append_check(
    validation_rows,
    "synthetic_retained_bounds_valid",
    np.all(synthetic_retained_array >= -NUMERIC_ATOL_USD)
    and np.all(synthetic_retained_array <= synthetic_gross + NUMERIC_ATOL_USD),
    (
        f"minimum={synthetic_retained_array.min():.6f}; "
        f"maximum={synthetic_retained_array.max():.6f}"
    ),
)

attachment_test = apply_occurrence_xol(
    np.array([attachment]), attachment, limit, REINSURER_PARTICIPATION
)
exhaustion_test = apply_occurrence_xol(
    np.array([exhaustion]), attachment, limit, REINSURER_PARTICIPATION
)
append_check(
    validation_rows,
    "ceded_loss_is_zero_at_attachment",
    abs(float(attachment_test[1][0])) <= NUMERIC_ATOL_USD,
    f"ceded={float(attachment_test[1][0]):.9f}",
)
append_check(
    validation_rows,
    "ceded_loss_equals_participating_limit_at_exhaustion",
    abs(float(exhaustion_test[1][0]) - REINSURER_PARTICIPATION * limit)
    <= NUMERIC_ATOL_USD,
    (
        f"ceded={float(exhaustion_test[1][0]):.9f}; "
        f"expected={REINSURER_PARTICIPATION * limit:.9f}"
    ),
)
append_check(
    validation_rows,
    "retained_loss_at_attachment_equals_attachment",
    abs(float(attachment_test[2][0]) - attachment) <= NUMERIC_ATOL_USD,
    f"retained={float(attachment_test[2][0]):.9f}; attachment={attachment:.9f}",
)
append_check(
    validation_rows,
    "retained_loss_at_exhaustion_matches_formula",
    abs(
        float(exhaustion_test[2][0])
        - (exhaustion - REINSURER_PARTICIPATION * limit)
    )
    <= NUMERIC_ATOL_USD,
    (
        f"retained={float(exhaustion_test[2][0]):.9f}; "
        f"expected={exhaustion - REINSURER_PARTICIPATION * limit:.9f}"
    ),
)

append_check(
    validation_rows,
    "response_grid_ceded_nondecreasing",
    np.all(np.diff(grid_ceded) >= -NUMERIC_ATOL_USD),
    f"minimum_difference={np.diff(grid_ceded).min():.6e}",
)
append_check(
    validation_rows,
    "response_grid_retained_nondecreasing",
    np.all(np.diff(grid_retained) >= -NUMERIC_ATOL_USD),
    f"minimum_difference={np.diff(grid_retained).min():.6e}",
)
append_check(
    validation_rows,
    "response_grid_loss_conservation",
    np.max(np.abs(grid_values - grid_ceded - grid_retained))
    <= NUMERIC_ATOL_USD,
    (
        "maximum_error="
        f"{np.max(np.abs(grid_values - grid_ceded - grid_retained)):.3e}"
    ),
)
append_check(
    validation_rows,
    "response_grid_no_cession_below_attachment",
    np.max(
        np.abs(
            grid_ceded[
                grid_values <= attachment + NUMERIC_ATOL_USD
            ]
        )
    )
    <= NUMERIC_ATOL_USD,
    "checked all grid points at or below attachment",
)
append_check(
    validation_rows,
    "response_grid_full_limit_above_exhaustion",
    np.max(
        np.abs(
            grid_ceded[grid_values >= exhaustion - NUMERIC_ATOL_USD]
            - REINSURER_PARTICIPATION * limit
        )
    )
    <= NUMERIC_ATOL_USD,
    "checked all grid points at or above exhaustion",
)

actual_conservation_error = np.max(np.abs(gross - ceded - retained))
append_check(
    validation_rows,
    "actual_occurrence_loss_conservation",
    actual_conservation_error <= NUMERIC_ATOL_USD,
    f"maximum_error={actual_conservation_error:.3e}",
)
append_check(
    validation_rows,
    "actual_occurrence_ceded_bounds_valid",
    np.all(ceded >= -NUMERIC_ATOL_USD)
    and np.all(ceded <= REINSURER_PARTICIPATION * limit + NUMERIC_ATOL_USD),
    f"minimum={ceded.min():.6f}; maximum={ceded.max():.6f}",
)
append_check(
    validation_rows,
    "actual_occurrence_retained_bounds_valid",
    np.all(retained >= -NUMERIC_ATOL_USD)
    and np.all(retained <= gross + NUMERIC_ATOL_USD),
    f"minimum={retained.min():.6f}; maximum={retained.max():.6f}",
)
append_check(
    validation_rows,
    "actual_occurrences_below_attachment_have_zero_cession",
    np.max(np.abs(ceded[below_attachment])) <= NUMERIC_ATOL_USD,
    f"occurrences={int(np.count_nonzero(below_attachment))}",
)
append_check(
    validation_rows,
    "actual_occurrences_at_or_above_exhaustion_receive_full_layer",
    np.max(
        np.abs(
            ceded[at_or_above_exhaustion]
            - REINSURER_PARTICIPATION * limit
        )
    )
    <= NUMERIC_ATOL_USD,
    f"occurrences={int(np.count_nonzero(at_or_above_exhaustion))}",
)
append_check(
    validation_rows,
    "actual_catalog_represents_all_layer_regimes",
    np.count_nonzero(below_attachment) > 0
    and np.count_nonzero(within_layer) > 0
    and np.count_nonzero(at_or_above_exhaustion) > 0,
    (
        f"below_or_at_attachment={int(np.count_nonzero(below_attachment))}; "
        f"within_layer={int(np.count_nonzero(within_layer))}; "
        f"at_or_above_exhaustion={int(np.count_nonzero(at_or_above_exhaustion))}"
    ),
)

gross_total = float(gross.sum())
ceded_total = float(ceded.sum())
retained_total = float(retained.sum())
gross_aal = gross_total / declared_years
ceded_aal = ceded_total / declared_years
retained_aal = retained_total / declared_years
append_check(
    validation_rows,
    "aggregate_gross_equals_ceded_plus_retained",
    abs(gross_total - ceded_total - retained_total) <= AGGREGATE_ATOL_USD,
    (
        f"gross={gross_total:.6f}; ceded={ceded_total:.6f}; "
        f"retained={retained_total:.6f}"
    ),
)
append_check(
    validation_rows,
    "aal_gross_equals_ceded_plus_retained",
    abs(gross_aal - ceded_aal - retained_aal) <= NUMERIC_ATOL_USD,
    (
        f"gross_aal={gross_aal:.9f}; ceded_aal={ceded_aal:.9f}; "
        f"retained_aal={retained_aal:.9f}"
    ),
)
append_check(
    validation_rows,
    "source_diagnostics_gross_reconcile",
    abs(
        float(source_diagnostics["gross_insured_loss_total_2022_usd"].sum())
        - gross_total
    )
    <= AGGREGATE_ATOL_USD,
    "source totals checked",
)
append_check(
    validation_rows,
    "source_diagnostics_ceded_reconcile",
    abs(float(source_diagnostics["ceded_loss_total_2022_usd"].sum()) - ceded_total)
    <= AGGREGATE_ATOL_USD,
    "source totals checked",
)
append_check(
    validation_rows,
    "source_diagnostics_retained_reconcile",
    abs(
        float(source_diagnostics["net_retained_loss_total_2022_usd"].sum())
        - retained_total
    )
    <= AGGREGATE_ATOL_USD,
    "source totals checked",
)

for output_path, check_id in [
    (LAYER_TERMS_PATH, "layer_terms_file_exists"),
    (CONTROLLED_CASES_PATH, "controlled_cases_file_exists"),
    (RESPONSE_GRID_PATH, "response_grid_file_exists"),
    (SOURCE_DIAGNOSTICS_PATH, "source_diagnostics_file_exists"),
    (ASSUMPTION_REGISTRY_PATH, "assumption_registry_file_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

formula_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "layer_scenario_id": LAYER_SCENARIO_ID,
    "underlying_policy_scenario_id": policy_scenario_id,
    "contract_type": "occurrence excess of loss",
    "currency": "USD",
    "dollar_year": DOLLAR_YEAR,
    "subject_loss": SUBJECT_LOSS,
    "layer_selection": {
        "attachment_basis": "gross insured OEP PML",
        "attachment_return_period_years": ATTACHMENT_RETURN_PERIOD_YEARS,
        "attachment_2022_usd": attachment,
        "attachment_supporting_descending_rank": attachment_rank,
        "exhaustion_basis": "gross insured OEP PML",
        "exhaustion_return_period_years": EXHAUSTION_RETURN_PERIOD_YEARS,
        "exhaustion_2022_usd": exhaustion,
        "exhaustion_supporting_descending_rank": exhaustion_rank,
        "limit_2022_usd": limit,
        "reinsurer_participation": REINSURER_PARTICIPATION,
    },
    "equations": {
        "layer_loss_before_participation": (
            "min(max(gross_insured_occurrence_loss - attachment, 0), limit)"
        ),
        "ceded_loss": (
            "reinsurer_participation * layer_loss_before_participation"
        ),
        "net_retained_loss": "gross_insured_occurrence_loss - ceded_loss",
        "conservation": "gross_insured_loss = ceded_loss + net_retained_loss",
    },
    "annual_aggregate_limit": ANNUAL_AGGREGATE_LIMIT,
    "reinstatement_assumption": REINSTATEMENT_ASSUMPTION,
    "excluded_from_baseline": [
        "reinsurance premium",
        "brokerage and commissions",
        "reinstatement premium",
        "annual aggregate deductible",
        "annual aggregate limit",
        "hours-clause event grouping",
        "claims development and settlement timing",
    ],
}
write_json(FORMULA_SPECIFICATION_PATH, formula_specification)
append_check(
    validation_rows,
    "formula_specification_file_exists",
    FORMULA_SPECIFICATION_PATH.exists()
    and FORMULA_SPECIFICATION_PATH.stat().st_size > 0,
    str(FORMULA_SPECIFICATION_PATH),
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
].copy()
validation.to_csv(VALIDATION_PATH, index=False)

triggering_occurrences = int(np.count_nonzero(gross > attachment))
exhausting_occurrences = int(np.count_nonzero(gross >= exhaustion))
summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "underlying_policy_scenario": {
        "policy_scenario_id": policy_scenario_id,
        "policy_scenario_name": policy_scenario_name,
    },
    "reinsurance_scenario": {
        "layer_scenario_id": LAYER_SCENARIO_ID,
        "layer_scenario_name": LAYER_SCENARIO_NAME,
        "contract_type": "occurrence excess of loss",
        "subject_loss": SUBJECT_LOSS,
        "attachment_return_period_years": ATTACHMENT_RETURN_PERIOD_YEARS,
        "attachment_2022_usd": attachment,
        "limit_2022_usd": limit,
        "exhaustion_return_period_years": EXHAUSTION_RETURN_PERIOD_YEARS,
        "exhaustion_2022_usd": exhaustion,
        "reinsurer_participation": REINSURER_PARTICIPATION,
        "annual_aggregate_limit_2022_usd": ANNUAL_AGGREGATE_LIMIT,
        "reinstatement_assumption": REINSTATEMENT_ASSUMPTION,
    },
    "annual_catalog": {
        "declared_duration_years": declared_years,
        "occurrences": expected_occurrences,
    },
    "portfolio": {
        "replacement_value_2022_usd": portfolio_replacement_value,
    },
    "diagnostic_application_to_catalog_occurrences": {
        "gross_insured_loss_total_2022_usd": gross_total,
        "ceded_loss_total_2022_usd": ceded_total,
        "net_retained_loss_total_2022_usd": retained_total,
        "gross_insured_aal_2022_usd": gross_aal,
        "ceded_preliminary_aal_2022_usd": ceded_aal,
        "net_retained_preliminary_aal_2022_usd": retained_aal,
        "ceded_share_of_gross_insured_loss": (
            ceded_total / gross_total if gross_total > 0.0 else 0.0
        ),
        "triggering_occurrences": triggering_occurrences,
        "exhausting_occurrences": exhausting_occurrences,
        "attachment_trigger_annual_frequency": (
            triggering_occurrences / declared_years
        ),
        "exhaustion_annual_frequency": exhausting_occurrences / declared_years,
        "maximum_gross_insured_occurrence_loss_2022_usd": float(gross.max()),
        "maximum_ceded_occurrence_loss_2022_usd": float(ceded.max()),
        "maximum_net_retained_occurrence_loss_2022_usd": float(retained.max()),
    },
    "source_files": {
        "cell4_event_loss_path": str(event_loss_path),
        "cell4_event_loss_sha256": actual_event_hash,
        "cell5_pml_path": str(pml_path),
        "cell5_pml_sha256": actual_pml_hash,
        "cell4_summary_path": str(CELL4_SUMMARY_PATH),
        "cell4_summary_sha256": sha256_file(CELL4_SUMMARY_PATH),
        "cell5_summary_path": str(CELL5_SUMMARY_PATH),
        "cell5_summary_sha256": sha256_file(CELL5_SUMMARY_PATH),
    },
    "outputs": {
        "layer_terms": {
            "path": str(LAYER_TERMS_PATH),
            "sha256": sha256_file(LAYER_TERMS_PATH),
            "rows": int(len(layer_terms)),
        },
        "controlled_cases": {
            "path": str(CONTROLLED_CASES_PATH),
            "sha256": sha256_file(CONTROLLED_CASES_PATH),
            "rows": int(len(controlled_cases)),
        },
        "response_grid": {
            "path": str(RESPONSE_GRID_PATH),
            "sha256": sha256_file(RESPONSE_GRID_PATH),
            "rows": int(len(response_grid)),
        },
        "source_diagnostics": {
            "path": str(SOURCE_DIAGNOSTICS_PATH),
            "sha256": sha256_file(SOURCE_DIAGNOSTICS_PATH),
            "rows": int(len(source_diagnostics)),
        },
        "assumption_registry": {
            "path": str(ASSUMPTION_REGISTRY_PATH),
            "sha256": sha256_file(ASSUMPTION_REGISTRY_PATH),
            "rows": int(len(assumption_rows)),
        },
        "formula_specification": {
            "path": str(FORMULA_SPECIFICATION_PATH),
            "sha256": sha256_file(FORMULA_SPECIFICATION_PATH),
        },
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "validation": {
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warning_failures.to_dict(orient="records"),
    },
    "next_cell": (
        "Cell 7: apply the validated occurrence XoL layer to all occurrence-level "
        "gross insured losses, build the complete annual retained and ceded loss "
        "series, and calculate reinsurance AAL, OEP, AEP, and PML metrics."
    ),
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 6 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 6 BASELINE OCCURRENCE XOL LAYER COMPLETE")
print("=" * 78)
print(f"Underlying policy scenario:       {policy_scenario_id}")
print(f"Reinsurance scenario:             {LAYER_SCENARIO_ID}")
print(f"Contract type:                    occurrence excess of loss")
print(f"Attachment basis:                 {ATTACHMENT_RETURN_PERIOD_YEARS:,}-year gross OEP")
print(f"Attachment:                       ${attachment:,.0f}")
print(f"Limit:                            ${limit:,.0f}")
print(f"Exhaustion basis:                 {EXHAUSTION_RETURN_PERIOD_YEARS:,}-year gross OEP")
print(f"Exhaustion:                       ${exhaustion:,.0f}")
print(f"Reinsurer participation:          {REINSURER_PARTICIPATION:.1%}")
print(f"Triggering catalog occurrences:   {triggering_occurrences:,}")
print(f"Exhausting catalog occurrences:   {exhausting_occurrences:,}")
print(f"Gross insured preliminary AAL:    ${gross_aal:,.2f}")
print(f"Ceded preliminary AAL:            ${ceded_aal:,.2f}")
print(f"Net retained preliminary AAL:     ${retained_aal:,.2f}")
print(f"Ceded share of gross insured loss:{ceded_total / gross_total:>10.4%}")
print(
    "Critical validation checks:       "
    f"{int(validation['severity'].astype(str).str.lower().eq('critical').sum())}"
)
print(f"Critical failures:                {len(critical_failures)}")
print(f"Warnings requiring review:        {len(warning_failures)}")
print()
print("Layer terms:")
print(f"  {LAYER_TERMS_PATH}")
print("Controlled validation cases:")
print(f"  {CONTROLLED_CASES_PATH}")
print("Response grid:")
print(f"  {RESPONSE_GRID_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: calculate annual ceded and net retained reinsurance risk metrics.")



NOTEBOOK 6 CELL 6 BASELINE OCCURRENCE XOL LAYER COMPLETE
Underlying policy scenario:       baseline_full_coverage_10pct_deductible_v1
Reinsurance scenario:             baseline_occurrence_xol_500yr_to_2500yr_oep_v1
Contract type:                    occurrence excess of loss
Attachment basis:                 500-year gross OEP
Attachment:                       $18,811,084
Limit:                            $61,837,983
Exhaustion basis:                 2,500-year gross OEP
Exhaustion:                       $80,649,067
Reinsurer participation:          100.0%
Triggering catalog occurrences:   4,005
Exhausting catalog occurrences:   800
Gross insured preliminary AAL:    $122,979.56
Ceded preliminary AAL:            $63,676.60
Net retained preliminary AAL:     $59,302.96
Ceded share of gross insured loss:  51.7782%
Critical validation checks:       63
Critical failures:                0
Warnings requiring review:        0

Layer terms:
  c:\Users\USER\Documents\GitHub\seismic-correlation-in

In [14]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell7_annual_reinsurance_risk_metrics_v2"
DOLLAR_YEAR = 2022
NUMERIC_ATOL_USD = 1e-4
AGGREGATE_ATOL_USD = 0.01
TAIL_SUPPORT_WARNING_RANK = 20
EXPECTED_CELL5_WARNING_IDS = {"extreme_return_period_tail_support_documented"}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_cell_6_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def inspect_prior_validation(
    path: Path,
    allowed_warning_ids: set[str] | None = None,
) -> dict[str, Any]:
    table = pd.read_csv(path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")

    table["passed"] = parse_bool_series(table["passed"])
    severity = table["severity"].astype(str).str.strip().str.lower()
    failures = table.loc[severity.eq("critical") & ~table["passed"]].copy()
    warnings = table.loc[severity.eq("warning") & ~table["passed"]].copy()

    allowed = allowed_warning_ids or set()
    warning_ids = set(warnings["check_id"].astype(str).tolist())
    unexpected_warning_ids = sorted(warning_ids.difference(allowed))
    return {
        "critical_failures": failures,
        "warnings": warnings,
        "warning_ids": sorted(warning_ids),
        "unexpected_warning_ids": unexpected_warning_ids,
    }


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw_output:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_output,
            compresslevel=6,
            mtime=0,
        ) as compressed_output:
            with io.TextIOWrapper(
                compressed_output,
                encoding="utf-8",
                newline="",
            ) as text_output:
                frame.to_csv(text_output, index=False, lineterminator="\n")
    temporary.replace(path)


def apply_occurrence_xol(
    gross_insured_loss: np.ndarray | pd.Series | float,
    attachment: float,
    limit: float,
    participation: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    gross = np.asarray(gross_insured_loss, dtype=np.float64)
    layer_loss_before_participation = np.minimum(
        np.maximum(gross - attachment, 0.0),
        limit,
    )
    ceded = participation * layer_loss_before_participation
    retained = gross - ceded
    return layer_loss_before_participation, ceded, retained


def descending_values(losses: np.ndarray) -> np.ndarray:
    return np.sort(np.asarray(losses, dtype=np.float64))[::-1]


def empirical_pml_from_ordered(
    ordered_descending: np.ndarray,
    return_period_years: int,
) -> tuple[float, int, float]:
    n_years = int(ordered_descending.size)
    if return_period_years <= 0 or return_period_years > n_years:
        raise ValueError(
            f"Return period must be in [1, {n_years}], "
            f"received {return_period_years}."
        )
    descending_rank = max(1, int(math.ceil(n_years / return_period_years)))
    loss = float(ordered_descending[descending_rank - 1])
    empirical_aep = descending_rank / n_years
    return loss, descending_rank, empirical_aep


def build_ep_curve_from_ordered(
    ordered_descending: np.ndarray,
    loss_measure: str,
    curve_type: str,
    n_years: int,
) -> pd.DataFrame:
    positive = ordered_descending[ordered_descending > 0.0]
    if positive.size == 0:
        return pd.DataFrame(
            columns=[
                "loss_measure",
                "curve_type",
                "descending_rank",
                "annual_exceedance_probability",
                "return_period_years",
                "loss_2022_usd",
            ]
        )
    ranks = np.arange(1, positive.size + 1, dtype=np.int64)
    return pd.DataFrame(
        {
            "loss_measure": loss_measure,
            "curve_type": curve_type,
            "descending_rank": ranks,
            "annual_exceedance_probability": ranks / n_years,
            "return_period_years": n_years / ranks,
            "loss_2022_usd": positive,
        }
    )


def annual_statistics(
    losses: np.ndarray,
    loss_measure: str,
    curve_type: str,
) -> dict[str, Any]:
    values = np.asarray(losses, dtype=np.float64)
    positive = values[values > 0.0]
    mean_loss = float(np.mean(values))
    standard_deviation = float(np.std(values, ddof=0))
    return {
        "loss_measure": loss_measure,
        "curve_type": curve_type,
        "years": int(values.size),
        "positive_loss_years": int(positive.size),
        "zero_loss_years": int(values.size - positive.size),
        "annual_probability_of_positive_loss": float(positive.size / values.size),
        "mean_annual_loss_2022_usd": mean_loss,
        "annual_loss_standard_deviation_2022_usd": standard_deviation,
        "annual_loss_cov": (
            float(standard_deviation / mean_loss) if mean_loss > 0.0 else 0.0
        ),
        "mean_positive_year_loss_2022_usd": (
            float(np.mean(positive)) if positive.size else 0.0
        ),
        "median_positive_year_loss_2022_usd": (
            float(np.median(positive)) if positive.size else 0.0
        ),
        "maximum_annual_loss_2022_usd": float(np.max(values)),
    }


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_baseline_reinsurance_risk"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CELL4_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_4_validation.csv"
CELL4_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_4_summary.json"
CELL5_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_5_validation.csv"
CELL5_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_5_summary.json"
CELL6_VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_6_validation.csv"
CELL6_SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_6_summary.json"

OCCURRENCE_LOSS_PATH = OUTPUT_DIR / "baseline_reinsurance_occurrence_loss.csv.gz"
ANNUAL_SERIES_PATH = OUTPUT_DIR / "baseline_reinsurance_annual_loss_series.csv.gz"
EP_CURVE_PATH = OUTPUT_DIR / "baseline_reinsurance_exceedance_probability_curve.csv.gz"
PML_TABLE_PATH = OUTPUT_DIR / "baseline_reinsurance_pml_table.csv"
RISK_METRICS_PATH = OUTPUT_DIR / "baseline_reinsurance_risk_metrics.csv"
SOURCE_AAL_PATH = OUTPUT_DIR / "baseline_reinsurance_source_aal_summary.csv"
RECOVERY_BY_RETURN_PERIOD_PATH = (
    OUTPUT_DIR / "baseline_reinsurance_recovery_by_return_period.csv"
)
AEP_PLOT_PNG_PATH = PLOT_DIR / "baseline_reinsurance_aep_comparison.png"
AEP_PLOT_PDF_PATH = PLOT_DIR / "baseline_reinsurance_aep_comparison.pdf"
OEP_PLOT_PNG_PATH = PLOT_DIR / "baseline_reinsurance_oep_comparison.png"
OEP_PLOT_PDF_PATH = PLOT_DIR / "baseline_reinsurance_oep_comparison.pdf"
RECOVERY_PLOT_PNG_PATH = PLOT_DIR / "baseline_reinsurance_recovery_by_return_period.png"
RECOVERY_PLOT_PDF_PATH = PLOT_DIR / "baseline_reinsurance_recovery_by_return_period.pdf"
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_7_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_7_summary.json"

validation_rows: list[dict[str, Any]] = []
required_inputs = [
    CELL4_VALIDATION_PATH,
    CELL4_SUMMARY_PATH,
    CELL5_VALIDATION_PATH,
    CELL5_SUMMARY_PATH,
    CELL6_VALIDATION_PATH,
    CELL6_SUMMARY_PATH,
]
for path in required_inputs:
    append_check(
        validation_rows,
        f"required_input_exists_{path.stem}",
        path.exists(),
        str(path),
    )

missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise FileNotFoundError(f"Required inputs are missing: {missing_inputs}")

cell4_validation = inspect_prior_validation(CELL4_VALIDATION_PATH)
cell5_validation = inspect_prior_validation(
    CELL5_VALIDATION_PATH,
    allowed_warning_ids=EXPECTED_CELL5_WARNING_IDS,
)
cell6_validation = inspect_prior_validation(CELL6_VALIDATION_PATH)
for cell_name, status in [
    ("cell4", cell4_validation),
    ("cell5", cell5_validation),
    ("cell6", cell6_validation),
]:
    append_check(
        validation_rows,
        f"{cell_name}_critical_validation_passed",
        status["critical_failures"].empty,
        f"critical_failures={len(status['critical_failures'])}",
    )
append_check(
    validation_rows,
    "cell4_has_no_unresolved_warnings",
    len(cell4_validation["warning_ids"]) == 0,
    f"warning_ids={cell4_validation['warning_ids']}",
)
append_check(
    validation_rows,
    "cell5_warnings_are_expected",
    len(cell5_validation["unexpected_warning_ids"]) == 0,
    (
        f"warning_ids={cell5_validation['warning_ids']}; "
        f"unexpected={cell5_validation['unexpected_warning_ids']}"
    ),
)
append_check(
    validation_rows,
    "cell6_has_no_unresolved_warnings",
    len(cell6_validation["warning_ids"]) == 0,
    f"warning_ids={cell6_validation['warning_ids']}",
)

cell4_summary = load_json(CELL4_SUMMARY_PATH)
cell5_summary = load_json(CELL5_SUMMARY_PATH)
cell6_summary = load_json(CELL6_SUMMARY_PATH)
for cell_name, summary in [
    ("cell4", cell4_summary),
    ("cell5", cell5_summary),
    ("cell6", cell6_summary),
]:
    append_check(
        validation_rows,
        f"{cell_name}_summary_declares_success",
        bool(summary.get("all_critical_checks_passed", False)),
        f"all_critical_checks_passed={summary.get('all_critical_checks_passed')}",
    )

policy_scenario_id = str(
    cell6_summary["underlying_policy_scenario"]["policy_scenario_id"]
)
policy_scenario_name = str(
    cell6_summary["underlying_policy_scenario"]["policy_scenario_name"]
)
layer = cell6_summary["reinsurance_scenario"]
layer_scenario_id = str(layer["layer_scenario_id"])
layer_scenario_name = str(layer["layer_scenario_name"])
attachment = float(layer["attachment_2022_usd"])
limit = float(layer["limit_2022_usd"])
exhaustion = float(layer["exhaustion_2022_usd"])
participation = float(layer["reinsurer_participation"])

append_check(
    validation_rows,
    "policy_scenarios_match_across_cells",
    (
        policy_scenario_id
        == str(cell4_summary["policy_scenario"]["policy_scenario_id"])
        == str(cell5_summary["policy_scenario"]["policy_scenario_id"])
    ),
    (
        f"cell4={cell4_summary['policy_scenario']['policy_scenario_id']}; "
        f"cell5={cell5_summary['policy_scenario']['policy_scenario_id']}; "
        f"cell6={policy_scenario_id}"
    ),
)
append_check(
    validation_rows,
    "layer_terms_positive_and_ordered",
    attachment >= 0.0
    and limit > 0.0
    and exhaustion > attachment
    and abs(exhaustion - attachment - limit) <= NUMERIC_ATOL_USD,
    (
        f"attachment={attachment:.6f}; limit={limit:.6f}; "
        f"exhaustion={exhaustion:.6f}"
    ),
)
append_check(
    validation_rows,
    "reinsurer_participation_bounded",
    0.0 <= participation <= 1.0,
    f"participation={participation:.6f}",
)
append_check(
    validation_rows,
    "baseline_has_no_annual_aggregate_limit",
    layer.get("annual_aggregate_limit_2022_usd") is None,
    f"annual_aggregate_limit={layer.get('annual_aggregate_limit_2022_usd')}",
)

annual_catalog = cell5_summary["annual_catalog"]
declared_years = int(annual_catalog["declared_duration_years"])
expected_occurrences = int(annual_catalog["occurrences"])
expected_occupied_years = int(annual_catalog["occupied_years"])
expected_multiple_event_years = int(annual_catalog["multiple_event_years"])
expected_zero_event_years = int(annual_catalog["zero_event_years"])
portfolio_replacement_value = float(
    cell5_summary["portfolio"]["replacement_value_2022_usd"]
)
return_periods = [
    int(value) for value in cell5_summary["pml_method"]["return_periods_years"]
]
append_check(
    validation_rows,
    "return_periods_supported_by_catalog",
    bool(return_periods)
    and min(return_periods) > 0
    and max(return_periods) <= declared_years,
    f"return_periods={return_periods}; catalog_years={declared_years}",
)

cell4_event_record = cell4_summary["outputs"]["event_insured_loss"]
event_loss_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(cell4_event_record["path"]),
)
actual_event_hash = sha256_file(event_loss_path)
append_check(
    validation_rows,
    "cell4_event_loss_hash_matches",
    actual_event_hash == str(cell4_event_record["sha256"]),
    f"expected={cell4_event_record['sha256']}; actual={actual_event_hash}",
)

cell5_annual_record = cell5_summary["outputs"]["annual_loss_series"]
cell5_annual_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(cell5_annual_record["path"]),
)
actual_cell5_annual_hash = sha256_file(cell5_annual_path)
append_check(
    validation_rows,
    "cell5_annual_series_hash_matches",
    actual_cell5_annual_hash == str(cell5_annual_record["sha256"]),
    f"expected={cell5_annual_record['sha256']}; actual={actual_cell5_annual_hash}",
)

cell5_pml_record = cell5_summary["outputs"]["pml_table"]
cell5_pml_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(cell5_pml_record["path"]),
)
actual_cell5_pml_hash = sha256_file(cell5_pml_path)
append_check(
    validation_rows,
    "cell5_pml_hash_matches",
    actual_cell5_pml_hash == str(cell5_pml_record["sha256"]),
    f"expected={cell5_pml_record['sha256']}; actual={actual_cell5_pml_hash}",
)

layer_terms_record = cell6_summary["outputs"]["layer_terms"]
layer_terms_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(layer_terms_record["path"]),
)
actual_layer_terms_hash = sha256_file(layer_terms_path)
append_check(
    validation_rows,
    "cell6_layer_terms_hash_matches",
    actual_layer_terms_hash == str(layer_terms_record["sha256"]),
    f"expected={layer_terms_record['sha256']}; actual={actual_layer_terms_hash}",
)

controlled_record = cell6_summary["outputs"]["controlled_cases"]
controlled_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(controlled_record["path"]),
)
actual_controlled_hash = sha256_file(controlled_path)
append_check(
    validation_rows,
    "cell6_controlled_cases_hash_matches",
    actual_controlled_hash == str(controlled_record["sha256"]),
    f"expected={controlled_record['sha256']}; actual={actual_controlled_hash}",
)

source_diagnostics_record = cell6_summary["outputs"]["source_diagnostics"]
source_diagnostics_path = resolve_recorded_path(
    PROJECT_ROOT,
    str(source_diagnostics_record["path"]),
)
actual_source_diagnostics_hash = sha256_file(source_diagnostics_path)
append_check(
    validation_rows,
    "cell6_source_diagnostics_hash_matches",
    actual_source_diagnostics_hash == str(source_diagnostics_record["sha256"]),
    (
        f"expected={source_diagnostics_record['sha256']}; "
        f"actual={actual_source_diagnostics_hash}"
    ),
)

layer_terms = pd.read_csv(layer_terms_path)
required_layer_columns = {
    "layer_scenario_id",
    "underlying_policy_scenario_id",
    "attachment_2022_usd",
    "limit_2022_usd",
    "exhaustion_2022_usd",
    "reinsurer_participation",
}
missing_layer_columns = sorted(required_layer_columns.difference(layer_terms.columns))
append_check(
    validation_rows,
    "layer_terms_schema_complete",
    len(layer_terms) == 1 and not missing_layer_columns,
    f"rows={len(layer_terms)}; missing={missing_layer_columns}",
)
if len(layer_terms) != 1 or missing_layer_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise ValueError("Cell 6 layer-terms table is incomplete.")
layer_row = layer_terms.iloc[0]
append_check(
    validation_rows,
    "layer_terms_table_matches_summary",
    (
        str(layer_row["layer_scenario_id"]) == layer_scenario_id
        and str(layer_row["underlying_policy_scenario_id"]) == policy_scenario_id
        and abs(float(layer_row["attachment_2022_usd"]) - attachment)
        <= NUMERIC_ATOL_USD
        and abs(float(layer_row["limit_2022_usd"]) - limit)
        <= NUMERIC_ATOL_USD
        and abs(float(layer_row["exhaustion_2022_usd"]) - exhaustion)
        <= NUMERIC_ATOL_USD
        and abs(float(layer_row["reinsurer_participation"]) - participation)
        <= 1e-12
    ),
    "layer scenario, policy scenario, attachment, limit, exhaustion, and participation checked",
)

events = pd.read_csv(event_loss_path, compression="gzip")
required_event_columns = {
    "catalog_year",
    "occurrence_id",
    "source_type",
    "gross_insured_loss_2022_usd",
}
missing_event_columns = sorted(required_event_columns.difference(events.columns))
append_check(
    validation_rows,
    "event_loss_schema_complete",
    not missing_event_columns,
    f"missing={missing_event_columns}",
)
if missing_event_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise KeyError(f"Cell 4 event-loss file is missing columns: {missing_event_columns}")

for column in ["catalog_year", "gross_insured_loss_2022_usd"]:
    events[column] = pd.to_numeric(events[column], errors="raise")
events["catalog_year"] = events["catalog_year"].astype(np.int64)
events["occurrence_id"] = events["occurrence_id"].astype(str).str.strip()
events["source_type"] = events["source_type"].astype(str).str.strip()

gross = events["gross_insured_loss_2022_usd"].to_numpy(dtype=np.float64)
layer_before, ceded, retained = apply_occurrence_xol(
    gross,
    attachment,
    limit,
    participation,
)
events["reinsurance_layer_scenario_id"] = layer_scenario_id
events["attachment_2022_usd"] = attachment
events["limit_2022_usd"] = limit
events["exhaustion_2022_usd"] = exhaustion
events["reinsurer_participation"] = participation
events["attachment_triggered"] = gross > attachment
events["layer_exhausted"] = gross >= exhaustion
events["layer_loss_before_participation_2022_usd"] = layer_before
events["ceded_loss_2022_usd"] = ceded
events["net_retained_loss_2022_usd"] = retained
events["ceded_share_of_gross_insured_loss"] = np.divide(
    ceded,
    gross,
    out=np.zeros_like(ceded),
    where=gross > 0.0,
)
events["net_retained_share_of_gross_insured_loss"] = np.divide(
    retained,
    gross,
    out=np.zeros_like(retained),
    where=gross > 0.0,
)

first_columns = [
    "catalog_year",
    "occurrence_id",
    "source_type",
    "gross_insured_loss_2022_usd",
    "reinsurance_layer_scenario_id",
    "attachment_2022_usd",
    "limit_2022_usd",
    "exhaustion_2022_usd",
    "reinsurer_participation",
    "attachment_triggered",
    "layer_exhausted",
    "layer_loss_before_participation_2022_usd",
    "ceded_loss_2022_usd",
    "net_retained_loss_2022_usd",
    "ceded_share_of_gross_insured_loss",
    "net_retained_share_of_gross_insured_loss",
]
remaining_columns = [column for column in events.columns if column not in first_columns]
events = events[first_columns + remaining_columns]
write_gzip_csv_deterministic(events, OCCURRENCE_LOSS_PATH)

append_check(
    validation_rows,
    "occurrence_count_and_keys_match",
    len(events) == expected_occurrences and events["occurrence_id"].is_unique,
    (
        f"rows={len(events)}; expected={expected_occurrences}; "
        f"unique_occurrences={events['occurrence_id'].nunique()}"
    ),
)
append_check(
    validation_rows,
    "catalog_years_within_declared_range",
    events["catalog_year"].between(1, declared_years).all(),
    (
        f"minimum={events['catalog_year'].min()}; "
        f"maximum={events['catalog_year'].max()}"
    ),
)
append_check(
    validation_rows,
    "occurrence_losses_finite_and_nonnegative",
    np.isfinite(np.column_stack([gross, layer_before, ceded, retained])).all()
    and np.all(gross >= 0.0)
    and np.all(layer_before >= 0.0)
    and np.all(ceded >= 0.0)
    and np.all(retained >= 0.0),
    "gross, layer, ceded, and retained occurrence losses checked",
)
append_check(
    validation_rows,
    "ceded_not_above_participating_limit",
    np.all(ceded <= participation * limit + NUMERIC_ATOL_USD),
    f"maximum_ceded={ceded.max():.6f}; participating_limit={participation * limit:.6f}",
)
append_check(
    validation_rows,
    "ceded_and_retained_not_above_gross",
    np.all(ceded <= gross + NUMERIC_ATOL_USD)
    and np.all(retained <= gross + NUMERIC_ATOL_USD),
    (
        f"maximum_ceded_minus_gross={np.max(ceded - gross):.3e}; "
        f"maximum_retained_minus_gross={np.max(retained - gross):.3e}"
    ),
)
occurrence_conservation_error = np.abs(gross - ceded - retained)
append_check(
    validation_rows,
    "occurrence_gross_equals_ceded_plus_retained",
    float(occurrence_conservation_error.max()) <= NUMERIC_ATOL_USD,
    f"maximum_error={occurrence_conservation_error.max():.3e}",
)
append_check(
    validation_rows,
    "below_attachment_occurrences_have_zero_ceded",
    np.all(ceded[gross <= attachment] <= NUMERIC_ATOL_USD),
    f"occurrences_checked={int(np.count_nonzero(gross <= attachment))}",
)
append_check(
    validation_rows,
    "exhausting_occurrences_have_full_participating_limit",
    np.allclose(
        ceded[gross >= exhaustion],
        participation * limit,
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    f"occurrences_checked={int(np.count_nonzero(gross >= exhaustion))}",
)
append_check(
    validation_rows,
    "triggering_occurrence_count_matches_cell6",
    int(np.count_nonzero(gross > attachment))
    == int(
        cell6_summary["diagnostic_application_to_catalog_occurrences"][
            "triggering_occurrences"
        ]
    ),
    (
        f"cell7={int(np.count_nonzero(gross > attachment))}; "
        "cell6="
        f"{cell6_summary['diagnostic_application_to_catalog_occurrences']['triggering_occurrences']}"
    ),
)
append_check(
    validation_rows,
    "exhausting_occurrence_count_matches_cell6",
    int(np.count_nonzero(gross >= exhaustion))
    == int(
        cell6_summary["diagnostic_application_to_catalog_occurrences"][
            "exhausting_occurrences"
        ]
    ),
    (
        f"cell7={int(np.count_nonzero(gross >= exhaustion))}; "
        "cell6="
        f"{cell6_summary['diagnostic_application_to_catalog_occurrences']['exhausting_occurrences']}"
    ),
)

cell6_diagnostics = cell6_summary["diagnostic_application_to_catalog_occurrences"]
for name, calculated, expected in [
    (
        "gross_total",
        float(gross.sum()),
        float(cell6_diagnostics["gross_insured_loss_total_2022_usd"]),
    ),
    (
        "ceded_total",
        float(ceded.sum()),
        float(cell6_diagnostics["ceded_loss_total_2022_usd"]),
    ),
    (
        "net_retained_total",
        float(retained.sum()),
        float(cell6_diagnostics["net_retained_loss_total_2022_usd"]),
    ),
]:
    append_check(
        validation_rows,
        f"occurrence_{name}_matches_cell6",
        math.isclose(
            calculated,
            expected,
            rel_tol=1e-12,
            abs_tol=AGGREGATE_ATOL_USD,
        ),
        f"cell7={calculated:.6f}; cell6={expected:.6f}",
    )

controlled = pd.read_csv(controlled_path)
actual_controls = controlled.loc[
    controlled["case_type"].astype(str).eq("actual_catalog_occurrence")
].copy()
control_comparison = actual_controls.merge(
    events[
        [
            "occurrence_id",
            "gross_insured_loss_2022_usd",
            "layer_loss_before_participation_2022_usd",
            "ceded_loss_2022_usd",
            "net_retained_loss_2022_usd",
        ]
    ],
    on="occurrence_id",
    how="left",
    suffixes=("_cell6", "_cell7"),
    validate="many_to_one",
)
control_error_columns = []
for measure in [
    "gross_insured_loss_2022_usd",
    "layer_loss_before_participation_2022_usd",
    "ceded_loss_2022_usd",
    "net_retained_loss_2022_usd",
]:
    error_column = f"{measure}_absolute_error"
    control_comparison[error_column] = np.abs(
        control_comparison[f"{measure}_cell6"]
        - control_comparison[f"{measure}_cell7"]
    )
    control_error_columns.append(error_column)
maximum_control_error = float(
    control_comparison[control_error_columns].to_numpy(dtype=float).max()
    if len(control_comparison)
    else 0.0
)
append_check(
    validation_rows,
    "cell6_actual_control_cases_reproduced",
    len(control_comparison) == len(actual_controls)
    and maximum_control_error <= NUMERIC_ATOL_USD,
    f"rows={len(control_comparison)}; maximum_error={maximum_control_error:.3e}",
)

cell5_annual = pd.read_csv(cell5_annual_path, compression="gzip")
required_annual_columns = {
    "catalog_year",
    "event_count",
    "zero_event_year",
    "ground_up_aep_2022_usd",
    "ground_up_oep_2022_usd",
    "gross_insured_aep_2022_usd",
    "gross_insured_oep_2022_usd",
    "uninsured_aep_2022_usd",
    "uninsured_oep_2022_usd",
}
missing_annual_columns = sorted(required_annual_columns.difference(cell5_annual.columns))
append_check(
    validation_rows,
    "cell5_annual_series_schema_complete",
    not missing_annual_columns,
    f"missing={missing_annual_columns}",
)
if missing_annual_columns:
    pd.DataFrame(validation_rows).to_csv(VALIDATION_PATH, index=False)
    raise KeyError(f"Cell 5 annual series is missing columns: {missing_annual_columns}")

cell5_annual["catalog_year"] = pd.to_numeric(
    cell5_annual["catalog_year"], errors="raise"
).astype(np.int64)
cell5_annual["event_count"] = pd.to_numeric(
    cell5_annual["event_count"], errors="raise"
).astype(np.int16)
for column in sorted(required_annual_columns.difference({"catalog_year", "event_count", "zero_event_year"})):
    cell5_annual[column] = pd.to_numeric(cell5_annual[column], errors="raise")

annual_occurrence = (
    events.groupby("catalog_year", sort=True)
    .agg(
        event_count=("occurrence_id", "size"),
        gross_insured_aep_2022_usd=("gross_insured_loss_2022_usd", "sum"),
        gross_insured_oep_2022_usd=("gross_insured_loss_2022_usd", "max"),
        ceded_aep_2022_usd=("ceded_loss_2022_usd", "sum"),
        ceded_oep_2022_usd=("ceded_loss_2022_usd", "max"),
        net_retained_aep_2022_usd=("net_retained_loss_2022_usd", "sum"),
        net_retained_oep_2022_usd=("net_retained_loss_2022_usd", "max"),
        triggering_occurrences=("attachment_triggered", "sum"),
        exhausting_occurrences=("layer_exhausted", "sum"),
    )
    .reset_index()
)

annual_series = cell5_annual.merge(
    annual_occurrence,
    on="catalog_year",
    how="left",
    suffixes=("_cell5", "_cell7"),
    validate="one_to_one",
)
fill_zero_columns = [
    "event_count_cell7",
    "gross_insured_aep_2022_usd_cell7",
    "gross_insured_oep_2022_usd_cell7",
    "ceded_aep_2022_usd",
    "ceded_oep_2022_usd",
    "net_retained_aep_2022_usd",
    "net_retained_oep_2022_usd",
    "triggering_occurrences",
    "exhausting_occurrences",
]
annual_series[fill_zero_columns] = annual_series[fill_zero_columns].fillna(0.0)
annual_series["event_count_cell7"] = annual_series["event_count_cell7"].astype(np.int16)
annual_series["triggering_occurrences"] = annual_series["triggering_occurrences"].astype(np.int16)
annual_series["exhausting_occurrences"] = annual_series["exhausting_occurrences"].astype(np.int16)

append_check(
    validation_rows,
    "annual_series_contains_all_catalog_years",
    len(annual_series) == declared_years
    and int(annual_series["catalog_year"].iloc[0]) == 1
    and int(annual_series["catalog_year"].iloc[-1]) == declared_years,
    (
        f"rows={len(annual_series)}; first={annual_series['catalog_year'].iloc[0]}; "
        f"last={annual_series['catalog_year'].iloc[-1]}"
    ),
)
append_check(
    validation_rows,
    "event_counts_reproduce_cell5",
    np.array_equal(
        annual_series["event_count_cell5"].to_numpy(dtype=np.int16),
        annual_series["event_count_cell7"].to_numpy(dtype=np.int16),
    ),
    (
        "maximum_difference="
        f"{np.max(np.abs(annual_series['event_count_cell5'].to_numpy(dtype=int) - annual_series['event_count_cell7'].to_numpy(dtype=int)))}"
    ),
)
append_check(
    validation_rows,
    "gross_aep_reproduces_cell5",
    np.allclose(
        annual_series["gross_insured_aep_2022_usd_cell5"],
        annual_series["gross_insured_aep_2022_usd_cell7"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_difference="
        f"{np.max(np.abs(annual_series['gross_insured_aep_2022_usd_cell5'] - annual_series['gross_insured_aep_2022_usd_cell7'])):.3e}"
    ),
)
append_check(
    validation_rows,
    "gross_oep_reproduces_cell5",
    np.allclose(
        annual_series["gross_insured_oep_2022_usd_cell5"],
        annual_series["gross_insured_oep_2022_usd_cell7"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_difference="
        f"{np.max(np.abs(annual_series['gross_insured_oep_2022_usd_cell5'] - annual_series['gross_insured_oep_2022_usd_cell7'])):.3e}"
    ),
)

annual_series = annual_series.rename(
    columns={
        "event_count_cell5": "event_count",
        "gross_insured_aep_2022_usd_cell5": "gross_insured_aep_2022_usd",
        "gross_insured_oep_2022_usd_cell5": "gross_insured_oep_2022_usd",
    }
).drop(
    columns=[
        "event_count_cell7",
        "gross_insured_aep_2022_usd_cell7",
        "gross_insured_oep_2022_usd_cell7",
    ]
)
annual_column_order = [
    "catalog_year",
    "event_count",
    "zero_event_year",
    "triggering_occurrences",
    "exhausting_occurrences",
    "ground_up_aep_2022_usd",
    "ground_up_oep_2022_usd",
    "uninsured_aep_2022_usd",
    "uninsured_oep_2022_usd",
    "gross_insured_aep_2022_usd",
    "gross_insured_oep_2022_usd",
    "ceded_aep_2022_usd",
    "ceded_oep_2022_usd",
    "net_retained_aep_2022_usd",
    "net_retained_oep_2022_usd",
]
annual_series = annual_series[annual_column_order]
write_gzip_csv_deterministic(annual_series, ANNUAL_SERIES_PATH)

occupied_years = int(np.count_nonzero(annual_series["event_count"].to_numpy()))
multiple_event_years = int(np.count_nonzero(annual_series["event_count"].to_numpy() > 1))
zero_event_years = int(np.count_nonzero(annual_series["event_count"].to_numpy() == 0))
append_check(
    validation_rows,
    "occupied_year_count_matches_cell5",
    occupied_years == expected_occupied_years,
    f"cell7={occupied_years}; cell5={expected_occupied_years}",
)
append_check(
    validation_rows,
    "multiple_event_year_count_matches_cell5",
    multiple_event_years == expected_multiple_event_years,
    f"cell7={multiple_event_years}; cell5={expected_multiple_event_years}",
)
append_check(
    validation_rows,
    "zero_event_year_count_matches_cell5",
    zero_event_years == expected_zero_event_years,
    f"cell7={zero_event_years}; cell5={expected_zero_event_years}",
)

annual_gross_aep = annual_series["gross_insured_aep_2022_usd"].to_numpy(dtype=float)
annual_gross_oep = annual_series["gross_insured_oep_2022_usd"].to_numpy(dtype=float)
annual_ceded_aep = annual_series["ceded_aep_2022_usd"].to_numpy(dtype=float)
annual_ceded_oep = annual_series["ceded_oep_2022_usd"].to_numpy(dtype=float)
annual_retained_aep = annual_series["net_retained_aep_2022_usd"].to_numpy(dtype=float)
annual_retained_oep = annual_series["net_retained_oep_2022_usd"].to_numpy(dtype=float)

aep_conservation_error = np.abs(
    annual_gross_aep - annual_ceded_aep - annual_retained_aep
)
oep_conservation_error = np.abs(
    annual_gross_oep - annual_ceded_oep - annual_retained_oep
)
append_check(
    validation_rows,
    "annual_aep_gross_equals_ceded_plus_retained",
    float(aep_conservation_error.max()) <= NUMERIC_ATOL_USD,
    f"maximum_error={aep_conservation_error.max():.3e}",
)
append_check(
    validation_rows,
    "annual_oep_gross_equals_ceded_plus_retained",
    float(oep_conservation_error.max()) <= NUMERIC_ATOL_USD,
    f"maximum_error={oep_conservation_error.max():.3e}",
)
for measure_name, aep_values, oep_values in [
    ("gross_insured", annual_gross_aep, annual_gross_oep),
    ("ceded", annual_ceded_aep, annual_ceded_oep),
    ("net_retained", annual_retained_aep, annual_retained_oep),
]:
    append_check(
        validation_rows,
        f"{measure_name}_aep_not_less_than_oep",
        np.all(aep_values + NUMERIC_ATOL_USD >= oep_values),
        f"minimum_difference={np.min(aep_values - oep_values):.6f}",
    )
    append_check(
        validation_rows,
        f"{measure_name}_single_and_zero_event_years_have_equal_aep_oep",
        np.allclose(
            aep_values[annual_series["event_count"].to_numpy() <= 1],
            oep_values[annual_series["event_count"].to_numpy() <= 1],
            rtol=0.0,
            atol=NUMERIC_ATOL_USD,
        ),
        (
            "years_checked="
            f"{int(np.count_nonzero(annual_series['event_count'].to_numpy() <= 1))}"
        ),
    )
append_check(
    validation_rows,
    "annual_ceded_and_retained_not_above_gross",
    np.all(annual_ceded_aep <= annual_gross_aep + NUMERIC_ATOL_USD)
    and np.all(annual_retained_aep <= annual_gross_aep + NUMERIC_ATOL_USD)
    and np.all(annual_ceded_oep <= annual_gross_oep + NUMERIC_ATOL_USD)
    and np.all(annual_retained_oep <= annual_gross_oep + NUMERIC_ATOL_USD),
    "AEP and OEP annual vectors checked",
)

for name, annual_values, event_values in [
    ("gross", annual_gross_aep, gross),
    ("ceded", annual_ceded_aep, ceded),
    ("net_retained", annual_retained_aep, retained),
]:
    append_check(
        validation_rows,
        f"annual_{name}_total_matches_occurrence_total",
        math.isclose(
            float(annual_values.sum()),
            float(event_values.sum()),
            rel_tol=1e-12,
            abs_tol=AGGREGATE_ATOL_USD,
        ),
        f"annual={annual_values.sum():.6f}; occurrence={event_values.sum():.6f}",
    )

calculated_aal = {
    "gross_insured": float(annual_gross_aep.mean()),
    "ceded": float(annual_ceded_aep.mean()),
    "net_retained": float(annual_retained_aep.mean()),
}
append_check(
    validation_rows,
    "gross_aal_matches_cell5",
    math.isclose(
        calculated_aal["gross_insured"],
        float(cell5_summary["risk_metrics"]["gross_insured_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell7={calculated_aal['gross_insured']:.9f}; "
        f"cell5={float(cell5_summary['risk_metrics']['gross_insured_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "ceded_aal_matches_cell6",
    math.isclose(
        calculated_aal["ceded"],
        float(cell6_diagnostics["ceded_preliminary_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell7={calculated_aal['ceded']:.9f}; "
        f"cell6={float(cell6_diagnostics['ceded_preliminary_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "net_retained_aal_matches_cell6",
    math.isclose(
        calculated_aal["net_retained"],
        float(cell6_diagnostics["net_retained_preliminary_aal_2022_usd"]),
        rel_tol=1e-12,
        abs_tol=NUMERIC_ATOL_USD,
    ),
    (
        f"cell7={calculated_aal['net_retained']:.9f}; "
        f"cell6={float(cell6_diagnostics['net_retained_preliminary_aal_2022_usd']):.9f}"
    ),
)
append_check(
    validation_rows,
    "aal_gross_equals_ceded_plus_retained",
    abs(
        calculated_aal["gross_insured"]
        - calculated_aal["ceded"]
        - calculated_aal["net_retained"]
    )
    <= NUMERIC_ATOL_USD,
    (
        f"gross={calculated_aal['gross_insured']:.9f}; "
        f"ceded={calculated_aal['ceded']:.9f}; "
        f"retained={calculated_aal['net_retained']:.9f}"
    ),
)

loss_vectors = {
    ("gross_insured", "AEP"): annual_gross_aep,
    ("gross_insured", "OEP"): annual_gross_oep,
    ("ceded", "AEP"): annual_ceded_aep,
    ("ceded", "OEP"): annual_ceded_oep,
    ("net_retained", "AEP"): annual_retained_aep,
    ("net_retained", "OEP"): annual_retained_oep,
}
ordered_vectors = {
    key: descending_values(values) for key, values in loss_vectors.items()
}
ep_curve = pd.concat(
    [
        build_ep_curve_from_ordered(
            ordered_vectors[(measure, curve_type)],
            measure,
            curve_type,
            declared_years,
        )
        for measure, curve_type in loss_vectors
    ],
    ignore_index=True,
)
write_gzip_csv_deterministic(ep_curve, EP_CURVE_PATH)

pml_rows: list[dict[str, Any]] = []
for return_period in return_periods:
    row: dict[str, Any] = {
        "return_period_years": int(return_period),
        "target_annual_exceedance_probability": 1.0 / return_period,
    }
    supporting_ranks: list[int] = []
    for measure, curve_type in loss_vectors:
        loss, rank, empirical_aep = empirical_pml_from_ordered(
            ordered_vectors[(measure, curve_type)],
            return_period,
        )
        prefix = f"{measure}_{curve_type.lower()}"
        row[f"{prefix}_pml_2022_usd"] = loss
        row[f"{prefix}_supporting_descending_rank"] = rank
        row[f"{prefix}_empirical_aep"] = empirical_aep
        supporting_ranks.append(rank)

    for curve_type in ["aep", "oep"]:
        gross_loss = float(row[f"gross_insured_{curve_type}_pml_2022_usd"])
        ceded_loss = float(row[f"ceded_{curve_type}_pml_2022_usd"])
        retained_loss = float(row[f"net_retained_{curve_type}_pml_2022_usd"])
        row[f"ceded_share_of_gross_{curve_type}"] = (
            ceded_loss / gross_loss if gross_loss > 0.0 else 0.0
        )
        row[f"net_retained_share_of_gross_{curve_type}"] = (
            retained_loss / gross_loss if gross_loss > 0.0 else 0.0
        )
        row[f"layer_limit_utilization_{curve_type}"] = (
            ceded_loss / (participation * limit)
            if participation * limit > 0.0
            else 0.0
        )

    row["minimum_supporting_descending_rank"] = min(supporting_ranks)
    row["tail_support_flag"] = (
        "thin_tail_support"
        if min(supporting_ranks) < TAIL_SUPPORT_WARNING_RANK
        else "adequate_order_statistic_support"
    )
    pml_rows.append(row)

pml_table = pd.DataFrame(pml_rows)
pml_table.to_csv(PML_TABLE_PATH, index=False)

for measure in ["gross_insured", "ceded", "net_retained"]:
    for curve_type in ["aep", "oep"]:
        column = f"{measure}_{curve_type}_pml_2022_usd"
        differences = np.diff(pml_table[column].to_numpy(dtype=float))
        append_check(
            validation_rows,
            f"{measure}_{curve_type}_pml_nondecreasing_with_return_period",
            np.all(differences >= -NUMERIC_ATOL_USD),
            (
                f"minimum_difference={float(differences.min()) if differences.size else 0.0:.6f}"
            ),
        )

for measure in ["gross_insured", "ceded", "net_retained"]:
    append_check(
        validation_rows,
        f"{measure}_aep_pml_not_less_than_oep_pml",
        np.all(
            pml_table[f"{measure}_aep_pml_2022_usd"].to_numpy(dtype=float)
            + NUMERIC_ATOL_USD
            >= pml_table[f"{measure}_oep_pml_2022_usd"].to_numpy(dtype=float)
        ),
        "checked all return periods",
    )

for curve_type in ["aep", "oep"]:
    gross_pml = pml_table[f"gross_insured_{curve_type}_pml_2022_usd"].to_numpy(
        dtype=float
    )
    ceded_pml = pml_table[f"ceded_{curve_type}_pml_2022_usd"].to_numpy(dtype=float)
    retained_pml = pml_table[
        f"net_retained_{curve_type}_pml_2022_usd"
    ].to_numpy(dtype=float)
    append_check(
        validation_rows,
        f"{curve_type}_ceded_and_retained_pml_not_above_gross",
        np.all(ceded_pml <= gross_pml + NUMERIC_ATOL_USD)
        and np.all(retained_pml <= gross_pml + NUMERIC_ATOL_USD),
        "checked all return periods",
    )
    ceded_share = pml_table[f"ceded_share_of_gross_{curve_type}"].to_numpy(
        dtype=float
    )
    retained_share = pml_table[
        f"net_retained_share_of_gross_{curve_type}"
    ].to_numpy(dtype=float)
    utilization = pml_table[f"layer_limit_utilization_{curve_type}"].to_numpy(
        dtype=float
    )
    shares_bounded = (
        np.all((ceded_share >= -1e-12) & (ceded_share <= 1.0 + 1e-12))
        and np.all((retained_share >= -1e-12) & (retained_share <= 1.0 + 1e-12))
    )
    if curve_type == "aep":
        utilization_upper_bound = float(
            annual_series["triggering_occurrences"].max()
        )
        utilization_interpretation = (
            "annual aggregate ceded PML divided by one participated occurrence "
            "layer limit; values may exceed 1 when multiple occurrences recover "
            "in the same year because the baseline has no annual aggregate cap "
            "or reinstatement constraint"
        )
    else:
        utilization_upper_bound = 1.0
        utilization_interpretation = (
            "maximum single-occurrence ceded PML divided by the participated "
            "occurrence layer limit"
        )

    utilization_bounded = np.all(utilization >= -1e-12) and np.all(
        utilization <= utilization_upper_bound + 1e-12
    )
    append_check(
        validation_rows,
        f"{curve_type}_reinsurance_shares_and_utilization_bounded",
        shares_bounded and utilization_bounded,
        (
            f"ceded_share_range=({ceded_share.min():.6f}, {ceded_share.max():.6f}); "
            f"retained_share_range=({retained_share.min():.6f}, {retained_share.max():.6f}); "
            f"utilization_range=({utilization.min():.6f}, {utilization.max():.6f}); "
            f"utilization_upper_bound={utilization_upper_bound:.6f}; "
            f"interpretation={utilization_interpretation}"
        ),
    )

cell5_pml = pd.read_csv(cell5_pml_path)
comparison = pml_table.merge(
    cell5_pml[
        [
            "return_period_years",
            "gross_insured_aep_pml_2022_usd",
            "gross_insured_oep_pml_2022_usd",
        ]
    ],
    on="return_period_years",
    how="left",
    suffixes=("_cell7", "_cell5"),
    validate="one_to_one",
)
append_check(
    validation_rows,
    "gross_insured_pml_reproduces_cell5",
    comparison["gross_insured_aep_pml_2022_usd_cell5"].notna().all()
    and comparison["gross_insured_oep_pml_2022_usd_cell5"].notna().all()
    and np.allclose(
        comparison["gross_insured_aep_pml_2022_usd_cell7"],
        comparison["gross_insured_aep_pml_2022_usd_cell5"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    )
    and np.allclose(
        comparison["gross_insured_oep_pml_2022_usd_cell7"],
        comparison["gross_insured_oep_pml_2022_usd_cell5"],
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_aep_difference="
        f"{np.max(np.abs(comparison['gross_insured_aep_pml_2022_usd_cell7'] - comparison['gross_insured_aep_pml_2022_usd_cell5'])):.3e}; "
        "maximum_oep_difference="
        f"{np.max(np.abs(comparison['gross_insured_oep_pml_2022_usd_cell7'] - comparison['gross_insured_oep_pml_2022_usd_cell5'])):.3e}"
    ),
)

expected_oep_ceded = np.minimum(
    np.maximum(
        pml_table["gross_insured_oep_pml_2022_usd"].to_numpy(dtype=float)
        - attachment,
        0.0,
    ),
    limit,
) * participation
expected_oep_retained = (
    pml_table["gross_insured_oep_pml_2022_usd"].to_numpy(dtype=float)
    - expected_oep_ceded
)
append_check(
    validation_rows,
    "oep_pml_reinsurance_formula_reproduced",
    np.allclose(
        pml_table["ceded_oep_pml_2022_usd"],
        expected_oep_ceded,
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    )
    and np.allclose(
        pml_table["net_retained_oep_pml_2022_usd"],
        expected_oep_retained,
        rtol=0.0,
        atol=NUMERIC_ATOL_USD,
    ),
    (
        "maximum_ceded_error="
        f"{np.max(np.abs(pml_table['ceded_oep_pml_2022_usd'].to_numpy(dtype=float) - expected_oep_ceded)):.3e}; "
        "maximum_retained_error="
        f"{np.max(np.abs(pml_table['net_retained_oep_pml_2022_usd'].to_numpy(dtype=float) - expected_oep_retained)):.3e}"
    ),
)

risk_metrics = pd.DataFrame(
    [
        annual_statistics(values, measure, curve_type)
        for (measure, curve_type), values in loss_vectors.items()
    ]
)
risk_metrics["aal_2022_usd"] = np.where(
    risk_metrics["curve_type"].eq("AEP"),
    risk_metrics["mean_annual_loss_2022_usd"],
    np.nan,
)
risk_metrics["portfolio_replacement_value_2022_usd"] = portfolio_replacement_value
risk_metrics["maximum_annual_loss_ratio"] = (
    risk_metrics["maximum_annual_loss_2022_usd"] / portfolio_replacement_value
)
risk_metrics["attachment_2022_usd"] = attachment
risk_metrics["limit_2022_usd"] = limit
risk_metrics["exhaustion_2022_usd"] = exhaustion
risk_metrics.to_csv(RISK_METRICS_PATH, index=False)

source_rows: list[dict[str, Any]] = []
for source_type, group in events.groupby("source_type", sort=True):
    gross_total = float(group["gross_insured_loss_2022_usd"].sum())
    ceded_total = float(group["ceded_loss_2022_usd"].sum())
    retained_total = float(group["net_retained_loss_2022_usd"].sum())
    source_rows.append(
        {
            "source_type": source_type,
            "occurrences": int(len(group)),
            "occurrences_above_attachment": int(group["attachment_triggered"].sum()),
            "occurrences_at_or_above_exhaustion": int(group["layer_exhausted"].sum()),
            "gross_insured_loss_total_2022_usd": gross_total,
            "ceded_loss_total_2022_usd": ceded_total,
            "net_retained_loss_total_2022_usd": retained_total,
            "gross_insured_aal_2022_usd": gross_total / declared_years,
            "ceded_aal_2022_usd": ceded_total / declared_years,
            "net_retained_aal_2022_usd": retained_total / declared_years,
            "ceded_share_of_source_gross": (
                ceded_total / gross_total if gross_total > 0.0 else 0.0
            ),
            "share_of_portfolio_ceded_aal": (
                ceded_total / float(ceded.sum()) if ceded.sum() > 0.0 else 0.0
            ),
            "share_of_portfolio_net_retained_aal": (
                retained_total / float(retained.sum())
                if retained.sum() > 0.0
                else 0.0
            ),
        }
    )
source_aal = pd.DataFrame(source_rows)
source_aal.to_csv(SOURCE_AAL_PATH, index=False)

source_diagnostics = pd.read_csv(source_diagnostics_path)
source_comparison = source_aal.merge(
    source_diagnostics,
    on="source_type",
    how="outer",
    suffixes=("_cell7", "_cell6"),
    validate="one_to_one",
)
source_error_columns = []
for measure in [
    "gross_insured_loss_total_2022_usd",
    "ceded_loss_total_2022_usd",
    "net_retained_loss_total_2022_usd",
    "gross_insured_aal_2022_usd",
    "ceded_aal_2022_usd",
    "net_retained_aal_2022_usd",
]:
    error_column = f"{measure}_absolute_error"
    source_comparison[error_column] = np.abs(
        source_comparison[f"{measure}_cell7"]
        - source_comparison[f"{measure}_cell6"]
    )
    source_error_columns.append(error_column)
maximum_source_error = float(
    source_comparison[source_error_columns].to_numpy(dtype=float).max()
    if len(source_comparison)
    else 0.0
)
append_check(
    validation_rows,
    "source_diagnostics_reproduce_cell6",
    source_comparison["source_type"].notna().all()
    and maximum_source_error <= AGGREGATE_ATOL_USD,
    f"rows={len(source_comparison)}; maximum_error={maximum_source_error:.3e}",
)
append_check(
    validation_rows,
    "source_aal_reconciles_to_portfolio",
    abs(source_aal["gross_insured_aal_2022_usd"].sum() - calculated_aal["gross_insured"])
    <= NUMERIC_ATOL_USD
    and abs(source_aal["ceded_aal_2022_usd"].sum() - calculated_aal["ceded"])
    <= NUMERIC_ATOL_USD
    and abs(
        source_aal["net_retained_aal_2022_usd"].sum()
        - calculated_aal["net_retained"]
    )
    <= NUMERIC_ATOL_USD,
    (
        f"gross_source_sum={source_aal['gross_insured_aal_2022_usd'].sum():.9f}; "
        f"ceded_source_sum={source_aal['ceded_aal_2022_usd'].sum():.9f}; "
        f"retained_source_sum={source_aal['net_retained_aal_2022_usd'].sum():.9f}"
    ),
)

recovery_by_return_period = pml_table[
    [
        "return_period_years",
        "target_annual_exceedance_probability",
        "gross_insured_aep_pml_2022_usd",
        "ceded_aep_pml_2022_usd",
        "net_retained_aep_pml_2022_usd",
        "ceded_share_of_gross_aep",
        "net_retained_share_of_gross_aep",
        "layer_limit_utilization_aep",
        "gross_insured_oep_pml_2022_usd",
        "ceded_oep_pml_2022_usd",
        "net_retained_oep_pml_2022_usd",
        "ceded_share_of_gross_oep",
        "net_retained_share_of_gross_oep",
        "layer_limit_utilization_oep",
        "minimum_supporting_descending_rank",
        "tail_support_flag",
    ]
].copy()
recovery_by_return_period.to_csv(RECOVERY_BY_RETURN_PERIOD_PATH, index=False)

for curve_type, output_png, output_pdf, title in [
    (
        "AEP",
        AEP_PLOT_PNG_PATH,
        AEP_PLOT_PDF_PATH,
        "Baseline reinsurance annual aggregate loss curves",
    ),
    (
        "OEP",
        OEP_PLOT_PNG_PATH,
        OEP_PLOT_PDF_PATH,
        "Baseline reinsurance annual occurrence loss curves",
    ),
]:
    fig, ax = plt.subplots(figsize=(8.0, 5.0))
    for measure, label in [
        ("gross_insured", "Gross insured"),
        ("ceded", "Ceded"),
        ("net_retained", "Net retained"),
    ]:
        subset = ep_curve.loc[
            ep_curve["loss_measure"].eq(measure)
            & ep_curve["curve_type"].eq(curve_type)
        ]
        ax.plot(
            subset["return_period_years"],
            subset["loss_2022_usd"] / 1_000_000.0,
            linewidth=1.8,
            label=label,
        )
    ax.set_xscale("log")
    ax.set_xlabel("Return period (years)")
    ax.set_ylabel("Loss (million 2022 USD)")
    ax.set_title(title)
    ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_png, dpi=300)
    fig.savefig(output_pdf)
    plt.close(fig)

positive_recovery = recovery_by_return_period.loc[
    recovery_by_return_period["gross_insured_oep_pml_2022_usd"] > 0.0
].copy()
fig, ax = plt.subplots(figsize=(8.0, 5.0))
ax.plot(
    positive_recovery["return_period_years"],
    positive_recovery["ceded_share_of_gross_aep"],
    linewidth=1.8,
    label="AEP ceded share",
)
ax.plot(
    positive_recovery["return_period_years"],
    positive_recovery["ceded_share_of_gross_oep"],
    linewidth=1.8,
    label="OEP ceded share",
)
ax.set_xscale("log")
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Ceded loss / gross insured loss")
ax.set_title("Baseline reinsurance ceded share by return period")
ax.set_ylim(0.0, 1.05)
ax.grid(True, which="both", linewidth=0.4, alpha=0.35)
ax.legend()
fig.tight_layout()
fig.savefig(RECOVERY_PLOT_PNG_PATH, dpi=300)
fig.savefig(RECOVERY_PLOT_PDF_PATH)
plt.close(fig)

for output_path, check_id in [
    (OCCURRENCE_LOSS_PATH, "occurrence_loss_file_exists"),
    (ANNUAL_SERIES_PATH, "annual_series_file_exists"),
    (EP_CURVE_PATH, "ep_curve_file_exists"),
    (PML_TABLE_PATH, "pml_table_file_exists"),
    (RISK_METRICS_PATH, "risk_metrics_file_exists"),
    (SOURCE_AAL_PATH, "source_aal_file_exists"),
    (RECOVERY_BY_RETURN_PERIOD_PATH, "recovery_table_file_exists"),
    (AEP_PLOT_PNG_PATH, "aep_plot_png_exists"),
    (AEP_PLOT_PDF_PATH, "aep_plot_pdf_exists"),
    (OEP_PLOT_PNG_PATH, "oep_plot_png_exists"),
    (OEP_PLOT_PDF_PATH, "oep_plot_pdf_exists"),
    (RECOVERY_PLOT_PNG_PATH, "recovery_plot_png_exists"),
    (RECOVERY_PLOT_PDF_PATH, "recovery_plot_pdf_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

thin_tail_return_periods = pml_table.loc[
    pml_table["tail_support_flag"].eq("thin_tail_support"),
    "return_period_years",
].astype(int).tolist()
append_check(
    validation_rows,
    "extreme_return_period_tail_support_documented",
    len(thin_tail_return_periods) == 0,
    (
        f"thin_tail_return_periods={thin_tail_return_periods}; "
        f"threshold_rank={TAIL_SUPPORT_WARNING_RANK}"
    ),
    severity="warning",
)

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
].copy()
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "underlying_policy_scenario": {
        "policy_scenario_id": policy_scenario_id,
        "policy_scenario_name": policy_scenario_name,
    },
    "reinsurance_scenario": {
        "layer_scenario_id": layer_scenario_id,
        "layer_scenario_name": layer_scenario_name,
        "contract_type": str(layer["contract_type"]),
        "subject_loss": str(layer["subject_loss"]),
        "attachment_2022_usd": attachment,
        "limit_2022_usd": limit,
        "exhaustion_2022_usd": exhaustion,
        "reinsurer_participation": participation,
        "annual_aggregate_limit_2022_usd": layer.get(
            "annual_aggregate_limit_2022_usd"
        ),
        "reinstatement_assumption": str(layer["reinstatement_assumption"]),
    },
    "annual_catalog": {
        "declared_duration_years": declared_years,
        "occurrences": expected_occurrences,
        "occupied_years": occupied_years,
        "multiple_event_years": multiple_event_years,
        "zero_event_years": zero_event_years,
        "triggering_occurrences": int(np.count_nonzero(gross > attachment)),
        "exhausting_occurrences": int(np.count_nonzero(gross >= exhaustion)),
        "years_with_ceded_loss": int(np.count_nonzero(annual_ceded_aep > 0.0)),
    },
    "portfolio": {
        "replacement_value_2022_usd": portfolio_replacement_value,
    },
    "risk_metrics": {
        "gross_insured_aal_2022_usd": calculated_aal["gross_insured"],
        "ceded_aal_2022_usd": calculated_aal["ceded"],
        "net_retained_aal_2022_usd": calculated_aal["net_retained"],
        "ceded_share_of_gross_insured_aal": (
            calculated_aal["ceded"] / calculated_aal["gross_insured"]
            if calculated_aal["gross_insured"] > 0.0
            else 0.0
        ),
        "net_retained_share_of_gross_insured_aal": (
            calculated_aal["net_retained"] / calculated_aal["gross_insured"]
            if calculated_aal["gross_insured"] > 0.0
            else 0.0
        ),
        "maximum_gross_insured_aep_2022_usd": float(annual_gross_aep.max()),
        "maximum_gross_insured_oep_2022_usd": float(annual_gross_oep.max()),
        "maximum_ceded_aep_2022_usd": float(annual_ceded_aep.max()),
        "maximum_ceded_oep_2022_usd": float(annual_ceded_oep.max()),
        "maximum_net_retained_aep_2022_usd": float(annual_retained_aep.max()),
        "maximum_net_retained_oep_2022_usd": float(annual_retained_oep.max()),
    },
    "pml_method": {
        "definition": (
            "For return period R, sort all declared annual losses in descending "
            "order and select rank ceil(N/R), where N is the declared catalog "
            "duration."
        ),
        "return_periods_years": return_periods,
        "tail_support_warning_rank": TAIL_SUPPORT_WARNING_RANK,
        "thin_tail_return_periods": thin_tail_return_periods,
        "layer_limit_utilization_interpretation": {
            "oep": (
                "Ceded OEP PML divided by the participated occurrence layer "
                "limit. This is bounded between zero and one."
            ),
            "aep": (
                "Annual aggregate ceded AEP PML divided by one participated "
                "occurrence layer limit. This may exceed one when multiple "
                "occurrences recover in the same year because the baseline has "
                "no annual aggregate cap or reinstatement constraint."
            ),
            "maximum_annual_triggering_occurrences": int(
                annual_series["triggering_occurrences"].max()
            ),
        },
    },
    "source_files": {
        "cell4_event_loss_path": str(event_loss_path),
        "cell4_event_loss_sha256": actual_event_hash,
        "cell5_annual_series_path": str(cell5_annual_path),
        "cell5_annual_series_sha256": actual_cell5_annual_hash,
        "cell5_pml_path": str(cell5_pml_path),
        "cell5_pml_sha256": actual_cell5_pml_hash,
        "cell6_layer_terms_path": str(layer_terms_path),
        "cell6_layer_terms_sha256": actual_layer_terms_hash,
        "cell6_controlled_cases_path": str(controlled_path),
        "cell6_controlled_cases_sha256": actual_controlled_hash,
        "cell6_source_diagnostics_path": str(source_diagnostics_path),
        "cell6_source_diagnostics_sha256": actual_source_diagnostics_hash,
    },
    "outputs": {
        "occurrence_reinsurance_loss": {
            "path": str(OCCURRENCE_LOSS_PATH),
            "sha256": sha256_file(OCCURRENCE_LOSS_PATH),
            "rows": int(len(events)),
        },
        "annual_loss_series": {
            "path": str(ANNUAL_SERIES_PATH),
            "sha256": sha256_file(ANNUAL_SERIES_PATH),
            "rows": declared_years,
        },
        "exceedance_probability_curve": {
            "path": str(EP_CURVE_PATH),
            "sha256": sha256_file(EP_CURVE_PATH),
            "rows": int(len(ep_curve)),
        },
        "pml_table": {
            "path": str(PML_TABLE_PATH),
            "sha256": sha256_file(PML_TABLE_PATH),
            "rows": int(len(pml_table)),
        },
        "risk_metrics": {
            "path": str(RISK_METRICS_PATH),
            "sha256": sha256_file(RISK_METRICS_PATH),
            "rows": int(len(risk_metrics)),
        },
        "source_aal_summary": {
            "path": str(SOURCE_AAL_PATH),
            "sha256": sha256_file(SOURCE_AAL_PATH),
            "rows": int(len(source_aal)),
        },
        "recovery_by_return_period": {
            "path": str(RECOVERY_BY_RETURN_PERIOD_PATH),
            "sha256": sha256_file(RECOVERY_BY_RETURN_PERIOD_PATH),
            "rows": int(len(recovery_by_return_period)),
        },
        "aep_plot_png": str(AEP_PLOT_PNG_PATH),
        "aep_plot_pdf": str(AEP_PLOT_PDF_PATH),
        "oep_plot_png": str(OEP_PLOT_PNG_PATH),
        "oep_plot_pdf": str(OEP_PLOT_PDF_PATH),
        "recovery_plot_png": str(RECOVERY_PLOT_PNG_PATH),
        "recovery_plot_pdf": str(RECOVERY_PLOT_PDF_PATH),
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "validation": {
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warning_failures.to_dict(orient="records"),
    },
    "next_cell": (
        "Cell 8: finalize the Notebook 6 baseline insurance and reinsurance "
        "handoff, create compact portfolio risk tables, and prepare outputs for "
        "the final results and documentation notebook."
    ),
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 7 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 7 ANNUAL REINSURANCE RISK METRICS COMPLETE")
print("=" * 78)
print(f"Underlying policy scenario:       {policy_scenario_id}")
print(f"Reinsurance scenario:             {layer_scenario_id}")
print(f"Declared catalog years:           {declared_years:,}")
print(f"Catalog occurrences:              {expected_occurrences:,}")
print(f"Attachment:                       ${attachment:,.0f}")
print(f"Limit:                            ${limit:,.0f}")
print(f"Exhaustion:                       ${exhaustion:,.0f}")
print(f"Gross insured AAL:                ${calculated_aal['gross_insured']:,.2f}")
print(f"Ceded AAL:                        ${calculated_aal['ceded']:,.2f}")
print(f"Net retained AAL:                 ${calculated_aal['net_retained']:,.2f}")
print(
    "Ceded share of gross AAL:         "
    f"{calculated_aal['ceded'] / calculated_aal['gross_insured']:.4%}"
)
print(f"Maximum ceded AEP loss:           ${annual_ceded_aep.max():,.0f}")
print(f"Maximum ceded OEP loss:           ${annual_ceded_oep.max():,.0f}")
print(f"Maximum net retained AEP loss:    ${annual_retained_aep.max():,.0f}")
print(f"Maximum net retained OEP loss:    ${annual_retained_oep.max():,.0f}")
print(
    f"Critical validation checks:       "
    f"{int(validation['severity'].astype(str).str.lower().eq('critical').sum())}"
)
print(f"Critical failures:                {len(critical_failures)}")
print(f"Warnings requiring review:        {len(warning_failures)}")
print()
print("Occurrence reinsurance loss:")
print(f"  {OCCURRENCE_LOSS_PATH}")
print("Annual loss series:")
print(f"  {ANNUAL_SERIES_PATH}")
print("PML table:")
print(f"  {PML_TABLE_PATH}")
print("Exceedance curve:")
print(f"  {EP_CURVE_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Next: finalize the Notebook 6 insurance and reinsurance handoff.")



NOTEBOOK 6 CELL 7 ANNUAL REINSURANCE RISK METRICS COMPLETE
Underlying policy scenario:       baseline_full_coverage_10pct_deductible_v1
Reinsurance scenario:             baseline_occurrence_xol_500yr_to_2500yr_oep_v1
Declared catalog years:           2,000,000
Catalog occurrences:              10,630
Attachment:                       $18,811,084
Limit:                            $61,837,983
Exhaustion:                       $80,649,067
Gross insured AAL:                $122,979.56
Ceded AAL:                        $63,676.60
Net retained AAL:                 $59,302.96
Ceded share of gross AAL:         51.7782%
Maximum ceded AEP loss:           $90,413,665
Maximum ceded OEP loss:           $61,837,983
Maximum net retained AEP loss:    $182,487,824
Maximum net retained OEP loss:    $182,487,824
Critical validation checks:       97
Critical failures:                0
Warnings requiring review:        1

Occurrence reinsurance loss:
  c:\Users\USER\Documents\GitHub\seismic-correlation-in

In [15]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PIPELINE_VERSION = "notebook6_cell8_finalize_baseline_handoff_v1"
DOLLAR_YEAR = 2022
AGGREGATE_ATOL_USD = 0.01
EXPECTED_TAIL_WARNING_ID = "extreme_return_period_tail_support_documented"
HEADLINE_RETURN_PERIODS = [100, 250, 500, 1000, 2500, 5000, 10000]


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "06_apply_insurance_terms.ipynb").exists():
            return candidate
        if (
            candidate
            / "data"
            / "metadata"
            / "notebook_6_insurance_terms"
            / "notebook_6_cell_7_summary.json"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not identify the project root. Run Notebook 6 from the "
        "seismic-correlation-insurance-loss repository."
    )


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def resolve_recorded_path(project_root: Path, recorded_path: str | Path) -> Path:
    direct = Path(recorded_path)
    if direct.exists():
        return direct

    normalized = str(recorded_path).replace("\\", "/")
    marker = "/data/"
    position = normalized.lower().find(marker)
    if position >= 0:
        candidate = project_root / Path(normalized[position + 1 :])
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Recorded path does not exist: {recorded_path}")


def parse_bool_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)

    normalized = values.astype(str).str.strip().str.lower()
    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "pass": True,
        "passed": True,
        "false": False,
        "0": False,
        "no": False,
        "fail": False,
        "failed": False,
    }
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        bad = sorted(normalized.loc[parsed.isna()].unique().tolist())
        raise ValueError(f"Unrecognized Boolean values: {bad}")
    return parsed.astype(bool)


def append_check(
    rows: list[dict[str, Any]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def inspect_validation(
    path: Path,
    allowed_warning_ids: set[str] | None = None,
) -> dict[str, Any]:
    table = pd.read_csv(path)
    required = {"check_id", "severity", "passed", "detail"}
    missing = sorted(required.difference(table.columns))
    if missing:
        raise KeyError(f"Validation file {path} is missing columns: {missing}")

    table["passed"] = parse_bool_series(table["passed"])
    severity = table["severity"].astype(str).str.strip().str.lower()
    critical_failures = table.loc[severity.eq("critical") & ~table["passed"]].copy()
    warnings = table.loc[severity.eq("warning") & ~table["passed"]].copy()

    allowed = allowed_warning_ids or set()
    warning_ids = set(warnings["check_id"].astype(str).tolist())
    return {
        "critical_failures": critical_failures,
        "warnings": warnings,
        "unexpected_warning_ids": sorted(warning_ids.difference(allowed)),
        "critical_checks": int(severity.eq("critical").sum()),
    }


def output_record(
    category: str,
    output_name: str,
    path: Path,
    source_cell: str,
    row_count: int | None = None,
) -> dict[str, Any]:
    return {
        "category": category,
        "output_name": output_name,
        "source_cell": source_cell,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": int(path.stat().st_size) if path.exists() else 0,
        "sha256": sha256_file(path) if path.exists() else "",
        "rows": row_count,
    }


PROJECT_ROOT = find_project_root()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "notebook_6_insurance_terms"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_6_baseline_handoff"
METADATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATHS = {
    cell: METADATA_DIR / f"notebook_6_cell_{cell}_summary.json"
    for cell in range(1, 8)
}
VALIDATION_PATHS = {
    cell: METADATA_DIR / f"notebook_6_cell_{cell}_validation.csv"
    for cell in range(1, 8)
}

LOSS_FLOW_PATH = OUTPUT_DIR / "notebook_6_baseline_loss_flow_summary.csv"
PML_COMPARISON_PATH = OUTPUT_DIR / "notebook_6_baseline_pml_comparison.csv"
HEADLINE_PML_PATH = OUTPUT_DIR / "notebook_6_baseline_headline_pml_table.csv"
ASSUMPTIONS_PATH = OUTPUT_DIR / "notebook_6_baseline_model_assumptions.csv"
OUTPUT_MANIFEST_PATH = OUTPUT_DIR / "notebook_6_baseline_output_manifest.csv"
FINAL_HANDOFF_PATH = METADATA_DIR / "notebook_6_final_handoff.json"
VALIDATION_PATH = METADATA_DIR / "notebook_6_cell_8_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_6_cell_8_summary.json"

validation_rows: list[dict[str, Any]] = []

for cell, path in SUMMARY_PATHS.items():
    append_check(
        validation_rows,
        f"cell{cell}_summary_exists",
        path.exists() and path.stat().st_size > 0,
        str(path),
    )
for cell, path in VALIDATION_PATHS.items():
    append_check(
        validation_rows,
        f"cell{cell}_validation_exists",
        path.exists() and path.stat().st_size > 0,
        str(path),
    )

summaries = {cell: load_json(path) for cell, path in SUMMARY_PATHS.items()}

for cell, summary in summaries.items():
    append_check(
        validation_rows,
        f"cell{cell}_summary_reports_critical_success",
        bool(summary.get("all_critical_checks_passed", False)),
        f"pipeline_version={summary.get('pipeline_version')}",
    )

for cell, path in VALIDATION_PATHS.items():
    allowed = (
        {EXPECTED_TAIL_WARNING_ID}
        if cell in {5, 7}
        else set()
    )
    inspected = inspect_validation(path, allowed_warning_ids=allowed)
    append_check(
        validation_rows,
        f"cell{cell}_has_no_critical_failures",
        inspected["critical_failures"].empty,
        (
            f"critical_checks={inspected['critical_checks']}; "
            f"critical_failures={len(inspected['critical_failures'])}"
        ),
    )
    append_check(
        validation_rows,
        f"cell{cell}_has_no_unexpected_warnings",
        len(inspected["unexpected_warning_ids"]) == 0,
        f"unexpected_warning_ids={inspected['unexpected_warning_ids']}",
    )

cell1 = summaries[1]
cell2 = summaries[2]
cell4 = summaries[4]
cell5 = summaries[5]
cell6 = summaries[6]
cell7 = summaries[7]

policy_id = str(cell5["policy_scenario"]["policy_scenario_id"])
layer_id = str(cell7["reinsurance_scenario"]["layer_scenario_id"])

append_check(
    validation_rows,
    "policy_scenario_consistent_across_cells",
    policy_id
    == str(cell4["policy_scenario"]["policy_scenario_id"])
    == str(cell6["underlying_policy_scenario"]["policy_scenario_id"])
    == str(cell7["underlying_policy_scenario"]["policy_scenario_id"]),
    f"policy_scenario_id={policy_id}",
)
append_check(
    validation_rows,
    "reinsurance_scenario_consistent_across_cells",
    layer_id == str(cell6["reinsurance_scenario"]["layer_scenario_id"]),
    f"layer_scenario_id={layer_id}",
)

annual_catalog = cell5["annual_catalog"]
portfolio = cell5["portfolio"]
insured_metrics = cell5["risk_metrics"]
reinsurance_metrics = cell7["risk_metrics"]
layer = cell7["reinsurance_scenario"]

catalog_years = int(annual_catalog["declared_duration_years"])
occurrences = int(annual_catalog["occurrences"])
buildings = int(portfolio["buildings"])
replacement_value = float(portfolio["replacement_value_2022_usd"])

ground_up_aal = float(insured_metrics["ground_up_aal_2022_usd"])
gross_aal = float(insured_metrics["gross_insured_aal_2022_usd"])
uninsured_aal = float(insured_metrics["uninsured_aal_2022_usd"])
ceded_aal = float(reinsurance_metrics["ceded_aal_2022_usd"])
retained_aal = float(reinsurance_metrics["net_retained_aal_2022_usd"])

append_check(
    validation_rows,
    "ground_up_aal_conserves_to_gross_and_uninsured",
    abs(ground_up_aal - gross_aal - uninsured_aal) <= AGGREGATE_ATOL_USD,
    f"difference={ground_up_aal - gross_aal - uninsured_aal:.6f}",
)
append_check(
    validation_rows,
    "gross_aal_conserves_to_ceded_and_retained",
    abs(gross_aal - ceded_aal - retained_aal) <= AGGREGATE_ATOL_USD,
    f"difference={gross_aal - ceded_aal - retained_aal:.6f}",
)
append_check(
    validation_rows,
    "risk_aals_nonnegative",
    min(ground_up_aal, gross_aal, uninsured_aal, ceded_aal, retained_aal) >= 0.0,
    (
        f"ground_up={ground_up_aal:.6f}; gross={gross_aal:.6f}; "
        f"uninsured={uninsured_aal:.6f}; ceded={ceded_aal:.6f}; "
        f"retained={retained_aal:.6f}"
    ),
)
append_check(
    validation_rows,
    "risk_aal_ordering_is_consistent",
    gross_aal <= ground_up_aal + AGGREGATE_ATOL_USD
    and ceded_aal <= gross_aal + AGGREGATE_ATOL_USD
    and retained_aal <= gross_aal + AGGREGATE_ATOL_USD,
    "gross<=ground_up and ceded,retained<=gross",
)

loss_flow = pd.DataFrame(
    [
        {
            "stage_order": 1,
            "loss_measure": "ground_up",
            "description": "Total structural and nonstructural repair loss before insurance",
            "aal_2022_usd": ground_up_aal,
            "share_of_ground_up_aal": 1.0,
            "share_of_gross_insured_aal": np.nan,
            "maximum_aep_loss_2022_usd": float(
                insured_metrics["maximum_ground_up_aep_2022_usd"]
            ),
            "maximum_oep_loss_2022_usd": float(
                insured_metrics["maximum_ground_up_oep_2022_usd"]
            ),
        },
        {
            "stage_order": 2,
            "loss_measure": "uninsured",
            "description": "Ground-up loss not recovered under the baseline policy",
            "aal_2022_usd": uninsured_aal,
            "share_of_ground_up_aal": uninsured_aal / ground_up_aal,
            "share_of_gross_insured_aal": np.nan,
            "maximum_aep_loss_2022_usd": float(
                insured_metrics["maximum_uninsured_aep_2022_usd"]
            ),
            "maximum_oep_loss_2022_usd": float(
                insured_metrics["maximum_uninsured_oep_2022_usd"]
            ),
        },
        {
            "stage_order": 3,
            "loss_measure": "gross_insured",
            "description": "Insurer liability after building policy terms and before reinsurance",
            "aal_2022_usd": gross_aal,
            "share_of_ground_up_aal": gross_aal / ground_up_aal,
            "share_of_gross_insured_aal": 1.0,
            "maximum_aep_loss_2022_usd": float(
                insured_metrics["maximum_gross_insured_aep_2022_usd"]
            ),
            "maximum_oep_loss_2022_usd": float(
                insured_metrics["maximum_gross_insured_oep_2022_usd"]
            ),
        },
        {
            "stage_order": 4,
            "loss_measure": "ceded",
            "description": "Gross insured loss transferred to the occurrence XoL layer",
            "aal_2022_usd": ceded_aal,
            "share_of_ground_up_aal": ceded_aal / ground_up_aal,
            "share_of_gross_insured_aal": ceded_aal / gross_aal,
            "maximum_aep_loss_2022_usd": float(
                reinsurance_metrics["maximum_ceded_aep_2022_usd"]
            ),
            "maximum_oep_loss_2022_usd": float(
                reinsurance_metrics["maximum_ceded_oep_2022_usd"]
            ),
        },
        {
            "stage_order": 5,
            "loss_measure": "net_retained",
            "description": "Insurer loss after occurrence XoL recovery",
            "aal_2022_usd": retained_aal,
            "share_of_ground_up_aal": retained_aal / ground_up_aal,
            "share_of_gross_insured_aal": retained_aal / gross_aal,
            "maximum_aep_loss_2022_usd": float(
                reinsurance_metrics["maximum_net_retained_aep_2022_usd"]
            ),
            "maximum_oep_loss_2022_usd": float(
                reinsurance_metrics["maximum_net_retained_oep_2022_usd"]
            ),
        },
    ]
)
loss_flow.to_csv(LOSS_FLOW_PATH, index=False)

cell5_pml_path = resolve_recorded_path(
    PROJECT_ROOT, cell5["outputs"]["pml_table"]["path"]
)
cell7_pml_path = resolve_recorded_path(
    PROJECT_ROOT, cell7["outputs"]["pml_table"]["path"]
)
cell5_pml = pd.read_csv(cell5_pml_path)
cell7_pml = pd.read_csv(cell7_pml_path)

for frame, name in [(cell5_pml, "cell5"), (cell7_pml, "cell7")]:
    if "return_period_years" not in frame.columns:
        raise KeyError(f"{name} PML table is missing return_period_years")
    frame["return_period_years"] = pd.to_numeric(
        frame["return_period_years"], errors="raise"
    ).astype(np.int64)

required_cell5_columns = {
    "return_period_years",
    "ground_up_aep_pml_2022_usd",
    "ground_up_oep_pml_2022_usd",
    "gross_insured_aep_pml_2022_usd",
    "gross_insured_oep_pml_2022_usd",
    "uninsured_aep_pml_2022_usd",
    "uninsured_oep_pml_2022_usd",
    "insurance_recovery_share_aep",
    "insurance_recovery_share_oep",
    "minimum_supporting_descending_rank",
    "tail_support_flag",
}
required_cell7_columns = {
    "return_period_years",
    "gross_insured_aep_pml_2022_usd",
    "gross_insured_oep_pml_2022_usd",
    "ceded_aep_pml_2022_usd",
    "ceded_oep_pml_2022_usd",
    "net_retained_aep_pml_2022_usd",
    "net_retained_oep_pml_2022_usd",
    "ceded_share_of_gross_aep",
    "ceded_share_of_gross_oep",
    "layer_limit_utilization_aep",
    "layer_limit_utilization_oep",
    "minimum_supporting_descending_rank",
    "tail_support_flag",
}
append_check(
    validation_rows,
    "cell5_pml_schema_complete",
    required_cell5_columns.issubset(cell5_pml.columns),
    f"missing={sorted(required_cell5_columns.difference(cell5_pml.columns))}",
)
append_check(
    validation_rows,
    "cell7_pml_schema_complete",
    required_cell7_columns.issubset(cell7_pml.columns),
    f"missing={sorted(required_cell7_columns.difference(cell7_pml.columns))}",
)

pml_comparison = cell5_pml[
    [
        "return_period_years",
        "target_annual_exceedance_probability",
        "ground_up_aep_pml_2022_usd",
        "ground_up_oep_pml_2022_usd",
        "gross_insured_aep_pml_2022_usd",
        "gross_insured_oep_pml_2022_usd",
        "uninsured_aep_pml_2022_usd",
        "uninsured_oep_pml_2022_usd",
        "insurance_recovery_share_aep",
        "insurance_recovery_share_oep",
        "minimum_supporting_descending_rank",
        "tail_support_flag",
    ]
].merge(
    cell7_pml[
        [
            "return_period_years",
            "gross_insured_aep_pml_2022_usd",
            "gross_insured_oep_pml_2022_usd",
            "ceded_aep_pml_2022_usd",
            "ceded_oep_pml_2022_usd",
            "net_retained_aep_pml_2022_usd",
            "net_retained_oep_pml_2022_usd",
            "ceded_share_of_gross_aep",
            "ceded_share_of_gross_oep",
            "layer_limit_utilization_aep",
            "layer_limit_utilization_oep",
            "minimum_supporting_descending_rank",
            "tail_support_flag",
        ]
    ],
    on="return_period_years",
    how="inner",
    validate="one_to_one",
    suffixes=("_insurance", "_reinsurance"),
)

for curve_type in ["aep", "oep"]:
    left = pml_comparison[
        f"gross_insured_{curve_type}_pml_2022_usd_insurance"
    ].to_numpy(dtype=np.float64)
    right = pml_comparison[
        f"gross_insured_{curve_type}_pml_2022_usd_reinsurance"
    ].to_numpy(dtype=np.float64)
    append_check(
        validation_rows,
        f"gross_insured_{curve_type}_pml_reconciles_between_cells5_and7",
        np.all(np.abs(left - right) <= AGGREGATE_ATOL_USD),
        f"maximum_difference={float(np.max(np.abs(left - right))):.6f}",
    )
    pml_comparison[f"gross_insured_{curve_type}_pml_2022_usd"] = left
    pml_comparison.drop(
        columns=[
            f"gross_insured_{curve_type}_pml_2022_usd_insurance",
            f"gross_insured_{curve_type}_pml_2022_usd_reinsurance",
        ],
        inplace=True,
    )

pml_comparison["minimum_supporting_descending_rank"] = np.minimum(
    pml_comparison["minimum_supporting_descending_rank_insurance"].to_numpy(
        dtype=np.int64
    ),
    pml_comparison["minimum_supporting_descending_rank_reinsurance"].to_numpy(
        dtype=np.int64
    ),
)
pml_comparison["tail_support_flag"] = np.where(
    pml_comparison["minimum_supporting_descending_rank"] < 20,
    "thin_tail_support",
    "adequate_order_statistic_support",
)
pml_comparison.drop(
    columns=[
        "minimum_supporting_descending_rank_insurance",
        "minimum_supporting_descending_rank_reinsurance",
        "tail_support_flag_insurance",
        "tail_support_flag_reinsurance",
    ],
    inplace=True,
)

ordered_columns = [
    "return_period_years",
    "target_annual_exceedance_probability",
    "ground_up_aep_pml_2022_usd",
    "gross_insured_aep_pml_2022_usd",
    "uninsured_aep_pml_2022_usd",
    "ceded_aep_pml_2022_usd",
    "net_retained_aep_pml_2022_usd",
    "ground_up_oep_pml_2022_usd",
    "gross_insured_oep_pml_2022_usd",
    "uninsured_oep_pml_2022_usd",
    "ceded_oep_pml_2022_usd",
    "net_retained_oep_pml_2022_usd",
    "insurance_recovery_share_aep",
    "insurance_recovery_share_oep",
    "ceded_share_of_gross_aep",
    "ceded_share_of_gross_oep",
    "layer_limit_utilization_aep",
    "layer_limit_utilization_oep",
    "minimum_supporting_descending_rank",
    "tail_support_flag",
]
pml_comparison = pml_comparison[ordered_columns].sort_values(
    "return_period_years"
).reset_index(drop=True)
pml_comparison.to_csv(PML_COMPARISON_PATH, index=False)

headline_pml = pml_comparison.loc[
    pml_comparison["return_period_years"].isin(HEADLINE_RETURN_PERIODS)
].copy()
headline_pml.to_csv(HEADLINE_PML_PATH, index=False)

append_check(
    validation_rows,
    "pml_return_periods_match_between_insurance_and_reinsurance",
    set(cell5_pml["return_period_years"].tolist())
    == set(cell7_pml["return_period_years"].tolist()),
    (
        f"cell5_count={len(cell5_pml)}; cell7_count={len(cell7_pml)}; "
        f"merged_count={len(pml_comparison)}"
    ),
)
append_check(
    validation_rows,
    "headline_return_periods_complete",
    set(headline_pml["return_period_years"].tolist())
    == set(HEADLINE_RETURN_PERIODS),
    f"headline_return_periods={headline_pml['return_period_years'].tolist()}",
)

policy = cell2["policy_scenario"]
assumption_rows = [
    {
        "assumption_group": "policy",
        "parameter": "policy_scenario_id",
        "value": policy["policy_scenario_id"],
        "unit": "identifier",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "policy",
        "parameter": "policy_take_up_share",
        "value": policy["policy_take_up_share"],
        "unit": "fraction",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "policy",
        "parameter": "covered_loss_share",
        "value": policy["covered_loss_share"],
        "unit": "fraction",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "policy",
        "parameter": "deductible_percent_of_replacement_value",
        "value": policy["deductible_percent_of_replacement_value"],
        "unit": "fraction",
        "basis": "per building, per occurrence",
    },
    {
        "assumption_group": "policy",
        "parameter": "policy_limit_percent_of_replacement_value",
        "value": policy["policy_limit_percent_of_replacement_value"],
        "unit": "fraction",
        "basis": "per building, per occurrence",
    },
    {
        "assumption_group": "policy",
        "parameter": "coinsurance_share",
        "value": policy["coinsurance_share"],
        "unit": "fraction",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "layer_scenario_id",
        "value": layer_id,
        "unit": "identifier",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "attachment_2022_usd",
        "value": layer["attachment_2022_usd"],
        "unit": "2022 USD",
        "basis": "500-year gross insured OEP PML",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "limit_2022_usd",
        "value": layer["limit_2022_usd"],
        "unit": "2022 USD",
        "basis": "2,500-year exhaustion minus 500-year attachment",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "exhaustion_2022_usd",
        "value": layer["exhaustion_2022_usd"],
        "unit": "2022 USD",
        "basis": "2,500-year gross insured OEP PML",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "reinsurer_participation",
        "value": layer["reinsurer_participation"],
        "unit": "fraction",
        "basis": "synthetic project baseline",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "annual_aggregate_limit_2022_usd",
        "value": layer.get("annual_aggregate_limit_2022_usd"),
        "unit": "2022 USD",
        "basis": "not modeled",
    },
    {
        "assumption_group": "reinsurance",
        "parameter": "reinstatement_assumption",
        "value": layer["reinstatement_assumption"],
        "unit": "text",
        "basis": "baseline contract simplification",
    },
]
assumptions = pd.DataFrame(assumption_rows)
assumptions.to_csv(ASSUMPTIONS_PATH, index=False)

key_prior_outputs: list[tuple[str, str, Path, str, int | None]] = []

def add_summary_output(
    summary: dict[str, Any],
    output_key: str,
    category: str,
    source_cell: str,
) -> None:
    record = summary["outputs"][output_key]
    if isinstance(record, dict):
        path = resolve_recorded_path(PROJECT_ROOT, record["path"])
        rows = record.get("rows")
    else:
        path = resolve_recorded_path(PROJECT_ROOT, record)
        rows = None
    key_prior_outputs.append((category, output_key, path, source_cell, rows))

add_summary_output(cell4, "event_insured_loss", "production_loss", "cell_4")
for key in [
    "annual_loss_series",
    "exceedance_probability_curve",
    "pml_table",
    "risk_metrics",
    "source_aal_summary",
    "recovery_by_return_period",
]:
    add_summary_output(cell5, key, "insured_risk", "cell_5")
for key in [
    "layer_terms",
    "controlled_cases",
    "response_grid",
    "source_diagnostics",
    "assumption_registry",
]:
    add_summary_output(cell6, key, "reinsurance_parameters", "cell_6")
for key in [
    "occurrence_reinsurance_loss",
    "annual_loss_series",
    "exceedance_probability_curve",
    "pml_table",
    "risk_metrics",
    "source_aal_summary",
    "recovery_by_return_period",
]:
    add_summary_output(cell7, key, "reinsurance_risk", "cell_7")

manifest_rows = [
    output_record(category, name, path, source_cell, rows)
    for category, name, path, source_cell, rows in key_prior_outputs
]
manifest_rows.extend(
    [
        output_record(
            "final_handoff", "loss_flow_summary", LOSS_FLOW_PATH, "cell_8", len(loss_flow)
        ),
        output_record(
            "final_handoff",
            "pml_comparison",
            PML_COMPARISON_PATH,
            "cell_8",
            len(pml_comparison),
        ),
        output_record(
            "final_handoff",
            "headline_pml_table",
            HEADLINE_PML_PATH,
            "cell_8",
            len(headline_pml),
        ),
        output_record(
            "final_handoff",
            "model_assumptions",
            ASSUMPTIONS_PATH,
            "cell_8",
            len(assumptions),
        ),
    ]
)
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(OUTPUT_MANIFEST_PATH, index=False)

append_check(
    validation_rows,
    "all_manifest_files_exist",
    output_manifest["exists"].astype(bool).all(),
    f"missing={output_manifest.loc[~output_manifest['exists'], 'path'].tolist()}",
)
append_check(
    validation_rows,
    "all_manifest_files_nonempty",
    (output_manifest["size_bytes"].astype(np.int64) > 0).all(),
    (
        "empty="
        f"{output_manifest.loc[output_manifest['size_bytes'].eq(0), 'path'].tolist()}"
    ),
)
append_check(
    validation_rows,
    "manifest_hashes_complete",
    output_manifest["sha256"].astype(str).str.len().eq(64).all(),
    f"manifest_rows={len(output_manifest)}",
)
append_check(
    validation_rows,
    "loss_flow_rows_complete",
    set(loss_flow["loss_measure"].tolist())
    == {"ground_up", "uninsured", "gross_insured", "ceded", "net_retained"},
    f"loss_measures={loss_flow['loss_measure'].tolist()}",
)
append_check(
    validation_rows,
    "portfolio_and_catalog_counts_positive",
    catalog_years > 0 and occurrences > 0 and buildings > 0 and replacement_value > 0.0,
    (
        f"catalog_years={catalog_years}; occurrences={occurrences}; "
        f"buildings={buildings}; replacement_value={replacement_value:.2f}"
    ),
)

thin_tail_return_periods = pml_comparison.loc[
    pml_comparison["tail_support_flag"].eq("thin_tail_support"),
    "return_period_years",
].astype(int).tolist()
append_check(
    validation_rows,
    "extreme_return_period_tail_support_carried_forward",
    len(thin_tail_return_periods) == 0,
    (
        f"thin_tail_return_periods={thin_tail_return_periods}; "
        "interpret_as_diagnostics_not_headline_metrics"
    ),
    severity="warning",
)

final_handoff = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete" if all(
        row["passed"] or row["severity"] == "warning" for row in validation_rows
    ) else "validation_pending",
    "project": {
        "name": "seismic-correlation-insurance-loss",
        "phase": "phase_1_no_spatial_correlation_baseline",
        "notebook": "06_apply_insurance_terms.ipynb",
        "scope": (
            "sampled total ground-up repair loss, synthetic building policy terms, "
            "and synthetic occurrence excess-of-loss reinsurance"
        ),
        "currency": "USD",
        "dollar_year": DOLLAR_YEAR,
    },
    "annual_catalog": {
        "declared_duration_years": catalog_years,
        "occurrences": occurrences,
        "occupied_years": int(annual_catalog["occupied_years"]),
        "multiple_event_years": int(annual_catalog["multiple_event_years"]),
        "zero_event_years": int(annual_catalog["zero_event_years"]),
    },
    "portfolio": {
        "buildings": buildings,
        "replacement_value_2022_usd": replacement_value,
    },
    "policy_scenario": cell2["policy_scenario"],
    "reinsurance_scenario": cell7["reinsurance_scenario"],
    "aal_2022_usd": {
        "ground_up": ground_up_aal,
        "uninsured": uninsured_aal,
        "gross_insured": gross_aal,
        "ceded": ceded_aal,
        "net_retained": retained_aal,
    },
    "aal_shares": {
        "insurance_recovery_share_of_ground_up": gross_aal / ground_up_aal,
        "uninsured_share_of_ground_up": uninsured_aal / ground_up_aal,
        "ceded_share_of_gross_insured": ceded_aal / gross_aal,
        "net_retained_share_of_gross_insured": retained_aal / gross_aal,
    },
    "maximum_annual_losses_2022_usd": {
        "ground_up_aep": float(insured_metrics["maximum_ground_up_aep_2022_usd"]),
        "ground_up_oep": float(insured_metrics["maximum_ground_up_oep_2022_usd"]),
        "gross_insured_aep": float(
            insured_metrics["maximum_gross_insured_aep_2022_usd"]
        ),
        "gross_insured_oep": float(
            insured_metrics["maximum_gross_insured_oep_2022_usd"]
        ),
        "ceded_aep": float(reinsurance_metrics["maximum_ceded_aep_2022_usd"]),
        "ceded_oep": float(reinsurance_metrics["maximum_ceded_oep_2022_usd"]),
        "net_retained_aep": float(
            reinsurance_metrics["maximum_net_retained_aep_2022_usd"]
        ),
        "net_retained_oep": float(
            reinsurance_metrics["maximum_net_retained_oep_2022_usd"]
        ),
    },
    "tail_support": {
        "warning_threshold_rank": 20,
        "thin_tail_return_periods_years": thin_tail_return_periods,
        "interpretation": (
            "Return periods with fewer than 20 supporting annual order statistics "
            "are retained as diagnostics and should not be headline results."
        ),
    },
    "final_outputs": {
        "loss_flow_summary": {
            "path": str(LOSS_FLOW_PATH),
            "sha256": sha256_file(LOSS_FLOW_PATH),
            "rows": int(len(loss_flow)),
        },
        "pml_comparison": {
            "path": str(PML_COMPARISON_PATH),
            "sha256": sha256_file(PML_COMPARISON_PATH),
            "rows": int(len(pml_comparison)),
        },
        "headline_pml_table": {
            "path": str(HEADLINE_PML_PATH),
            "sha256": sha256_file(HEADLINE_PML_PATH),
            "rows": int(len(headline_pml)),
        },
        "model_assumptions": {
            "path": str(ASSUMPTIONS_PATH),
            "sha256": sha256_file(ASSUMPTIONS_PATH),
            "rows": int(len(assumptions)),
        },
        "output_manifest": {
            "path": str(OUTPUT_MANIFEST_PATH),
            "sha256": sha256_file(OUTPUT_MANIFEST_PATH),
            "rows": int(len(output_manifest)),
        },
        "validation": str(VALIDATION_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "recommended_headline_return_periods_years": HEADLINE_RETURN_PERIODS,
    "next_stage": (
        "Prepare the baseline results, README, technical write-up, and interview "
        "explanation before beginning the spatial-correlation extension."
    ),
}
write_json(FINAL_HANDOFF_PATH, final_handoff)

for output_path, check_id in [
    (LOSS_FLOW_PATH, "loss_flow_output_exists"),
    (PML_COMPARISON_PATH, "pml_comparison_output_exists"),
    (HEADLINE_PML_PATH, "headline_pml_output_exists"),
    (ASSUMPTIONS_PATH, "assumptions_output_exists"),
    (OUTPUT_MANIFEST_PATH, "output_manifest_exists"),
    (FINAL_HANDOFF_PATH, "final_handoff_exists"),
]:
    append_check(
        validation_rows,
        check_id,
        output_path.exists() and output_path.stat().st_size > 0,
        str(output_path),
    )

validation = pd.DataFrame(validation_rows)
critical_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("critical")
    & ~validation["passed"]
].copy()
warning_failures = validation.loc[
    validation["severity"].astype(str).str.lower().eq("warning")
    & ~validation["passed"]
].copy()
validation.to_csv(VALIDATION_PATH, index=False)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "all_critical_checks_passed": critical_failures.empty,
    "policy_scenario_id": policy_id,
    "reinsurance_scenario_id": layer_id,
    "annual_catalog": final_handoff["annual_catalog"],
    "portfolio": final_handoff["portfolio"],
    "aal_2022_usd": final_handoff["aal_2022_usd"],
    "aal_shares": final_handoff["aal_shares"],
    "reinsurance_layer": {
        "attachment_2022_usd": float(layer["attachment_2022_usd"]),
        "limit_2022_usd": float(layer["limit_2022_usd"]),
        "exhaustion_2022_usd": float(layer["exhaustion_2022_usd"]),
        "reinsurer_participation": float(layer["reinsurer_participation"]),
    },
    "tail_support": final_handoff["tail_support"],
    "outputs": {
        **final_handoff["final_outputs"],
        "final_handoff": {
            "path": str(FINAL_HANDOFF_PATH),
            "sha256": sha256_file(FINAL_HANDOFF_PATH),
        },
    },
    "validation": {
        "critical_checks": int(
            validation["severity"].astype(str).str.lower().eq("critical").sum()
        ),
        "critical_failures": critical_failures.to_dict(orient="records"),
        "warnings": warning_failures.to_dict(orient="records"),
    },
    "notebook_status": "complete" if critical_failures.empty else "failed",
    "next_stage": final_handoff["next_stage"],
}
write_json(SUMMARY_PATH, summary)

if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 6 Cell 8 failed critical validation. Review "
        f"{VALIDATION_PATH}."
    )

print()
print("=" * 78)
print("NOTEBOOK 6 CELL 8 BASELINE INSURANCE HANDOFF COMPLETE")
print("=" * 78)
print(f"Policy scenario:                   {policy_id}")
print(f"Reinsurance scenario:              {layer_id}")
print(f"Declared catalog years:            {catalog_years:,}")
print(f"Catalog occurrences:               {occurrences:,}")
print(f"Portfolio buildings:               {buildings:,}")
print(f"Portfolio replacement value:       ${replacement_value:,.0f}")
print(f"Ground-up AAL:                     ${ground_up_aal:,.2f}")
print(f"Gross insured AAL:                 ${gross_aal:,.2f}")
print(f"Uninsured AAL:                     ${uninsured_aal:,.2f}")
print(f"Ceded AAL:                         ${ceded_aal:,.2f}")
print(f"Net retained AAL:                  ${retained_aal:,.2f}")
print(f"Insurance recovery share:          {gross_aal / ground_up_aal:.4%}")
print(f"Ceded share of gross insured AAL:  {ceded_aal / gross_aal:.4%}")
print(f"Critical validation checks:        {int(validation['severity'].astype(str).str.lower().eq('critical').sum())}")
print(f"Critical failures:                 {len(critical_failures)}")
print(f"Warnings requiring review:         {len(warning_failures)}")
print()
print("Loss-flow summary:")
print(f"  {LOSS_FLOW_PATH}")
print("PML comparison:")
print(f"  {PML_COMPARISON_PATH}")
print("Headline PML table:")
print(f"  {HEADLINE_PML_PATH}")
print("Output manifest:")
print(f"  {OUTPUT_MANIFEST_PATH}")
print("Final Notebook 6 handoff:")
print(f"  {FINAL_HANDOFF_PATH}")
print("Validation:")
print(f"  {VALIDATION_PATH}")
print("Summary:")
print(f"  {SUMMARY_PATH}")
print()
print("Notebook 6 is complete. Next: prepare the baseline results and documentation.")



NOTEBOOK 6 CELL 8 BASELINE INSURANCE HANDOFF COMPLETE
Policy scenario:                   baseline_full_coverage_10pct_deductible_v1
Reinsurance scenario:              baseline_occurrence_xol_500yr_to_2500yr_oep_v1
Declared catalog years:            2,000,000
Catalog occurrences:               10,630
Portfolio buildings:               470
Portfolio replacement value:       $384,236,605
Ground-up AAL:                     $195,922.45
Gross insured AAL:                 $122,979.56
Uninsured AAL:                     $72,942.89
Ceded AAL:                         $63,676.60
Net retained AAL:                  $59,302.96
Insurance recovery share:          62.7695%
Ceded share of gross insured AAL:  51.7782%
Critical validation checks:        58
Critical failures:                 0
Warnings requiring review:         1

Loss-flow summary:
  c:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_6_baseline_handoff\notebook_6_baseline_loss_flow_summary.csv
PML co